# Test C — selezione v7 · C-sel-8 · CPU

Questa è la correzione di C-sel-7: 197/200 coppie DUE-nat. Alcuni tokenizer,
nel controllo di integrità, pulivano spazi prima dei segni di interpunzione durante
la decodifica, producendo falsi scarti. C-sel-8 disabilita esplicitamente quella
pulizia **solo per il controllo** (nessuna modifica ai token dei modelli).

Mantiene invariati i criteri semantici C-nat-7, il generatore C-gen-4,
i semi e i limiti di 512 token SIRIN. I tre casi che superano il limite
non vengono riammessi. Le quote e la verifica indipendente devono ancora passare.

Questa è la correzione della selezione naturale dopo la revisione di C-sel-6.
**La prova GPU TestC_v2 resta valida per il sintetico. Non occorre ripeterla.**

1. Apri questo notebook in Colab con **CPU** e `HF_TOKEN` abilitato nei Secrets.
2. Il notebook riutilizza ambienti e tokenizer se sono già presenti nel runtime;
   se Colab assegna una macchina nuova, li prepara di nuovo.
3. Scegli **Runtime → Esegui tutte**.
4. Manda il nuovo **testC_selezione.zip** da
   **MyDrive/testC_parziali/selezione_C-sel-8/**.

Le uscite C-sel-6 restano nella cartella precedente. Non occorre spostare file a mano.
Il notebook non esegue inferenze dei modelli e non richiede GPU. La durata dipende
dalle installazioni mancanti; la selezione precedente, con l'ambiente pronto, è durata
4.8 minuti. Questo dato non garantisce la durata della nuova selezione.

## Perché è necessaria la correzione

Lo ZIP C-sel-6 è integro e i controlli automatici passano. La revisione del testo ha
però trovato **sei errori e undici casi dubbi nelle 50 coppie naturali del campione**:
due errori e sei dubbi in RIM-nat, quattro errori e cinque dubbi in DUE-nat.
Le altre 33 coppie non mostrano problemi evidenti nella lettura; questo non è una
certificazione semantica. Le 20 coppie sintetiche del campione sono identiche a quelle
già esaminate. Il sintetico selezionato è invariato per contenuto e ID.

I sei errori osservati:

| Famiglia | Problema nel testo C |
| --- | --- |
| RIM-nat | Victoria resta riconoscibile in Victorian. |
| RIM-nat | Louis Agassiz è eliminato per esteso, ma Agassiz compare ancora. |
| DUE-nat | Si propone two or seven ma le sette aree sono elencate dopo i due punti. |
| DUE-nat | Ten of the seven lines rende impossibile il distrattore dieci. |
| DUE-nat | Central Asian Muslims è un gruppo, mentre il seguito He died richiede un individuo. |
| DUE-nat | Sei pacchetti totali e i quattro remaining di Sky determinano due per Setanta anche in un'altra frase. |

La verifica automatica non misurava tutte queste proprietà. Il suo esito positivo
non approvava la revisione del significato dei casi naturali.

## Regole C-nat-7

Le regole sono generali: non contengono una lista di ID da eliminare e non leggono
risposte o punteggi dei modelli. Sono controlli di qualità aggiunti dopo l'audit,
documentati separatamente dalle ipotesi preregistrate. Il naturale resta descrittivo.

- RIM: si riconoscono anche componenti distintive dei nomi composti e forme come
  Victoria/Victorian. Si scartano dipendenze introdotte dalle rimozioni: It/Its,
  sintagmi come The treaty o The eye e riferimenti temporali rimasti senza antecedente.
  Un nome completo che comincia con The, come The University of Example, non viene
  trattato automaticamente come anafora.
- DUE: i calcoli residui si cercano nell'intero paragrafo; si scartano elenchi
  quantitativi, conteggi di sottoinsiemi e contesti con medie/estremi o vincoli cronologici
  che il sostitutore meccanico non certifica. Sono esclusioni prudenti, non la tesi
  che tutti quei contesti sarebbero necessariamente sbagliati.
- I distrattori nominali devono avere evidenza del ruolo: un gruppo non sostituisce
  una persona; alumni non basta a certificare autore di un libro; il fondatore di
  un'impresa richiede una fonte con quel ruolo. Restano stesso articolo e classe precisa.
- Si applicano vincoli elementari ulteriori: a una quantità intera non si associa
  un distrattore frazionario; le quote percentuali, escluse le domande su variazioni,
  non superano 100. Le distanze numeriche originarie restano invariate.
- Il tipo viene ricavato da tutti i riferimenti SQuAD coerenti e si usa il più breve
  del tipo, con il suo answer_start. Non dipende più dal primo annotatore. Si riconosce
  anche one come numero. Un tipo ambiguo viene scartato.
- Per domande sulla nascita si richiede l'evento esplicito nella frase della risposta:
  una durata di una carica fra parentesi non certifica una data di nascita.

Il generatore resta C-gen-4, con le stesse riserve, semi, quote e tokenizer.
La selezione prosegue nell'ordine deterministico fino a 200 coppie per famiglia.
Se le quote non sono raggiunte, il risultato viene dichiarato incompleto: non si
allentano automaticamente le regole.

## Verifiche e limiti

**22 regressioni** si eseguono prima delle installazioni. Il verificatore C-verifica-4
è separato e non importa selettore, generatore o filtri C-nat-7. Ricostruisce lo span
effettivamente modificato in DUE e non scambia eight rows of combs per un sottoinsieme
numerico. I numeri attaccati a unità come 6am non sono addendi isolati.

Le 22 regressioni passano localmente; il verificatore aggiornato intercetta i sei
errori sui dati effettivi di C-sel-6. Una prova del bacino originale SQuAD raggiunge
200 RIM-nat e 200 DUE-nat **senza l'ammissione dei tokenizer reali**: non costituisce
la selezione finale. Questa si esegue qui su Colab. Il campione risultante deve essere
letto di nuovo: nessun filtro lessicale certifica da solo l'indeterminatezza semantica.

Lo ZIP conserva anche l'audit delle 50 coppie naturali di C-sel-6. Questo audit è una
revisione dell'assistente, non un'annotazione cieca effettuata da Simona. Il manifest
mantiene revisione_manuale_approvata=False. Prima del completo restano revisione del
campione effettivo, HC0 di M1 con la generazione aggiornata e aggiornamento degli hash
nel notebook di inferenza. Non sostituire a mano i dati della prova GPU.


In [ ]:
# 1. Impostazioni (non serve cambiare nulla)
USA_DRIVE = True
VERSIONE = "C-sel-8"          # generatore C-gen-4
SIRIN_COMMIT = "ef3bed143890dbee56b34755f8c27a218aef0aa2"
MODELLI = {"qwen": ["Qwen/Qwen3-8B", "b968826d9c46dd6066d109eabc6255188de91218"],
           "mistral": ["mistralai/Mistral-Nemo-Instruct-2407", "04d8a90549d23fc6bd7f642064003592df51e9b3"],
           "olmo": ["allenai/OLMo-2-1124-7B-Instruct", "470b1fba1ae01581f270116362ee4aa1b97f4c84"],
           "llama": ["meta-llama/Llama-3.1-8B-Instruct", "0e9e39f249a16976918f6564b8830bc894c89659"]}
GIUDICE = ["Qwen/Qwen3-4B", "1cfa9a7208912126459214e8b04321603b3df60c"]
SQUAD_REVISIONE = "3ffb306f725f7d2ce8394bc1873b24868140c412"
SHA_SELEZIONE_TEST4 = "5ac6e6e0ebd4ceedc8c230d8510f025720a41acf3a29c5dbb61e1f7f7fdf9d0b"
SHA_RISERVE = {"definitivo": "32007443261cb4205d774f23ffc327c4b2b0ef3d2c162a3897fee93053c30937",   # C-gen-4
               "prova": "cdecb33aeaef00989f4c8dc6bf423a41072138c68c4f8898b2449f47df2eeb9d"}
PIN_GENERATORE = ["geonamescache==2.0.0", "faker==30.8.2", "english-words==2.0.1"]
TOLLERANZA, N_PROVA = 3, 10
# quote fisse per tipo di attributo × schema di posizione (protocollo v3, §3.3c): 50 per tipo, 67/67/66 per schema
QUOTE = {"anno/0": 17, "anno/1": 17, "anno/2": 16, "numero/0": 17, "numero/1": 16, "numero/2": 17, "luogo/0": 16, "luogo/1": 17, "luogo/2": 17, "persona/0": 17, "persona/1": 17, "persona/2": 16}
SEME_NATURALE, N_NATURALE, SEME_REVISIONE = 20261113, 200, 20261114
# Hash canonici dei passaggi di C-sel-5: non includono il metadato versione.
SHA_CONTENUTO_SINTETICO = {'definitivo': 'ee5ba0ce191c98b57504aacfe6347a7cd7641fcd5b5e775b3dbff77b002c4915', 'prova': 'c9317dcb02782a52abd84d535ec7e0bdf9e6f9cf29444494923b7cba2e7ecfd2'}
SHA_ID_SINTETICO = {'definitivo': 'c672d9947d4ecd1a7696623c3c265397046b8612e84688ada78d2e2845511840', 'prova': 'b577c51760d54011fadbbdf7cadf9de1de7cd08f51307df1a475cb25a95c6c03'}


In [ ]:
# 2. Cartelle e funzioni comuni
import hashlib, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

LAVORO = Path("/content/testC_sel6")
SCRIPT, ENVS = LAVORO / "script", LAVORO / "envs"
for d in (LAVORO, SCRIPT, ENVS):
    d.mkdir(parents=True, exist_ok=True)
if USA_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RADICE = Path("/content/drive/MyDrive/testC_parziali/selezione_C-sel-8")
else:
    RADICE = LAVORO / "uscite_C-sel-7"
RADICE.mkdir(parents=True, exist_ok=True)
precedenti = [f for f in RADICE.iterdir() if not f.name.startswith("archivio_")]
if precedenti:
    archivio = RADICE / ("archivio_" + time.strftime("%Y%m%d-%H%M%S") + "_" + str(time.time_ns() % 1000000))
    archivio.mkdir()
    for f in precedenti:
        shutil.move(str(f), str(archivio / f.name))
    print("Uscite precedenti conservate in", archivio)
LOG = RADICE / "log"
LOG.mkdir(exist_ok=True)
print("Nuove uscite in", RADICE)

def sha_file(p):
    return hashlib.sha256(Path(p).read_bytes()).hexdigest()

def sha_oggetto(x):
    return hashlib.sha256(json.dumps(x, sort_keys=True, ensure_ascii=False, allow_nan=False).encode()).hexdigest()

def scrivi_json(p, x):
    p = Path(p)
    p.parent.mkdir(parents=True, exist_ok=True)
    tmp = p.with_suffix(p.suffix + ".tmp")
    tmp.write_text(json.dumps(x, sort_keys=True, ensure_ascii=False, allow_nan=False, indent=1))
    tmp.replace(p)

def crea_zip():
    zp = RADICE / "testC_selezione.zip"
    tmp = RADICE / "testC_selezione.zip.tmp"
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in sorted(RADICE.rglob("*")):
            rel = f.relative_to(RADICE)
            if (f.is_file() and not rel.parts[0].startswith("archivio_")
                    and f.suffix in (".json", ".md", ".txt", ".log", ".py")):
                z.write(f, rel)
    tmp.replace(zp)
    print("ZIP:", zp, f"{zp.stat().st_size / 1e6:.1f} MB")
    return zp

scrivi_json(RADICE / "manifest_selezione.json", dict(
    versione=VERSIONE, regole_naturale="C-nat-7", fase="avviata",
    pronto_per_revisione_manuale=False, revisione_manuale_approvata=False))

def sh(cmd, log, cwd=None, env=None):
    with open(log, "a") as fh:
        fh.write(f"\n$ {cmd}\n")
        fh.flush()
        p = subprocess.run(cmd, shell=True, cwd=cwd, env=env, stdout=fh, stderr=subprocess.STDOUT)
    if p.returncode != 0:
        crea_zip()
        raise RuntimeError(f"comando fallito ({p.returncode}): {cmd}\n--- ultime righe del log ---\n{Path(log).read_text()[-4000:]}")

_TOKEN = []
def hf_token():
    if _TOKEN:
        return _TOKEN[0]
    token = os.environ.get("HF_TOKEN")
    for _ in range(3):
        if token:
            break
        try:
            from google.colab import userdata
            token = userdata.get("HF_TOKEN")
        except Exception:
            token = None
    if token:
        _TOKEN.append(token)
    return token

print("Token Hugging Face:", "presente" if hf_token() else "ASSENTE: abilita HF_TOKEN nei Secrets prima di proseguire")
# Il token viene passato solo nell'ambiente del processo: mai salvato nella configurazione.

(RADICE / 'note_Cnat7.md').write_text("# Test C — selezione v7 · C-sel-8 · CPU\n\nQuesta è la correzione tecnica C-sel-8 della decodifica per il controllo di integrità.\nLa selezione precedente C-sel-7 ha raggiunto 200/200 RIM-nat e 197/200 DUE-nat.\nNel registro risultano 10 scarti DUE-nat per \"testo ricevuto non valido\":\nsette mostrano esclusivamente la falsa alterazione del testo in Llama, tre hanno\nanche lunghezze SIRIN oltre il limite di 512 token e restano esclusi.\nQuesto aggiornamento rende esplicito clean_up_tokenization_spaces=False nella\nsola decodifica della verifica del prompt di risposta: NON cambia i token codificati,\ni limiti SIRIN, i criteri semantici C-nat-7, i semi, i tokenizer o le riserve.\nNon è garantita la quota prima dell'esecuzione su Colab; se resta incompleta,\nil notebook produce lo ZIP diagnostico e non concede l'approvazione.\n\nQuesta è la correzione della selezione naturale dopo la revisione di C-sel-6.\n**La prova GPU TestC_v2 resta valida per il sintetico. Non occorre ripeterla.**\n\n1. Apri questo notebook in Colab con **CPU** e `HF_TOKEN` abilitato nei Secrets.\n2. Il notebook riutilizza ambienti e tokenizer se sono già presenti nel runtime;\n   se Colab assegna una macchina nuova, li prepara di nuovo.\n3. Scegli **Runtime → Esegui tutte**.\n4. Manda il nuovo **testC_selezione.zip** da\n   **MyDrive/testC_parziali/selezione_C-sel-8/**.\n\nLe uscite C-sel-6 restano nella cartella precedente. Non occorre spostare file a mano.\nIl notebook non esegue inferenze dei modelli e non richiede GPU. La durata dipende\ndalle installazioni mancanti; la selezione precedente, con l'ambiente pronto, è durata\n4.8 minuti. Questo dato non garantisce la durata della nuova selezione.\n\n## Perché è necessaria la correzione\n\nLo ZIP C-sel-6 è integro e i controlli automatici passano. La revisione del testo ha\nperò trovato **sei errori e undici casi dubbi nelle 50 coppie naturali del campione**:\ndue errori e sei dubbi in RIM-nat, quattro errori e cinque dubbi in DUE-nat.\nLe altre 33 coppie non mostrano problemi evidenti nella lettura; questo non è una\ncertificazione semantica. Le 20 coppie sintetiche del campione sono identiche a quelle\ngià esaminate. Il sintetico selezionato è invariato per contenuto e ID.\n\nI sei errori osservati:\n\n| Famiglia | Problema nel testo C |\n| --- | --- |\n| RIM-nat | Victoria resta riconoscibile in Victorian. |\n| RIM-nat | Louis Agassiz è eliminato per esteso, ma Agassiz compare ancora. |\n| DUE-nat | Si propone two or seven ma le sette aree sono elencate dopo i due punti. |\n| DUE-nat | Ten of the seven lines rende impossibile il distrattore dieci. |\n| DUE-nat | Central Asian Muslims è un gruppo, mentre il seguito He died richiede un individuo. |\n| DUE-nat | Sei pacchetti totali e i quattro remaining di Sky determinano due per Setanta anche in un'altra frase. |\n\nLa verifica automatica non misurava tutte queste proprietà. Il suo esito positivo\nnon approvava la revisione del significato dei casi naturali.\n\n## Regole C-nat-7\n\nLe regole sono generali: non contengono una lista di ID da eliminare e non leggono\nrisposte o punteggi dei modelli. Sono controlli di qualità aggiunti dopo l'audit,\ndocumentati separatamente dalle ipotesi preregistrate. Il naturale resta descrittivo.\n\n- RIM: si riconoscono anche componenti distintive dei nomi composti e forme come\n  Victoria/Victorian. Si scartano dipendenze introdotte dalle rimozioni: It/Its,\n  sintagmi come The treaty o The eye e riferimenti temporali rimasti senza antecedente.\n  Un nome completo che comincia con The, come The University of Example, non viene\n  trattato automaticamente come anafora.\n- DUE: i calcoli residui si cercano nell'intero paragrafo; si scartano elenchi\n  quantitativi, conteggi di sottoinsiemi e contesti con medie/estremi o vincoli cronologici\n  che il sostitutore meccanico non certifica. Sono esclusioni prudenti, non la tesi\n  che tutti quei contesti sarebbero necessariamente sbagliati.\n- I distrattori nominali devono avere evidenza del ruolo: un gruppo non sostituisce\n  una persona; alumni non basta a certificare autore di un libro; il fondatore di\n  un'impresa richiede una fonte con quel ruolo. Restano stesso articolo e classe precisa.\n- Si applicano vincoli elementari ulteriori: a una quantità intera non si associa\n  un distrattore frazionario; le quote percentuali, escluse le domande su variazioni,\n  non superano 100. Le distanze numeriche originarie restano invariate.\n- Il tipo viene ricavato da tutti i riferimenti SQuAD coerenti e si usa il più breve\n  del tipo, con il suo answer_start. Non dipende più dal primo annotatore. Si riconosce\n  anche one come numero. Un tipo ambiguo viene scartato.\n- Per domande sulla nascita si richiede l'evento esplicito nella frase della risposta:\n  una durata di una carica fra parentesi non certifica una data di nascita.\n\nIl generatore resta C-gen-4, con le stesse riserve, semi, quote e tokenizer.\nLa selezione prosegue nell'ordine deterministico fino a 200 coppie per famiglia.\nSe le quote non sono raggiunte, il risultato viene dichiarato incompleto: non si\nallentano automaticamente le regole.\n\n## Verifiche e limiti\n\n**22 regressioni** si eseguono prima delle installazioni. Il verificatore C-verifica-4\nè separato e non importa selettore, generatore o filtri C-nat-7. Ricostruisce lo span\neffettivamente modificato in DUE e non scambia eight rows of combs per un sottoinsieme\nnumerico. I numeri attaccati a unità come 6am non sono addendi isolati.\n\nLe 22 regressioni passano localmente; il verificatore aggiornato intercetta i sei\nerrori sui dati effettivi di C-sel-6. Una prova del bacino originale SQuAD raggiunge\n200 RIM-nat e 200 DUE-nat **senza l'ammissione dei tokenizer reali**: non costituisce\nla selezione finale. Questa si esegue qui su Colab. Il campione risultante deve essere\nletto di nuovo: nessun filtro lessicale certifica da solo l'indeterminatezza semantica.\n\nLo ZIP conserva anche l'audit delle 50 coppie naturali di C-sel-6. Questo audit è una\nrevisione dell'assistente, non un'annotazione cieca effettuata da Simona. Il manifest\nmantiene revisione_manuale_approvata=False. Prima del completo restano revisione del\ncampione effettivo, HC0 di M1 con la generazione aggiornata e aggiornamento degli hash\nnel notebook di inferenza. Non sostituire a mano i dati della prova GPU.\n")
scrivi_json(RADICE / 'audit_Csel6.json', json.loads('{"selezione": "C-sel-6", "revisore": "assistente", "revisione_cieca_Simona": false, "sha256_righe_C": "59d7a56741078a82f9e0305c0af0263ee755464b2769ce7e30aee83be5242e4b", "errori": 6, "dubbi": 11, "senza_problemi_evidenti": 33, "annotazioni": [{"n": 21, "famiglia": "RIM-nat", "id": "570d4606b3d812140066d619", "domanda": "What state in Australia is the center of dairy farming?", "esito": "errore", "nota": "Victoria resta riconoscibile in Victorian; It rimane senza l’antecedente eliminato."}, {"n": 22, "famiglia": "RIM-nat", "id": "57286b003acd2414000df9c1", "domanda": "Where had the Mongol capital been before Kublai moved it?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 23, "famiglia": "RIM-nat", "id": "57264845f1498d1400e8db0b", "domanda": "Who did the army siege belong to?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 24, "famiglia": "RIM-nat", "id": "5726241189a1e219009ac2e1", "domanda": "The oil crisis caused oil companies to increase oil supplies in which area?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 25, "famiglia": "RIM-nat", "id": "57112686b654c5140001fbd4", "domanda": "What ideal thermodynamic cycle analyzes the process by which steam engines work?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 26, "famiglia": "RIM-nat", "id": "572820512ca10214002d9e74", "domanda": "What is the name of the river that is completely contained inside Jacksonville?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 27, "famiglia": "RIM-nat", "id": "5733266d4776f41900660716", "domanda": "What year did King Sigismund III Vasa move his court to Warsaw?", "esito": "dubbio", "nota": "C inizia con In the following years senza l’evento che ne fissa l’anno."}, {"n": 28, "famiglia": "RIM-nat", "id": "57297547af94a219006aa45d", "domanda": "In the mid 18th century, who did not concur that 1 should be the first prime number?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 29, "famiglia": "RIM-nat", "id": "571126dfa58dae1900cd6cb6", "domanda": "Who conceptualized the piston?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 30, "famiglia": "RIM-nat", "id": "572867d72ca10214002da2fc", "domanda": "Who was Kaidu\'s grandfather?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 31, "famiglia": "RIM-nat", "id": "5727213c708984140094da38", "domanda": "Inspired by Shelley what was the name of Gandhi\'s doctrine?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 32, "famiglia": "RIM-nat", "id": "5729f8516aef05140015516d", "domanda": "What antibody is transported from the mother to baby across the placenta?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 33, "famiglia": "RIM-nat", "id": "572759dbdd62a815002e9b96", "domanda": "What notable private school has an endowment of several hundred million dollars?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 34, "famiglia": "RIM-nat", "id": "572fd8efb2c2fd14005684f9", "domanda": "What is the number of Constituency MSPs?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 35, "famiglia": "RIM-nat", "id": "5728202c4b864d19001644ee", "domanda": "The Hungarians performed this civil disobedience under the direction of what person?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 36, "famiglia": "RIM-nat", "id": "5727c0402ca10214002d9564", "domanda": "in 1846 who\'s natural history lectures were acclaimed in New York and Harvard?", "esito": "errore", "nota": "Louis Agassiz è rimosso per esteso, ma Agassiz compare ripetutamente in C."}, {"n": 37, "famiglia": "RIM-nat", "id": "572fffb404bcaa1900d76ff1", "domanda": "Where did the discharge from glaciers go in Europe in the last Ice Age?", "esito": "dubbio", "nota": "La domanda Where è ampia: il riferimento residuo al mare può fornire una risposta geografica alternativa, pur non nominando il Reno."}, {"n": 38, "famiglia": "RIM-nat", "id": "5726b58f5951b619008f7b59", "domanda": " What entity has taken the view that the goals of free trade are underpinned by the aims to improve people\'s well being?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 39, "famiglia": "RIM-nat", "id": "570d2af0fed7b91900d45c96", "domanda": "What city is the capital of Victoria?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 40, "famiglia": "RIM-nat", "id": "56de10b44396321400ee2593", "domanda": "What was the name of the Norman castle?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 41, "famiglia": "RIM-nat", "id": "57286f373acd2414000df9df", "domanda": "When did Buyantu resume testing potential government employees?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 42, "famiglia": "RIM-nat", "id": "5728804b4b864d1900164a47", "domanda": "When did Zhu Shijie die?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 43, "famiglia": "RIM-nat", "id": "572f57c704bcaa1900d7686e", "domanda": "What is the delta in the Rhine delimited in the west by?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 44, "famiglia": "RIM-nat", "id": "57340a094776f41900661780", "domanda": "Although some Acadians went to France and other destiantions, what North American city did many move to?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 45, "famiglia": "RIM-nat", "id": "5725ec8289a1e219009ac0b0", "domanda": "In what year did developers Billings & Meyering acquire the Alta Vista Tract?", "esito": "dubbio", "nota": "A mere half decade later perde l’evento del 1914 a cui si riferiva."}, {"n": 46, "famiglia": "RIM-nat", "id": "57274ca8dd62a815002e9aa8", "domanda": "In private schools, what language are classes taught in?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 47, "famiglia": "RIM-nat", "id": "57339902d058e614000b5e73", "domanda": "Who designed the garden for the University Library?", "esito": "dubbio", "nota": "Eliminata la frase sul giardino, It e The building possono far riferire il progetto del giardino agli architetti dell’edificio."}, {"n": 48, "famiglia": "RIM-nat", "id": "572881704b864d1900164a50", "domanda": "How many schools of medicine were recognized in China?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 49, "famiglia": "RIM-nat", "id": "572824f13acd2414000df58f", "domanda": "What storm had the most significant impact on Jacksonville?", "esito": "dubbio", "nota": "The eye e Similarly, four years prior to this perdono gli antecedenti; restano altri uragani ed effetti comparativi."}, {"n": 50, "famiglia": "RIM-nat", "id": "5733e771d058e614000b6545", "domanda": "What was the North American portion of War of Austrian Succession?", "esito": "dubbio", "nota": "C comincia con The treaty dopo aver eliminato la frase che identifica il trattato."}, {"n": 51, "famiglia": "DUE-nat", "id": "571093aba58dae1900cd6a5d", "domanda": "When did Henry issue the Edict of Nantes?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 52, "famiglia": "DUE-nat", "id": "57286bf84b864d19001649d2", "domanda": "What alumni member also write the bestseller Before I Fall?", "esito": "dubbio", "nota": "Masaaki Shirakawa è documentato come governatore e alumni, senza evidenza del ruolo di autore del bestseller."}, {"n": 53, "famiglia": "DUE-nat", "id": "5705e3f252bb89140068966b", "domanda": "How many metropolitan areas does Southern California\'s population encompass?", "esito": "errore", "nota": "Le sette aree sono ancora elencate dopo i due punti. L’alternativa two è anche inferiore alle three aree maggiori citate dopo."}, {"n": 54, "famiglia": "DUE-nat", "id": "572691545951b619008f76e4", "domanda": "Which country refused to content to changes in the Treaty of Lisbon 2007?", "esito": "dubbio", "nota": "Per Greenland la fonte attesta un trattato diverso del 1985; la comune parola treaty non documenta il ruolo nel contesto della domanda."}, {"n": 55, "famiglia": "DUE-nat", "id": "57286951ff5b5019007da20f", "domanda": "Who was the founder of the Oracle Corporation?", "esito": "dubbio", "nota": "Per Saul Alinsky la fonte attesta un fondatore di community organizing, non un fondatore di impresa."}, {"n": 56, "famiglia": "DUE-nat", "id": "57106d2fb654c5140001f8f1", "domanda": "France laid siege to Montpellier in what year?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 57, "famiglia": "DUE-nat", "id": "57300a9a04bcaa1900d77067", "domanda": "When did the German army reoccupy Rhineland?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 58, "famiglia": "DUE-nat", "id": "572a0c541d046914007796f6", "domanda": "Who wrote \\"The Hidden Prosperity of the Poor\\"?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 59, "famiglia": "DUE-nat", "id": "57113be3a58dae1900cd6d28", "domanda": "Who patented a steam engine in 1781?", "esito": "dubbio", "nota": "Watt’s e Watt restano nel paragrafo: indizio asimmetrico a favore di James Watt, anche se non prova da solo la data del brevetto."}, {"n": 60, "famiglia": "DUE-nat", "id": "5706111a52bb89140068984e", "domanda": "How many lines run out of Downtown Los Angeles?", "esito": "errore", "nota": "Ten of the seven lines è impossibile. Il totale sette permette di scartare il distrattore dieci."}, {"n": 61, "famiglia": "DUE-nat", "id": "572881d34b864d1900164a5c", "domanda": "When was the Office of Western Medicine founded?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 62, "famiglia": "DUE-nat", "id": "572867212ca10214002da2f2", "domanda": "Who led a Mongol attack on the Song dynasty?", "esito": "errore", "nota": "Central Asian Muslims è un gruppo, contrapposto a una persona; He died in 1259 continua a richiedere un referente individuale."}, {"n": 63, "famiglia": "DUE-nat", "id": "57094a79efce8f15003a7dc5", "domanda": "How many of the six total packages available to broadcasters was Setanta awarded?", "esito": "errore", "nota": "Sei pacchetti totali e quattro remaining per Sky determinano due per Setanta, anche se in un’altra frase si aggiunge or 5."}, {"n": 64, "famiglia": "DUE-nat", "id": "5709686c200fba1400367f7a", "domanda": "What was the number of customers that the BBC  reported had yet to receive the service due to failed deliveries?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 65, "famiglia": "DUE-nat", "id": "5727cebc2ca10214002d96f2", "domanda": "How many graduate students does Harvard have?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 66, "famiglia": "DUE-nat", "id": "57264a74708984140094c18e", "domanda": "When did the name black death officially take root in England?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 67, "famiglia": "DUE-nat", "id": "5730285a04bcaa1900d7726f", "domanda": "When was the FIS founded?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 68, "famiglia": "DUE-nat", "id": "572fd6aa04bcaa1900d76da2", "domanda": "How many days does the Parliament have to nominate a First Minister after a General Election?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}, {"n": 69, "famiglia": "DUE-nat", "id": "5710e9f8a58dae1900cd6b34", "domanda": "When was the colony destroyed?", "esito": "dubbio", "nota": "Colonia fondata nel 1564 e sopravvissuta only a short time: il 1604 ha un forte indizio cronologico sfavorevole rispetto al 1565."}, {"n": 70, "famiglia": "DUE-nat", "id": "57339eb9d058e614000b5ef7", "domanda": "How many museums are in Warsaw?", "esito": "nessun_problema_rilevato", "nota": "La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione."}]}'))


In [ ]:
%%writefile /content/testC_sel6/script/gen_testC.py
# Test C, generatore dei passaggi sintetici (protocollo v3, §3.1–§3.3).
#
# Ogni passaggio nasce da una rappresentazione formale (entità, attributo bersaglio, fonti, regole, disgiunzioni).
# Da questa si scrive il testo con template fissi. Poi:
#   1. un risolutore calcola i valori compatibili con la rappresentazione;
#   2. un analizzatore inverso rilegge il testo, ricostruisce la rappresentazione e rilancia il risolutore;
#   3. si controllano le occorrenze dei valori, le «or», i marcatori di riserva, le parole vietate, i nomi e il bilanciamento.
# Un passaggio che non supera un controllo esce dalla riserva (con il motivo registrato). La riserva è ordinata e
# riproducibile: la selezione finale (prime 50 per tipo di attributo che superano anche i controlli sui tokenizer,
# fatti su Colab) non dipende da nessuna risposta dei modelli.
#
# Uso: python -I gen_testC.py <cartella_di_uscita>
import hashlib
import json
import random
import re
import sys
from collections import Counter, defaultdict

VERSIONE = "C-gen-4"
SEME_DEFINITIVO = 20261111
SEME_PROVA = 20261112
N_RISERVA = 400            # riserva ordinata per i 200 definitivi (100 per tipo, circa 33 per tipo e schema di posizione)
N_RISERVA_PROVA = 16       # riserva per i 10 della prova
FAMIGLIE = ["RIM", "DUE", "CONF", "LOG"]
TIPI = ["anno", "numero", "luogo", "persona"]
GENERI = ["citta", "azienda", "nave", "festival"]
POSIZIONI = ["inizio", "meta", "fine"]
N_FRASI = 8
PAROLE_MIN, PAROLE_MAX = 120, 180
ANNI_DOPO = (3, 70)         # anni di contorno fra 3 e 70 anni dopo l'anno di riferimento (C-gen-4)
ANNO_MAX_CONTORNO = 1995

NUMERI = {2: "two", 3: "three", 4: "four", 5: "five", 6: "six", 7: "seven", 8: "eight", 9: "nine",
          10: "ten", 11: "eleven", 12: "twelve"}
PAROLE_NUMERO = set(NUMERI.values()) | {"one", "zero", "thirteen", "fourteen", "fifteen", "twenty"}
MARCATORI_RISERVA = ["maybe", "probably", "possibly", "perhaps", "likely", "unlikely", "uncertain", "unknown",
                     "unclear", "seems", "appears to", "may", "might", "could", "believed", "said to", "reportedly",
                     "allegedly", "estimated", "about", "around", "roughly", "approximately", "not known", "no record"]

# --------------------------------------------------------------------------- lessico per genere
# soggetto del passaggio, gruppo per LOG, fonte per CONF, predicati dell'attributo bersaglio (un solo valore per
# definizione), domanda, frase neutra di RIM, frase con «or» irrilevante (DUE), frasi di contorno.
GEN = {
    "citta": dict(
        soggetto="{N}", nome_tipo="town",
        intro="{N} is a small and quiet town in the green hills of the {R} province, far from the main roads and the large cities of the region.",
        gruppo_appartenenza="{S} lies on the {G1} river or on the {G2} river.",
        gruppo_regola="Every town on the {G} river {P}.",
        fonte="According to the {F} archive, {s} {P}.",
        pred={"anno": "was founded in {v}", "numero": "has {v} bridges",
              "luogo": "keeps its original charter in {v}", "persona": "has Saint {v} as its only patron saint"},
        pads={"anno": ["was founded in the year {v}", "was founded in the course of the year {v}"],
              "numero": ["has a total of {v} bridges"],
              "luogo": ["keeps its original charter in the city of {v}", "keeps the original charter of the town in the city of {v}"],
              "persona": ["has Saint {v} as the only patron saint of the town", "has Saint {v} as the one and only patron saint of the town"]},
        domanda={"anno": "In what year was {N} founded?", "numero": "How many bridges does {N} have?",
                 "luogo": "In which city is the original charter of {N} kept?", "persona": "Who is the only patron saint of {N}?"},
        vietate={"anno": ["found"], "numero": ["bridge"], "luogo": ["charter"], "persona": ["saint", "patron"]},
        neutre=["{S} appears in an old travel guide.",
                "{S} was described in an old travel guide.",
                "{S} is described in some detail in an old travel guide.",
                "{S} was described at length in an old travel guide for visitors.",
                "{S} is described in some detail in an old travel guide for visitors from abroad.",
                "{S} was described at length and in some detail in an old travel guide for visitors from abroad."],
        or_frase=("Visitors usually reach {S} by boat or by train.", "Visitors usually reach {S} by boat and by train."),
        contorno=[
            ("anno", "A stone school with a red roof and a small garden for the pupils was built near the eastern gate of {S} in {y}."),
            ("anno", "The old covered market in the centre of {S}, where farmers still sell their vegetables, was rebuilt after a fire in {y}."),
            ("numero", "The main square of {S} has {n} public fountains, a long row of chestnut trees and a few wooden benches in the shade."),
            ("numero", "Each spring the people of {S} plant {n} new gardens along the edge of the old wall that once protected the town."),
            ("luogo", "Merchants from {L} often travel to {S} in the autumn to buy wool, cheese, honey and dried fruit from the local farms."),
            ("luogo", "The heavy bells of the clock tower in {S} were cast in a family workshop in {L} and carried up the hill on carts."),
            ("persona", "The local choir of {S}, which sings in the church every Sunday morning, is led by a retired teacher called {Q}."),
            ("persona", "A painter named {Q} once lived in a narrow house beside the river in {S} and painted the hills in every season."),
            ("libera", "Many families in {S} keep small orchards behind their houses and sell apples, pears and plums at the weekly market in autumn."),
            ("libera", "The streets of {S} are narrow and steep, and most of them are paved with grey stone that becomes slippery in winter."),
        ]),
    "azienda": dict(
        soggetto="{N} Works", nome_tipo="company",
        intro="{N} Works is a family company that makes hand tools, small machines and spare parts for farmers in the {R} province.",
        gruppo_appartenenza="{S} is registered in the {G1} district or in the {G2} district.",
        gruppo_regola="Every company registered in the {G} district {P}.",
        fonte="According to the {F} registry, {s} {P}.",
        pred={"anno": "was founded in {v}", "numero": "has {v} offices",
              "luogo": "has its only warehouse in {v}", "persona": "has Doctor {v} as its sole auditor"},
        pads={"anno": ["was founded in the year {v}", "was founded in the course of the year {v}"],
              "numero": ["has a total of {v} offices"],
              "luogo": ["has its only warehouse in the city of {v}", "has the only warehouse of the company in the city of {v}"],
              "persona": ["has Doctor {v} as the sole auditor of the company", "has Doctor {v} as the one and only auditor of the company"]},
        domanda={"anno": "In what year was {N} Works founded?", "numero": "How many offices does {N} Works have?",
                 "luogo": "In which city is the only warehouse of {N} Works?", "persona": "Who is the sole auditor of {N} Works?"},
        vietate={"anno": ["found"], "numero": ["office"], "luogo": ["warehouse"], "persona": ["audit", "doctor"]},
        neutre=["{S} appears in an old trade journal.",
                "{S} was mentioned in an old trade journal.",
                "{S} is mentioned in some detail in an old trade journal.",
                "{S} was mentioned at length in an old trade journal for merchants.",
                "{S} is mentioned in some detail in an old trade journal for merchants from abroad.",
                "{S} was mentioned at length and in some detail in an old trade journal for merchants from abroad."],
        or_frase=("Clients can contact {S} by letter or by telephone.", "Clients can contact {S} by letter and by telephone."),
        contorno=[
            ("anno", "A second workshop with large windows and a high wooden ceiling was opened behind the main building of {S} in {y}."),
            ("anno", "The engineers of {S} won a prize for a new kind of light hand drill at a crowded trade fair in {y}."),
            ("numero", "The main workshop of {S} has {n} long benches where the workers sharpen the blades and polish the metal parts by hand."),
            ("numero", "Every year {S} trains {n} young apprentices who come from the nearby villages and learn the trade over several long winters."),
            ("luogo", "Most of the steel used by {S} for its blades and gears is bought from a large steelworks in {L}."),
            ("luogo", "The strong wooden boxes in which {S} packs its tools are made by a group of carpenters in {L}."),
            ("persona", "In every meeting with the managers, the workers of {S} are represented by a calm and patient woman called {Q}."),
            ("persona", "The first printed catalogue of {S}, with pictures of every tool, was drawn by hand by an engraver named {Q}."),
            ("libera", "The tools made by {S} are known in the whole region for their smooth wooden handles and their simple, practical design."),
            ("libera", "The workers of {S} take a long break at noon and often eat bread and soup together in the sunny yard behind the workshop."),
        ]),
    "nave": dict(
        soggetto="The {N}", nome_tipo="ship",
        intro="The {N} is a wooden sailing ship that carries passengers, letters and goods along the long rocky coast of the {R} province.",
        gruppo_appartenenza="{S} belongs to the {G1} fleet or to the {G2} fleet.",
        gruppo_regola="Every ship of the {G} fleet {P}.",
        fonte="According to the {F} register, {s} {P}.",
        pred={"anno": "was launched in {v}", "numero": "carries {v} lifeboats",
              "luogo": "has {v} as its only home port", "persona": "has Admiral {v} as its sole commander"},
        pads={"anno": ["was launched in the year {v}", "was launched in the course of the year {v}"],
              "numero": ["carries a total of {v} lifeboats"],
              "luogo": ["has the city of {v} as its only home port", "has the city of {v} as the only home port of the ship"],
              "persona": ["has Admiral {v} as the sole commander of the ship", "has Admiral {v} as the one and only commander of the ship"]},
        domanda={"anno": "In what year was the {N} launched?", "numero": "How many lifeboats does the {N} carry?",
                 "luogo": "What is the only home port of the {N}?", "persona": "Who is the sole commander of the {N}?"},
        vietate={"anno": ["launch"], "numero": ["lifeboat"], "luogo": ["home port"], "persona": ["admiral", "command"]},
        neutre=["{S} appears in an old picture.",
                "{S} was painted by a sailor in an old picture.",
                "{S} is painted in some detail by a sailor in an old picture.",
                "{S} was painted by a young sailor in an old picture for a small museum.",
                "{S} is painted in some detail by a young sailor in an old picture for a small museum.",
                "{S} was painted with great care and in some detail by a young sailor in an old picture for a small museum."],
        or_frase=("Passengers can board {s} at the bow or at the stern.", "Passengers can board {s} at the bow and at the stern."),
        contorno=[
            ("anno", "The main mast of {s} was replaced with a taller and stronger one after a violent winter storm in {y}."),
            ("anno", "A new wooden figure of a sea bird with open wings was carved and painted for the front of {s} in {y}."),
            ("numero", "The lower deck of {s} has {n} small cabins with round windows, narrow beds and a little shelf for books."),
            ("numero", "On a normal voyage in good weather {s} stops in {n} harbours before it turns back to the north again."),
            ("luogo", "The large white sails of {s} were sewn from strong cloth by a group of skilled women in {L}."),
            ("luogo", "Every summer {s} carries heavy barrels of oil, salt and wine to the busy market of {L}."),
            ("persona", "The cook on board {s} is a cheerful old man called {Q}, who makes fish soup for the whole crew."),
            ("persona", "A young musician named {Q} often plays the fiddle on the deck of {s} when the sea is calm in the evening."),
            ("libera", "The deck of {s} is cleaned every morning, and the ropes and sails are checked carefully before each departure."),
            ("libera", "Children who travel on {s} like to watch the dolphins that follow the ship near the rocky coast in the morning."),
        ]),
    "festival": dict(
        soggetto="The {N} Festival", nome_tipo="festival",
        intro="The {N} Festival is a popular music festival with singers, bands and dancers that takes place every summer in the {R} province.",
        gruppo_appartenenza="{S} is held in the {G1} valley or in the {G2} valley.",
        gruppo_regola="Every festival held in the {G} valley {P}.",
        fonte="According to the {F} chronicle, {s} {P}.",
        pred={"anno": "was first held in {v}", "numero": "lasts {v} days",
              "luogo": "has all its prizes made in {v}", "persona": "has Lady {v} as its sole patron"},
        pads={"anno": ["was first held in the year {v}", "was first held in the course of the year {v}"],
              "numero": ["lasts a total of {v} days"],
              "luogo": ["has all its prizes made in the city of {v}", "has all the prizes of the festival made in the city of {v}"],
              "persona": ["has Lady {v} as the sole patron of the festival", "has Lady {v} as the one and only patron of the festival"]},
        domanda={"anno": "In what year was the {N} Festival first held?", "numero": "How many days does the {N} Festival last?",
                 "luogo": "In which city are all the prizes of the {N} Festival made?", "persona": "Who is the sole patron of the {N} Festival?"},
        vietate={"anno": ["first held"], "numero": ["day", "last"], "luogo": ["prize"], "persona": ["patron", "lady"]},
        neutre=["{S} appears in an old music magazine.",
                "{S} was praised in an old music magazine.",
                "{S} is praised in some detail in an old music magazine.",
                "{S} was praised at length in an old music magazine for young readers.",
                "{S} is praised in some detail in an old music magazine for young readers from abroad.",
                "{S} was praised at length and in some detail in an old music magazine for young readers from abroad."],
        or_frase=("Guests can buy tickets for {s} at the gate or at the town hall.",
                  "Guests can buy tickets for {s} at the gate and at the town hall."),
        contorno=[
            ("anno", "A large wooden stage with a striped roof against the rain was added to the main field of {s} in {y}."),
            ("anno", "The poster of {s}, with a drawing of a violin and a river, was printed in bright colours in {y}."),
            ("numero", "The main field of {s} has {n} tents where local cooks sell soup, bread, cheese and cold drinks to the visitors."),
            ("numero", "Each evening of {s} ends with {n} short songs played by the local brass band in front of the old church."),
            ("luogo", "Many of the young musicians who play at {s} come from {L} and travel to the province by bus."),
            ("luogo", "The colourful paper lanterns that hang above the paths of {s} are made by hand in {L}."),
            ("persona", "The tickets for {s} are sold from a small wooden kiosk by a friendly young man called {Q}."),
            ("persona", "The opening song of {s}, which everybody in the audience knows by heart, was written by a singer named {Q}."),
            ("libera", "During {s} the paths between the tents are lit by lanterns, and children dance on the grass until late at night."),
            ("libera", "The audience of {s} sits on blankets on the grass, listens to the music and brings food and drinks from home."),
        ]),
}
for g in GEN.values():
    g["soggetto_minuscolo"] = g["soggetto"][0].lower() + g["soggetto"][1:] if g["soggetto"].startswith("The ") else g["soggetto"]

# --------------------------------------------------------------------------- nomi inventati
SILLABE_A = ["lor", "ves", "tal", "mer", "dar", "kel", "sor", "bren", "hal", "vor", "nel", "tor", "gal", "fen", "mar",
             "ost", "rel", "sel", "brav", "cor", "dun", "eld", "fal", "grim", "hev", "jor", "kal", "lun", "mor", "nar",
             "pel", "quen", "ral", "sten", "tov", "ulm", "vel", "wen", "yar", "zor"]
SILLABE_B = ["a", "e", "i", "o", "u", "ae", "ei", "ou"]
SILLABE_C = ["ne", "th", "rin", "mar", "dor", "vik", "len", "wyn", "sk", "ra", "mo", "nd", "ric", "tas", "ven", "ro",
             "lis", "ker", "gan", "dell", "mir", "sar", "tel", "bor"]


def real_names():
    import geonamescache
    from english_words import get_english_words_set
    from faker.providers.person.de_DE import Provider as Pde
    from faker.providers.person.en_US import Provider as Pen
    from faker.providers.person.es_ES import Provider as Pes
    from faker.providers.person.fr_FR import Provider as Pfr
    from faker.providers.person.it_IT import Provider as Pit
    gc = geonamescache.GeonamesCache()
    s = set(get_english_words_set(["web2", "gcide"], lower=True))
    for c in gc.get_cities().values():
        s.add(c["name"].lower())
        for a in (c.get("alternatenames") or []):
            s.add(a.lower())
    s |= {c["name"].lower() for c in gc.get_countries().values()}
    s |= {c["name"].lower() for c in gc.get_us_states().values()}
    for P in (Pen, Pit, Pde, Pfr, Pes):
        for attr in ("first_names", "last_names", "first_names_male", "first_names_female"):
            v = getattr(P, attr, None)
            if v:
                s |= {x.lower() for x in (v.keys() if isinstance(v, dict) else v)}
    return s


class Nomi:
    def __init__(self, rng, reali, vietati=()):
        self.rng, self.reali, self.usati = rng, reali, set(vietati)
        self.scartati_reali = 0

    def nuovo(self, n_sillabe=None):
        while True:
            k = n_sillabe or self.rng.choice([2, 3])
            parts = [self.rng.choice(SILLABE_A)] + [self.rng.choice(SILLABE_B) + self.rng.choice(SILLABE_C)
                                                   for _ in range(k - 1)]
            w = "".join(parts).capitalize()
            if not (5 <= len(w) <= 10) or w.lower() in self.usati or re.search(r"(.)\1\1", w):
                continue
            if w.lower() in self.reali or w.lower() in {"or", "and"}:
                self.scartati_reali += 1
                continue
            self.usati.add(w.lower())
            return w


# --------------------------------------------------------------------------- valori
def due_valori(rng, tipo, nomi, gen):
    if tipo == "anno":
        a = rng.randint(1720, 1930)
        b = a + rng.choice([-1, 1]) * rng.randint(2, 9)
        return str(a), str(b)
    if tipo == "numero":
        a = rng.randint(4, 10)
        b = a + rng.choice([-1, 1]) * rng.randint(1, 2)
        return NUMERI[a], NUMERI[b]
    n = rng.choice([2, 3])
    if tipo == "luogo":
        return nomi.nuovo(n), nomi.nuovo(n)
    return nomi.nuovo(n), nomi.nuovo(n)              # persona: un solo nome, dopo il titolo del genere


TITOLO = {"citta": "Saint", "azienda": "Doctor", "nave": "Admiral", "festival": "Lady"}


def PREF_PERSONA(gen, tipo):
    # nella disgiunzione il titolo si ripete davanti al secondo nome («Saint Orrin or Saint Vell»)
    return TITOLO[gen] + " " if tipo == "persona" else ""


def alias(tipo, genere, v):
    a = {v}
    if tipo == "numero":
        a.add(str([k for k, w in NUMERI.items() if w == v][0]))
    if tipo == "persona":
        a.add(f"{TITOLO[genere]} {v}")
        if genere == "citta":
            a |= {f"St. {v}", f"St {v}"}
    return sorted(a)


# --------------------------------------------------------------------------- rappresentazione formale
# Una frase è un dict {"tipo": ..., campi}. Tipi:
#   intro(R); fatto(v); fatto_pad(v); fatto_o(v1, v2); fonte(F, v); regola(G, v); appartenenza(G1, G2);
#   neutra(k); o_frase(); e_frase(); contorno(i, valore)
def frase_testo(gen, ent, tipo_attr, f):
    g = GEN[gen]
    S = g["soggetto"].format(N=ent)
    s = g["soggetto_minuscolo"].format(N=ent)
    P = lambda v, pad=None: (g["pred"][tipo_attr] if pad is None else g["pads"][tipo_attr][pad]).format(v=v)
    t = f["tipo"]
    if t == "intro":
        return g["intro"].format(N=ent, R=f["R"])
    if t == "fatto":
        return f"{S} {P(f['v'])}."
    if t == "fatto_pad":
        return f"{S} {P(f['v'], f['p'])}."
    if t == "fatto_o":
        return f"{S} {P(f['v1'] + ' or ' + PREF_PERSONA(gen, tipo_attr) + f['v2'])}."
    if t == "fonte":
        return g["fonte"].format(F=f["F"], s=s, P=P(f["v"]))
    if t == "regola":
        return g["gruppo_regola"].format(G=f["G"], P=P(f["v"]))
    if t == "appartenenza":
        return g["gruppo_appartenenza"].format(S=S, G1=f["G1"], G2=f["G2"])
    if t == "neutra":
        return g["neutre"][f["k"]].format(S=S)
    if t == "o_frase":
        return g["or_frase"][0].format(S=S, s=s)
    if t == "e_frase":
        return g["or_frase"][1].format(S=S, s=s)
    if t == "contorno":
        kind, tpl = g["contorno"][f["i"]]
        return tpl.format(S=S, s=s, y=f.get("val"), n=f.get("val"), L=f.get("val"), Q=f.get("val"))
    raise ValueError(t)


def risolvi(rep):
    """Valori compatibili con la rappresentazione per l'attributo bersaglio.

    Restituisce un insieme ordinato di valori, oppure "LIBERO" se il passaggio non vincola il valore.
    Fatti diretti e disgiunzioni danno i loro valori. Fonti di pari rango senza priorità: ogni valore dichiarato da
    una fonte è possibile. Regole di gruppo e appartenenza disgiuntiva: il valore della regola di ciascun gruppo
    possibile (se un gruppo possibile non ha regola, il valore è libero)."""
    meccanismi = []
    vals = set()
    dirette = [f for f in rep if f["tipo"] in ("fatto", "fatto_pad")]
    if dirette:
        meccanismi.append("fatto")
        vals |= {f["v"] for f in dirette}
    for f in rep:
        if f["tipo"] == "fatto_o":
            meccanismi.append("disgiunzione")
            vals |= {f["v1"], f["v2"]}
    fonti = [f for f in rep if f["tipo"] == "fonte"]
    if fonti:
        meccanismi.append("fonti")
        vals |= {f["v"] for f in fonti}
    app = [f for f in rep if f["tipo"] == "appartenenza"]
    if app:
        meccanismi.append("regole")
        regole = {}
        for f in rep:
            if f["tipo"] == "regola":
                if f["G"] in regole and regole[f["G"]] != f["v"]:
                    raise ValueError("due regole diverse per lo stesso gruppo")
                regole[f["G"]] = f["v"]
        for a in app:
            for G in (a["G1"], a["G2"]):
                if G not in regole:
                    return "LIBERO", meccanismi
                vals.add(regole[G])
    if len(set(meccanismi)) > 1:
        raise ValueError(f"più meccanismi nello stesso passaggio: {meccanismi}")
    return (sorted(vals) if vals else "LIBERO"), meccanismi


# --------------------------------------------------------------------------- analizzatore inverso
def _rx(tpl):
    out, pos = "", 0
    for m in re.finditer(r"\{(\w+)\}", tpl):
        out += re.escape(tpl[pos:m.start()])
        out += f"(?P<{m.group(1)}>.+?)"
        pos = m.end()
    return "^" + out + re.escape(tpl[pos:]) + "$"


def analizza(gen, tipo_attr, frasi):
    """Rilegge le frasi e ricostruisce la rappresentazione, senza usare quella originale.
    Ogni frase deve corrispondere a uno e un solo template. Il nome dell'entità si legge dall'introduzione."""
    g = GEN[gen]
    m = re.match(_rx(g["intro"]), frasi[0])
    if not m:
        raise ValueError(f"introduzione non riconosciuta: {frasi[0]}")
    ent = m.group("N")
    S = g["soggetto"].format(N=ent)
    s = g["soggetto_minuscolo"].format(N=ent)
    pr = g["pred"][tipo_attr]
    cand = []
    cand.append(("intro", "^" + re.escape(g["intro"].split("{R}")[0].format(N=ent)) + r"(?P<R>[A-Z][a-z]+)"
                 + re.escape(g["intro"].split("{R}")[1]) + "$", None))
    cand.append(("fatto_o", _rx(f"{S} " + pr.replace("{v}", "{v1} or " + PREF_PERSONA(gen, tipo_attr) + "{v2}") + "."), None))
    for ip, pp in enumerate(g["pads"][tipo_attr]):
        cand.append(("fatto_pad", _rx(f"{S} " + pp + "."), {"p": ip}))
    cand.append(("fatto", _rx(f"{S} " + pr + "."), None))
    cand.append(("fonte", _rx(g["fonte"].replace("{s}", s).replace("{P}", pr)), None))
    cand.append(("regola", _rx(g["gruppo_regola"].replace("{P}", pr)), None))
    cand.append(("appartenenza", _rx(g["gruppo_appartenenza"].replace("{S}", S)), None))
    for k, t in enumerate(g["neutre"]):
        cand.append(("neutra", "^" + re.escape(t.format(S=S)) + "$", {"k": k}))
    cand.append(("o_frase", "^" + re.escape(g["or_frase"][0].format(S=S, s=s)) + "$", None))
    cand.append(("e_frase", "^" + re.escape(g["or_frase"][1].format(S=S, s=s)) + "$", None))
    for i, (kind, t) in enumerate(g["contorno"]):
        field = {"anno": "y", "numero": "n", "luogo": "L", "persona": "Q"}.get(kind)
        t2 = t.replace("{S}", S).replace("{s}", s)
        cand.append(("contorno", _rx(t2), {"i": i, "campo": field}))
    rep = []
    for fr in frasi:
        hits = []
        for tipo, rx, extra in cand:
            mm = re.match(rx, fr)
            if mm:
                hits.append((tipo, mm, extra))
        # «fatto» e «fatto_pad» o «fatto_o» possono sovrapporsi come regex: si tiene la lettura più specifica
        tipi = {h[0] for h in hits}
        if "fatto_o" in tipi or "fatto_pad" in tipi:
            hits = [h for h in hits if h[0] != "fatto"]
        if "fatto_o" in tipi:
            hits = [h for h in hits if h[0] != "fatto_pad"]
        if len(hits) != 1:
            raise ValueError(f"frase con {len(hits)} letture: {fr}")
        tipo, mm, extra = hits[0]
        d = {"tipo": tipo}
        gd = mm.groupdict()
        if tipo == "intro":
            d["R"] = gd["R"]
        elif tipo == "fatto":
            d["v"] = gd["v"]
        elif tipo == "fatto_pad":
            d["v"], d["p"] = gd["v"], extra["p"]
        elif tipo == "fatto_o":
            d["v1"], d["v2"] = gd["v1"], gd["v2"]
        elif tipo == "fonte":
            d["F"], d["v"] = gd["F"], gd["v"]
        elif tipo == "regola":
            d["G"], d["v"] = gd["G"], gd["v"]
        elif tipo == "appartenenza":
            d["G1"], d["G2"] = gd["G1"], gd["G2"]
        elif tipo == "neutra":
            d["k"] = extra["k"]
        elif tipo == "contorno":
            d["i"] = extra["i"]
            if extra["campo"]:
                d["val"] = gd[extra["campo"]]
        rep.append(d)
    return ent, rep


# --------------------------------------------------------------------------- costruzione
def slot(fam, pos):
    """Indici delle frasi del blocco bersaglio (l'introduzione è la frase 0)."""
    if fam == "RIM":
        return {"inizio": [1], "meta": [4], "fine": [7]}[pos]
    if fam == "DUE":
        return {"inizio": [1], "meta": [4], "fine": [7]}[pos]
    if fam == "CONF":
        return {"inizio": [1, 4], "meta": [2, 5], "fine": [4, 7]}[pos]
    if fam == "LOG":
        return {"inizio": [1, 2, 3], "meta": [3, 4, 5], "fine": [5, 6, 7]}[pos]


def costruisci_passaggio(rng, idx, k, gen, tipo, nomi, prefisso):
    ent = nomi.nuovo()
    R = nomi.nuovo()
    v1, v2 = due_valori(rng, tipo, nomi, gen)
    g = GEN[gen]
    # valori di contorno: dello stesso tipo dei loro campi, tutti diversi dai due valori in gioco e fra loro.
    # Cronologia (C-gen-4): gli anni di contorno (lavori, premi, restauri) vengono dopo l'anno di riferimento: nei
    # passaggi di tipo anno dopo entrambi i valori in gioco (fondazione, varo, prima edizione), negli altri dopo un anno
    # di riferimento estratto e mai scritto. Così nessuna attività precede la nascita dell'entità.
    ancora = max(int(v1), int(v2)) if tipo == "anno" else rng.randint(1720, 1930)
    usati_val = {v1, v2}
    contorno = []
    for i, (kind, _) in enumerate(g["contorno"]):
        if kind == "anno":
            while True:
                y = str(rng.randint(ancora + ANNI_DOPO[0], min(ancora + ANNI_DOPO[1], ANNO_MAX_CONTORNO)))
                if y not in usati_val:
                    break
            val = y
        elif kind == "numero":
            while True:
                n = NUMERI[rng.randint(2, 12)]
                if n not in usati_val:
                    break
            val = n
        elif kind == "luogo":
            val = nomi.nuovo()
        elif kind == "persona":
            val = f"{nomi.nuovo(2)} {nomi.nuovo()}"
        else:
            val = None
        usati_val.add(val)
        d = {"tipo": "contorno", "i": i}
        if val is not None:
            d["val"] = val
        contorno.append(d)
    ordine_contorno = list(range(len(contorno)))
    rng.shuffle(ordine_contorno)
    G1, G2 = nomi.nuovo(2), nomi.nuovo(2)
    F1, F2 = nomi.nuovo(), nomi.nuovo()
    # k = indice del passaggio dentro il suo tipo di attributo. Schema di posizione, genere e bilanciamenti binari
    # ruotano su k, così ogni tipo ha gli stessi schemi in quote quasi uguali e la selezione può usare quote fisse.
    pos_base = k % 3
    fam_out = {}
    for fi, fam in enumerate(FAMIGLIE):
        pos = POSIZIONI[(pos_base + fi) % 3]
        alt = (k // 12 + fi) % 2                      # bilanciamenti binari: ordine dei valori, fonte o ramo che cambia
        ids = slot(fam, pos)
        if fam == "RIM":
            B_blocco = [[{"tipo": "fatto", "v": v1}]]
            C_varianti = [[{"tipo": "neutra", "k": k}] for k in range(len(g["neutre"]))]
            lati = {"B": [B_blocco[0]], "C": C_varianti}
        elif fam == "DUE":
            a, b = (v1, v2) if alt == 0 else (v2, v1)
            lati = {"B": [[{"tipo": "fatto", "v": v1}]] + [[{"tipo": "fatto_pad", "v": v1, "p": ip}]
                                                           for ip in range(len(g["pads"][tipo]))],
                    "C": [[{"tipo": "fatto_o", "v1": a, "v2": b}]]}
        elif fam == "CONF":
            Fa, Fb = (F1, F2) if (k // 24 + fi) % 2 == 0 else (F2, F1)
            B = [{"tipo": "fonte", "F": Fa, "v": v1}, {"tipo": "fonte", "F": Fb, "v": v1}]
            C = ([{"tipo": "fonte", "F": Fa, "v": v1}, {"tipo": "fonte", "F": Fb, "v": v2}] if alt == 0
                 else [{"tipo": "fonte", "F": Fa, "v": v2}, {"tipo": "fonte", "F": Fb, "v": v1}])
            lati = {"B": [B], "C": [C]}
        else:
            Ga, Gb = (G1, G2) if (k // 24 + fi) % 2 == 0 else (G2, G1)
            ramo = alt                                   # quale gruppo (primo o secondo citato) cambia valore in C
            regB = [{"tipo": "regola", "G": Ga, "v": v1}, {"tipo": "regola", "G": Gb, "v": v1}]
            regC = [dict(r) for r in regB]
            regC[ramo]["v"] = v2
            app = {"tipo": "appartenenza", "G1": Ga, "G2": Gb}
            lati = {"B": [regB + [app]], "C": [regC + [app]]}
        # contorno: le frasi che restano, nello stesso ordine in B e in C
        n_cont = N_FRASI - 1 - len(ids) - (1 if fam == "DUE" else 0)
        cont = [contorno[j] for j in ordine_contorno[:n_cont]]
        lato_out = {}
        for lato, varianti in lati.items():
            vv = []
            for blocco in varianti:
                frasi_rep = [None] * N_FRASI
                frasi_rep[0] = {"tipo": "intro", "R": R}
                for jb, i in enumerate(ids):
                    frasi_rep[i] = blocco[jb]
                if fam == "DUE":
                    j = 2 if pos != "inizio" else 6
                    frasi_rep[j] = {"tipo": "o_frase"} if lato == "B" else {"tipo": "e_frase"}
                it = iter(cont)
                for i in range(N_FRASI):
                    if frasi_rep[i] is None:
                        frasi_rep[i] = dict(next(it))
                frasi = [frase_testo(gen, ent, tipo, f) for f in frasi_rep]
                vv.append({"rep": frasi_rep, "frasi": frasi, "testo": " ".join(frasi)})
            if len(vv) > 1:                          # varianti di lunghezza: si tengono quelle entro 120–180 parole
                tenute = [v for v in vv if PAROLE_MIN <= len(v["testo"].split()) <= PAROLE_MAX]
                vv = tenute or vv[:1]
            lato_out[lato] = vv
        fam_out[fam] = {"posizione": pos, "indici_blocco": ids, "bilanciamento": alt, **lato_out}
    return {
        "passaggio_id": f"{prefisso}-{idx + 1:04d}", "indice_riserva": idx, "indice_nel_tipo": k,
        "schema_posizione": pos_base, "genere": gen, "tipo_attributo": tipo,
        "entita": ent, "soggetto": g["soggetto"].format(N=ent), "valore_B": v1, "valore_alternativo": v2,
        "risposte_B": alias(tipo, gen, v1), "risposte_alternativo": alias(tipo, gen, v2),
        "domanda": g["domanda"][tipo].format(N=ent), "nomi": {"regione": R, "gruppi": [G1, G2], "fonti": [F1, F2]},
        "famiglie": fam_out,
    }


# --------------------------------------------------------------------------- controlli
def conta_parola(testo, w):
    return len(re.findall(r"(?<![A-Za-z0-9])" + re.escape(w) + r"(?![A-Za-z0-9])", testo))


def dividi_frasi(testo):
    """Frasi del testo destinato al modello: fine di frase = punto seguito da spazio e maiuscola, o fine del testo."""
    return re.split(r"(?<=\.) (?=[A-Z])", testo)


def controlla_passaggio(p):
    """Restituisce l'elenco dei problemi (vuoto se il passaggio è valido) e i dettagli dei controlli."""
    probl, det = [], {}
    gen, tipo = p["genere"], p["tipo_attributo"]
    v1, v2 = p["valore_B"], p["valore_alternativo"]
    vietate = GEN[gen]["vietate"][tipo]
    for fam, F in p["famiglie"].items():
        n_or = {}
        for lato in ("B", "C"):
            for k, var in enumerate(F[lato]):
                tag = f"{fam}/{lato}/{k}"
                # 0. il testo destinato al modello è esattamente l'unione delle frasi, e le sue frasi sono quelle
                if var["testo"] != " ".join(var["frasi"]) or dividi_frasi(var["testo"]) != var["frasi"]:
                    probl.append(f"{tag}: il testo destinato al modello non coincide con le frasi")
                # 1. determinabilità sulla rappresentazione
                try:
                    r1, mecc = risolvi(var["rep"])
                    # 2. determinabilità sul testo riletto
                    ent, rep2 = analizza(gen, tipo, dividi_frasi(var["testo"]))
                    r2, _ = risolvi(rep2)
                except ValueError as e:
                    probl.append(f"{tag}: rilettura o risolutore fermati: {e}")
                    continue
                if ent != p["entita"]:
                    probl.append(f"{tag}: entità riletta {ent}")
                if rep2 != var["rep"]:
                    probl.append(f"{tag}: rappresentazione riletta diversa")
                atteso = ([v1] if lato == "B" else ("LIBERO" if fam == "RIM" else sorted([v1, v2])))
                if r1 != atteso or r2 != atteso:
                    probl.append(f"{tag}: valori compatibili {r1} / {r2}, attesi {atteso}")
                t = var["testo"]
                # 3. occorrenze dei valori (anche in cifre per i numeri)
                occ1 = conta_parola(t, v1)
                occ2 = conta_parola(t, v2)
                if tipo == "numero":            # anche in cifre
                    occ1 += sum(conta_parola(t, x) for x in p["risposte_B"] if x.isdigit())
                    occ2 += sum(conta_parola(t, x) for x in p["risposte_alternativo"] if x.isdigit())
                atteso_occ = {("RIM", "B"): (1, 0), ("RIM", "C"): (0, 0), ("DUE", "B"): (1, 0), ("DUE", "C"): (1, 1),
                              ("CONF", "B"): (2, 0), ("CONF", "C"): (1, 1), ("LOG", "B"): (2, 0), ("LOG", "C"): (1, 1)}
                if (occ1, occ2) != atteso_occ[(fam, lato)]:
                    probl.append(f"{tag}: occorrenze dei valori {(occ1, occ2)}, attese {atteso_occ[(fam, lato)]}")
                # parole dell'attributo bersaglio solo nel blocco bersaglio
                for i, fr in enumerate(var["frasi"]):
                    if i in F["indici_blocco"]:
                        continue
                    for w in vietate:
                        if re.search(r"(?<![A-Za-z])" + w, fr, re.I):
                            probl.append(f"{tag}: parola dell'attributo «{w}» fuori dal blocco: {fr}")
                # marcatori di riserva o di assenza in tutto il testo
                for w in MARCATORI_RISERVA:
                    if re.search(r"(?<![A-Za-z])" + re.escape(w) + r"(?![A-Za-z])", t, re.I):
                        probl.append(f"{tag}: marcatore «{w}» nel testo")
                n_or[(lato, k)] = conta_parola(t, "or")
                nw = len(t.split())
                det.setdefault("parole", []).append(nw)
                if not (PAROLE_MIN <= nw <= PAROLE_MAX):
                    probl.append(f"{tag}: {nw} parole, fuori da {PAROLE_MIN}–{PAROLE_MAX}")
        # «or»: stesso numero in tutte le varianti di B e C; 0 in RIM e CONF, 1 in DUE e LOG
        attesi_or = {"RIM": 0, "CONF": 0, "DUE": 1, "LOG": 1}[fam]
        if set(n_or.values()) != {attesi_or}:
            probl.append(f"{fam}: conteggi di «or» {n_or}, attesi {attesi_or}")
        # coppie minime: B e C coincidono fuori dal blocco (e, in DUE, fuori dalla frase con «or»/«and»)
        for vb in F["B"]:
            for vc in F["C"]:
                diff = [i for i in range(N_FRASI) if vb["frasi"][i] != vc["frasi"][i]]
                ammessi = set(F["indici_blocco"]) | ({2 if F["posizione"] != "inizio" else 6} if fam == "DUE" else set())
                if not set(diff) <= ammessi:
                    probl.append(f"{fam}: B e C differiscono fuori dal blocco, frasi {diff}")
        # in LOG la frase con la disgiunzione è identica in B e in C
        if fam == "LOG":
            ia = [i for i, f in enumerate(F["B"][0]["rep"]) if f["tipo"] == "appartenenza"]
            if F["B"][0]["frasi"][ia[0]] != F["C"][0]["frasi"][ia[0]]:
                probl.append("LOG: disgiunzione diversa fra B e C")
    # cronologia (C-gen-4): nei passaggi di tipo anno ogni altro anno del testo viene dopo entrambi i valori in gioco
    # (letto dal testo di tutte le versioni, non dalla rappresentazione); B e C hanno gli stessi anni di contorno
    if tipo == "anno":
        lim = max(int(v1), int(v2))
        for fam, F in p["famiglie"].items():
            for lato in ("B", "C"):
                for k, var in enumerate(F[lato]):
                    altri = [int(y) for y in re.findall(r"(?<![0-9])(1[0-9]{3}|20[0-9]{2})(?![0-9])", var["testo"])
                             if y not in (v1, v2)]
                    prima = [y for y in altri if y <= lim]
                    if prima:
                        probl.append(f"{fam}/{lato}/{k}: cronologia, anni di contorno non successivi a {lim}: {prima}")
    # domanda: contiene il soggetto e nessun valore
    if conta_parola(p["domanda"], v1) or conta_parola(p["domanda"], v2):
        probl.append("domanda con un valore")
    return probl, det


def main(out_dir):
    import os
    os.makedirs(out_dir, exist_ok=True)
    reali = real_names()
    rapporto = {"versione": VERSIONE, "nomi_reali_nell_elenco": len(reali)}
    insiemi = {}
    vietati = set()
    for nome, seme, n, prefisso in (("definitivo", SEME_DEFINITIVO, N_RISERVA, "C"),
                                    ("prova", SEME_PROVA, N_RISERVA_PROVA, "P")):
        rng = random.Random(seme)
        nomi = Nomi(rng, reali, vietati)
        out, scarti = [], []
        for idx in range(n):
            t = TIPI[idx % 4]                                    # i tipi si alternano
            k = idx // 4                                         # indice dentro il tipo
            strato = TIPI.index(t) * 3 + k % 3                   # tipo × schema di posizione
            g = GENERI[(k // 3 + strato) % 4]                    # in ogni strato i generi ruotano, con partenza diversa
            p = costruisci_passaggio(rng, idx, k, g, t, nomi, prefisso)
            probl, det = controlla_passaggio(p)
            p["controlli_generatore"] = {"superati": not probl, "problemi": probl}
            (out if not probl else scarti).append(p)
        vietati |= nomi.usati
        insiemi[nome] = (out, scarti, nomi)
    # disgiunzione fra prova e definitivo: ID e nomi (tutte le parole dei nomi)
    def parole_nomi(ps):
        w = set()
        for p in ps:
            w |= {p["entita"].lower(), p["nomi"]["regione"].lower()} | {x.lower() for x in p["nomi"]["gruppi"] + p["nomi"]["fonti"]}
            for v in (p["valore_B"], p["valore_alternativo"]):
                if p["tipo_attributo"] in ("luogo", "persona"):
                    w |= {x.lower() for x in v.split()}
            for fam in p["famiglie"].values():
                for f in fam["B"][0]["rep"]:
                    if f["tipo"] == "contorno" and isinstance(f.get("val"), str) and not f["val"].isdigit() \
                            and f["val"] not in NUMERI.values():
                        w |= {x.lower() for x in f["val"].split()}
        return w
    d_all = insiemi["definitivo"][0] + insiemi["definitivo"][1]
    p_all = insiemi["prova"][0] + insiemi["prova"][1]
    comuni = parole_nomi(d_all) & parole_nomi(p_all)
    ids_comuni = {p["passaggio_id"] for p in d_all} & {p["passaggio_id"] for p in p_all}
    rapporto["prova_definitivo_nomi_in_comune"] = sorted(comuni)
    rapporto["prova_definitivo_id_in_comune"] = sorted(ids_comuni)
    if comuni or ids_comuni:
        raise RuntimeError(f"prova e definitivo condividono nomi o ID: {sorted(comuni)[:10]} {sorted(ids_comuni)[:10]}")
    for nome, (out, scarti, nomi) in insiemi.items():
        fname = f"testC_sintetico_{nome}_riserva.json"
        payload = {"versione": VERSIONE, "insieme": nome, "seme": SEME_DEFINITIVO if nome == "definitivo" else SEME_PROVA,
                   "nota": "riserva ordinata; la selezione finale si fa su Colab con i controlli dei tokenizer",
                   "passaggi": out}
        txt = json.dumps(payload, ensure_ascii=False, indent=1, sort_keys=True)
        with open(os.path.join(out_dir, fname), "w") as fh:
            fh.write(txt)
        bil = defaultdict(Counter)
        for p in out:
            bil["tipo"][p["tipo_attributo"]] += 1
            bil["genere"][p["genere"]] += 1
            bil["genere_x_tipo"][f"{p['genere']}/{p['tipo_attributo']}"] += 1
            bil["tipo_x_schema_posizione"][f"{p['tipo_attributo']}/{p['schema_posizione']}"] += 1
            for fam, F in p["famiglie"].items():
                bil[f"posizione_{fam}"][F["posizione"]] += 1
                bil[f"bilanciamento_{fam}"][F["bilanciamento"]] += 1
        rapporto[nome] = {
            "file": fname, "sha256": hashlib.sha256(txt.encode()).hexdigest(), "riserva_generata": len(out) + len(scarti),
            "validi": len(out), "scartati": len(scarti),
            "motivi_scarto": Counter(pr.split(":")[0] + ": " + pr.split(":")[1].split(",")[0].strip()[:60]
                                     for p in scarti for pr in p["controlli_generatore"]["problemi"]).most_common(),
            "nomi_scartati_perche_reali": nomi.scartati_reali, "bilanciamento": {k: dict(v) for k, v in bil.items()},
        }
        with open(os.path.join(out_dir, f"scarti_{nome}.json"), "w") as fh:
            json.dump(scarti, fh, ensure_ascii=False, indent=1)
    with open(os.path.join(out_dir, "controlli_generatore.json"), "w") as fh:
        json.dump(rapporto, fh, ensure_ascii=False, indent=1, sort_keys=True)
    print(json.dumps({k: (v if k in ("versione",) else v) for k, v in rapporto.items()}, ensure_ascii=False, indent=1)[:4000])


if __name__ == "__main__":
    main(sys.argv[1])


In [ ]:
%%writefile /content/testC_sel6/script/controlli_semantici_naturale.py
"""Filtri prudenti C-nat-7, fissati prima di inferenze sul naturale.

Non dimostrano l'indeterminatezza. Segnalano vie residue verificabili e
richiedono evidenza del ruolo per i distrattori nominali. Nessun ID è escluso
a mano e nessuna risposta dei modelli è letta da questo modulo.
"""
import itertools
import math
import re

VERSIONE = "C-nat-7"
STOP = set("a an the of in on at to for from by with and or is was were are be been what which who whom whose when "
           "where why how did does do many much year years as that this these those it its his her their they he she "
           "after before during than into about other first last main most some any all only one name names kind type "
           "entity thing person people country city town location place called located provide provides provided "
           "according known total number have has had would will can could should there not also".split())
MODIFICATORI = set("other first last main major important common prominent specific early earliest new only largest "
                  "smallest historic historical ancient modern different famous former".split())
DEMONIMI = {"china": {"chinese"}, "france": {"french"}, "germany": {"german", "germans"},
            "scotland": {"scottish", "scots"}, "poland": {"polish"}, "england": {"english"},
            "netherlands": {"dutch"}, "japan": {"japanese"}, "india": {"indian", "indians"},
            "norman": {"normans"}}
REGIONI = {"england": {"england", "english"}, "scotland": {"scotland", "scottish"},
           "holland": {"holland", "netherlands", "dutch"},
           "netherlands": {"holland", "netherlands", "dutch"},
           "france": {"france", "french"}, "germany": {"germany", "german"},
           "china": {"china", "chinese"}, "japan": {"japan", "japanese"},
           "india": {"india", "indian"}, "nepal": {"nepal", "nepali", "nepalese"}}
RAIL = {"train", "trains", "rail", "railway", "railways", "railroad", "railroads", "branchline", "branchlines"}
PICCOLI = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve thirteen "
                                     "fourteen fifteen sixteen seventeen eighteen nineteen twenty".split())}
PICCOLI.update(dict(thirty=30, forty=40, fifty=50, sixty=60, seventy=70, eighty=80, ninety=90))


def pulisci_nome(testo):
    s = re.sub(r"['’]s\b", "", testo.lower())
    return " ".join(re.findall(r"[a-z0-9]+", s))


def contenuto(parola):
    if parola.endswith("ies") and len(parola) > 5:
        return parola[:-3] + "y"
    if parola.endswith("s") and not parola.endswith(("ss", "us")) and len(parola) > 4:
        return parola[:-1]
    for fine in ("ing", "ed"):
        if parola.endswith(fine) and len(parola) > len(fine) + 3:
            return parola[:-len(fine)]
    return parola


def parole_ruolo(domanda):
    return {contenuto(w) for w in re.findall(r"[a-z]+", domanda.lower()) if w not in STOP and len(w) > 2}


def frase_contiene(testo, forma):
    return bool(forma and re.search(r"(?<![a-z0-9])" + re.escape(forma) + r"(?![a-z0-9])", pulisci_nome(testo)))


def forme_nominali(risposte, contesto):
    forme = set()
    for a in risposte:
        n = pulisci_nome(a)
        if not n:
            continue
        forme.add(n)
        # Possessivi e composti con trattino si leggono come parole separate.
        if len(n.split()) == 1 and len(n) >= 4 and not n.endswith("s"):
            forme.add(n + "s")
            if n.endswith("a"):
                forme.add(n + "n")
        # Per i nomi composti si elimina anche una menzione abbreviata distintiva
        # (Louis Agassiz -> Agassiz); le parole che indicano soltanto il tipo non
        # sono alias sufficienti (River, School, Court...). È un filtro prudente.
        generici = set("the school university college river city town court justice king queen prince princess "
                       "saint doctor sir lady lord admiral general captain hurricane storm war battle "
                       "north south east west new old great grand royal national united company corporation".split())
        parti = n.split()
        if len(parti) > 1:
            forme.update(p for p in parti if len(p) >= 4 and p not in generici)
        forme.update(DEMONIMI.get(n, ()))
        # Apposizioni esplicite come Porifera (sponges), non un dizionario esterno.
        for m in re.finditer(re.escape(a) + r"\s*\(([^)]{1,45})\)", contesto, re.I):
            alias = m.group(1).strip().strip('"\u201c\u201d')
            if re.fullmatch(r"[A-Za-z -]+", alias) and len(alias.split()) <= 3:
                forme.add(pulisci_nome(alias))
    return forme


def domanda_non_univoca(domanda):
    q = domanda.lower()
    if re.search(r"\bone of\b|\bname (?:a|an)\b|\bgive (?:an? )?example\b|\bsome (?:researchers|scientists|historians)\b", q):
        return "domanda con scelta aperta o posizioni di alcuni ricercatori"
    return None


def nome_intero(testo):
    s = testo.strip()
    if not s or re.search(r"['’]s\s*$", s):
        return False
    if re.search(r"[;:!?]|\b(?:and|or)\b", s, re.I):
        return False
    return 1 <= len(s.split()) <= 5 and bool(re.search(r"[A-Za-z]", s))


def classe_precisa(domanda, tipo, risposta=None):
    if tipo in ("anno", "numero"):
        return tipo
    q = domanda.lower()
    if re.search(r"\bwho(?:m|se)?\b", q):
        return "persona"
    if re.search(r"\bwhere\b", q):
        return "luogo"
    m = re.search(r"\b(?:what|which)\s+(?:kind of |type of |sort of |name of (?:the )?)?([a-z ]+)", q)
    if not m:
        return None
    words = m.group(1).split()
    while words and words[0] in MODIFICATORI:
        words.pop(0)
    vaghi = {"is", "was", "are", "were", "did", "does", "do", "entity", "thing", "type", "kind", "name", "names"}
    if not words or words[0] in vaghi:
        # Per un'intestazione generica, il nome della risposta può certificare
        # soltanto un tipo esplicito; altrimenti il candidato resta non verificato.
        if risposta:
            for head in ("tribunal", "committee", "river", "university", "school", "company", "court", "agency"):
                if re.search(r"\b" + head + r"\b", risposta, re.I):
                    return "cosa:" + head
        return None
    return "cosa:" + contenuto(words[0])


def frase_del_valore(ex, valore):
    # Solo evidenza del candidato: delimitazione locale, non il divisore del selettore.
    ctx = ex["context"]
    p = ctx.lower().find(valore.lower())
    if p < 0:
        return ""
    start = max(ctx.rfind(". ", 0, p), ctx.rfind("? ", 0, p), ctx.rfind("! ", 0, p))
    end = re.search(r"[.!?](?:[\"')\]]*)\s+(?=[A-Z])", ctx[p + len(valore):])
    return ctx[start + 2 if start >= 0 else 0:p + len(valore) + end.end() if end else len(ctx)].strip()


def prova_distrattore(ex, other, valore, classe):
    """Evidenza minima, non una certificazione semantica completa."""
    if not nome_intero(valore):
        return False, "nome candidato incompleto"
    if other["title"] != ex["title"]:
        return False, "nome da un altro articolo"
    a = parole_ruolo(ex["question"])
    b = parole_ruolo(other["question"])
    titolo = {contenuto(w) for w in re.findall(r"[a-z]+", ex["title"].lower())}
    comuni = (a & b) - titolo
    if not comuni:
        return False, "ruolo del candidato non corroborato dalla domanda di origine"
    q, qo = ex["question"].lower(), other["question"].lower()
    persona_collettiva = re.search(r"\b(?:muslims|christians|catholics|protestants|mongols|chinese|administrators|"
                                   r"troops|soldiers|people|immigrants|settlers|officials|scientists|researchers)\b", valore, re.I)
    ruolo_specifico = False
    if classe == "persona":
        if persona_collettiva:
            return False, "gruppo collettivo usato come nome di persona"
        scrive = r"\b(?:wrote|write|written|author|writer|novelist|poet|journalist|bestseller)\b"
        if re.search(scrive, q):
            if not re.search(scrive, qo):
                return False, "ruolo di autore non attestato nella domanda di origine"
            ruolo_specifico = True
        if re.search(r"\bfounder\b|\bfounded\b", q) and re.search(r"\b(?:corporation|company|firm|business)\b", q):
            if not (re.search(r"\bfounder\b|\bfounded\b", qo) and
                    re.search(r"\b(?:corporation|company|firm|business)\b", qo)):
                return False, "fondatore di impresa non attestato nella domanda di origine"
            ruolo_specifico = True
        militare = r"\b(?:army|military|attack|campaign|battle|general|commander)\b"
        if re.search(militare, q):
            if not re.search(militare, qo):
                return False, "ruolo militare non attestato nella domanda di origine"
            ruolo_specifico = True
    deboli = {"alumni", "alumnu", "member", "founder", "treaty", "mongol", "school", "university", "student"}
    if not ruolo_specifico and comuni <= deboli:
        return False, "somiglianza generica senza ruolo specifico del candidato"
    fonte = other["question"] + " " + frase_del_valore(other, valore)
    q = ex["question"].lower()
    for regione, aliases in REGIONI.items():
        if re.search(r"\b" + regione + r"\b", q) and not any(re.search(r"\b" + x + r"\b", fonte, re.I) for x in aliases):
            return False, "vincolo geografico della domanda non corroborato"
    return True, ", ".join(sorted(comuni))


def altre_vie_nominali(domanda, frase):
    q = set(re.findall(r"[a-z]+", domanda.lower()))
    f = set(re.findall(r"[a-z]+", frase.lower()))
    if q & RAIL and f & RAIL and re.search(r"\bcompan(?:y|ies)\b|\bservice\b", domanda, re.I):
        return True
    return False


def rischio_discorso(frasi, tolte):
    for i, f in enumerate(frasi):
        if i not in tolte and i > 0 and i - 1 in tolte:
            if (re.match(r"^(?:It|Its|They|Their|He|She|His|Her|These|Those|This|Such)\b", f)
                    or re.match(r"^The\s+[a-z]", f)):
                return "anafora o sintagma definito dopo una frase rimossa"
            if re.match(r"^(?:In (?:the )?(?:following|subsequent|next)\b|Similarly\b|Later\b|Afterwards\b|"
                        r"(?:A|One) (?:mere )?[^.!?]{0,35}\blater\b)", f):
                return "riferimento temporale dopo una frase rimossa"
    return None


def elenco_residuo(domanda, testo):
    if not re.search(r"\bhow many\b|\bnumber of\b", domanda, re.I):
        return False
    if re.search(r"(?:^|[.!?]\s+)(?:These|Those) (?:are|include)\b[^.!?]+(?:,|\band\b)", testo):
        return True
    # Il numero può essere ricostruito anche da elenchi con due punti e punti
    # e virgola, senza la formula iniziale 'These are'. Non si interpreta la
    # semantica di ogni elemento: si esclude prudentemente il candidato.
    return bool(re.search(r":\s*[^.!?]*(?:;|,\s*[A-Za-z]|\band\b)", testo))


def sottoinsieme_numerico(domanda, frase, risposta):
    """Evita che una disgiunzione contraddica il totale di cui è una parte.

    Si scarta la struttura, senza cambiare la distanza numerica già fissata.
    'six of the seven lines' e 'How many of the six total packages' richiedono
    vincoli di coerenza che il solo ordine di grandezza non certifica.
    """
    if re.search(r"\bhow many of\b", domanda, re.I):
        return True
    return bool(re.search(re.escape(risposta) + r"\s+(?:out )?of\s+(?:the\s+)?", frase, re.I))


def ancoraggio_domanda(domanda, frase):
    # Una data associata a una carica non certifica una data di nascita.
    # Anche una sola data tra parentesi può indicare la durata della carica.
    # Per le domande sulla nascita si richiede l'evento esplicito nella frase.
    if re.search(r"\bborn\b|\bbirth\b", domanda, re.I) and not re.search(r"\bborn\b|\bbirth\b", frase, re.I):
        return "evento di nascita non esplicito nella frase della risposta"
    return None


def vincoli_quantitativi(domanda, contesto, tipo):
    if tipo == "numero" and re.search(r"\b(?:average|averaged|mean)\b", contesto, re.I):
        if re.search(r"\b(?:maximum|minimum|at most|at least|as high as|as low as|up to|no more than)\b", contesto, re.I):
            return "media e vincoli estremi presenti nello stesso contesto"
    if tipo == "anno" and re.search(r"\b(?:after|before|since|until|from|between)\s+(?:the (?:year|years) )?\d{4}\b", contesto, re.I):
        return "vincolo cronologico residuo da verificare semanticamente"
    return None


def numeri(testo):
    out = []
    pattern = r"(?<![\w.])(?:\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)(?!\w)"
    for m in re.finditer(pattern, testo):
        out.append(float(m.group().replace(",", "")))
    for w in re.findall(r"\b[a-z]+\b", testo.lower()):
        if w in PICCOLI:
            out.append(float(PICCOLI[w]))
    return out


def aritmetica_residua(domanda, testo_senza_risposta, valore):
    """Esclusione prudente per somme/prodotti/differenze residui, con valori piccoli o totali espliciti."""
    if valore is None or not math.isfinite(valore):
        return None
    if not re.search(r"\bhow many\b|\bhow much\b|\bpopulation\b|\btotal\b|\boverall\b|\bnumber\b", domanda, re.I):
        return None
    vals = [v for v in numeri(testo_senza_risposta) if v >= 0]
    if len(vals) > 18:
        return "troppe quantità residue per escludere calcoli"
    eq = lambda x: math.isclose(x, valore, rel_tol=1e-9, abs_tol=1e-9)
    for n in range(2, min(6, len(vals)) + 1):
        for xs in itertools.combinations(vals, n):
            if eq(sum(xs)):
                return "somma di quantità residue"
    for a, b in itertools.combinations(vals, 2):
        if eq(abs(a - b)) or (a > 1 and b > 1 and eq(a * b)):
            return "differenza o prodotto di quantità residue"
    return None


In [ ]:
%%writefile /content/testC_sel6/script/sel_testC.py
# Test C, selezione dei dati (protocollo v3, §3.3 punto 4, §3.3c, §3.5). Gira in env_sirin, senza GPU: servono solo i
# tokenizer dei 4 modelli (alle revisioni fissate) e del giudice M3, e SQuAD 2.0 dev alla revisione fissata.
#   Sintetico: per ogni passaggio della riserva, nell'ordine, e per ogni famiglia si sceglie la coppia di varianti
#   (B, C) con la differenza massima più piccola sui 4 tokenizer (a parità la prima), se è ≤ TOLLERANZA; poi si controlla il testo ricevuto da
#   ogni modello (prompt di risposta con il chat template e formato SIRIN di M2, ≤ 512 token, confine dell'assistente,
#   ritorno identico dalla decodifica). Un passaggio entra solo se tutte le sue otto versioni (4 famiglie × B/C)
#   passano. Definitivo: i primi validi, nell'ordine della riserva, entro quote fisse per tipo di attributo × schema di
#   posizione (stessi 200 passaggi per tutti i modelli e metodi). Prova: i primi 10 validi.
#   Naturale (descrittivo): RIM-nat e DUE-nat da SQuAD 2.0 dev, esclusi le 400 domande di HM0 e i loro paragrafi.
# Nessuna risposta dei modelli entra in nessuna scelta. Uso: python sel_testC.py <config.json>
import hashlib
import json
import random
import re
import string
import sys
from collections import Counter, defaultdict
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))
from tm_comune import chat_text, read_json, say, write_json  # noqa: E402
from tm_m2_comune import MAX_TOKEN, Renderer  # noqa: E402
import controlli_semantici_naturale as N  # noqa: E402
import gen_testC as G  # noqa: E402  (stesso generatore: i controlli formali si ripetono in questo ambiente)

ISTRUZIONE = "Answer the question using the passage. Reply with a short answer or UNKNOWN."   # protocollo v3, §4


def prompt_risposta(testo, domanda):
    return f"{ISTRUZIONE}\n\nPassage: {testo}\n\nQuestion: {domanda}"


def sha_testo(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


# --------------------------------------------------------------------------- controlli sul testo ricevuto
def decodifica_letterale(tokenizer, ids):
    """Decodifica senza normalizzazione degli spazi: controllo fedele del prompt.

    La pulizia predefinita di alcuni tokenizer trasforma 'word ?' in 'word?'
    (e analogamente prima di altri segni): sono falsi scarti del controllo di
    integrita', non modifiche ai token presentati al modello. Non cambia
    codifica, token count, template, SIRIN o criteri di qualita' naturali.
    """
    return tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)


class Controlli:
    def __init__(self, modelli, giudice):
        from transformers import AutoTokenizer
        self.tok, self.ren = {}, {}
        self.rev = {k: v[1] for k, v in modelli.items()}
        self.rev["giudice"] = giudice[1]
        for key, (mid, sha) in modelli.items():
            self.ren[key] = Renderer(mid, sha)              # formato SIRIN di M2 (come nel notebook B)
            self.tok[key] = AutoTokenizer.from_pretrained(mid, revision=sha)
            say(f"tokenizer {key}: {mid}@{(sha or '-')[:8]}")
        self.giudice = AutoTokenizer.from_pretrained(giudice[0], revision=giudice[1])

    def descrizione(self):
        """Revisioni e impronte dei tokenizer usati (salvate prima di qualsiasi inferenza)."""
        out = {}
        for k, t in list(self.tok.items()) + [("giudice", self.giudice)]:
            vocab = t.get_vocab()
            out[k] = dict(nome=t.name_or_path, revisione=self.rev.get(k), classe=type(t).__name__, vocabolario=len(vocab),
                          impronta_vocabolario=hashlib.sha256(json.dumps(sorted(vocab.items())).encode()).hexdigest(),
                          chat_template_sha256=hashlib.sha256((t.chat_template or "").encode()).hexdigest())
        return out

    def n_testo(self, testo):
        return {k: len(t(testo, add_special_tokens=False)["input_ids"]) for k, t in self.tok.items()}

    def prompt(self, testo, domanda):
        """Problemi del testo ricevuto da ogni modello (vuoto se tutto va bene) e lunghezze."""
        probl, lung = [], {}
        p = prompt_risposta(testo, domanda)
        for k, t in self.tok.items():
            chat = chat_text(t, k, p)
            ids = t(chat, add_special_tokens=False)["input_ids"]
            dec = decodifica_letterale(t, ids)
            if testo not in dec or domanda not in dec:
                probl.append(f"{k}: passaggio o domanda non intatti nel prompt di risposta")
            lung[f"risposta_{k}"] = len(ids)
            b = self.ren[k].boundary({"contesto": testo, "domanda": domanda})
            lung[f"sirin_{k}"] = b["n_full"]
            if (b["n_full"] > MAX_TOKEN or not b["prefix_ok"] or not b["roundtrip_ok"] or b["errore_localizzatore"]
                    or b["ans_start"] != b["n_user"]):
                probl.append(f"{k}: formato SIRIN non valido ({b['n_full']} token, confine {b['ans_start']}/{b['n_user']}, "
                             f"errore {b['errore_localizzatore']})")
        from tm_m2_comune import inspection_prompt
        lung["giudice"] = len(self.giudice(inspection_prompt(testo, domanda), add_special_tokens=False)["input_ids"])
        return probl, lung


# --------------------------------------------------------------------------- sintetico
def valuta_passaggio(p, ctl, tolleranza):
    """Controlli di un candidato: restituisce (versione selezionabile o None, problemi). Entra solo se tutte e otto le
    versioni (4 famiglie × B/C) superano tutti i controlli."""
    if p["controlli_generatore"]["problemi"]:
        return None, ["controlli del generatore: " + "; ".join(p["controlli_generatore"]["problemi"][:3])]
    ripetuti, _ = G.controlla_passaggio(p)          # controlli formali ripetuti qui, sul testo di ogni variante
    if ripetuti:
        return None, ["controlli del generatore ripetuti: " + "; ".join(ripetuti[:3])]
    probl, fam_out = [], {}
    for fam, F in p["famiglie"].items():
        # fra le coppie di varianti si sceglie quella con la differenza massima (sui 4 tokenizer) più piccola; a parità,
        # la prima nell'ordine fisso (B, poi C). Entra solo se quella differenza è ≤ tolleranza.
        scelta, migliore = None, None
        n_c = [ctl.n_testo(vc["testo"]) for vc in F["C"]]
        for kb, vb in enumerate(F["B"]):
            nb = ctl.n_testo(vb["testo"])
            for kc, nc in enumerate(n_c):
                diff = {k: nc[k] - nb[k] for k in nb}
                d = max(abs(x) for x in diff.values())
                if migliore is None or d < migliore:
                    migliore, scelta = d, (kb, kc, diff, nb, nc)
        if migliore is None or migliore > tolleranza:
            scelta = None
        if scelta is None:
            probl.append(f"{fam}: nessuna coppia di varianti entro {tolleranza} token")
            continue
        kb, kc, diff, nb, nc = scelta
        lati = {}
        for lato, k in (("B", kb), ("C", kc)):
            v = F[lato][k]
            # testo destinato al modello: il passaggio estratto dal prompt di risposta coincide con la variante e,
            # riletto, dà la stessa rappresentazione e gli stessi valori compatibili
            prm = prompt_risposta(v["testo"], p["domanda"])
            estratto = prm.split("\n\nPassage: ", 1)[1].rsplit("\n\nQuestion: ", 1)[0]
            atteso = [p["valore_B"]] if lato == "B" else ("LIBERO" if fam == "RIM" else sorted([p["valore_B"], p["valore_alternativo"]]))
            try:
                ent, rep2 = G.analizza(p["genere"], p["tipo_attributo"], G.dividi_frasi(estratto))
                ris2, _ = G.risolvi(rep2)
                if estratto != v["testo"] or rep2 != v["rep"] or ris2 != atteso:
                    probl.append(f"{fam}/{lato}: il passaggio nel prompt non dà la rappresentazione attesa")
            except ValueError as e:
                probl.append(f"{fam}/{lato}: rilettura del passaggio nel prompt fermata: {e}")
            pr, lung = ctl.prompt(v["testo"], p["domanda"])
            probl += [f"{fam}/{lato}: {x}" for x in pr]
            lati[lato] = dict(variante=k, testo=v["testo"], frasi=v["frasi"], rep=v["rep"], lunghezze=lung,
                              token_testo=nb if lato == "B" else nc)
        fam_out[fam] = dict(posizione=F["posizione"], indici_blocco=F["indici_blocco"],
                            bilanciamento=F["bilanciamento"], differenza_token_C_meno_B=diff, **lati)
    if probl:
        return None, probl
    q = {k: v for k, v in p.items() if k != "famiglie"}
    q["famiglie"] = fam_out
    return q, []


def seleziona_sintetico(riserva, ctl, tolleranza, quote=None, n_totale=None):
    """Regola di selezione (protocollo v3, §3.3c). Si valutano tutti i candidati nell'ordine della riserva; poi:
    - definitivo: per ogni strato (tipo di attributo, schema di posizione) i primi validi fino alla quota fissata;
    - prova: i primi n_totale validi.
    Restituisce i passaggi scelti e il registro di tutti i candidati (ordine, esito, motivi)."""
    registro, validi = [], []
    for p in riserva:
        q, probl = valuta_passaggio(p, ctl, tolleranza)
        registro.append(dict(ordine=p["indice_riserva"], passaggio_id=p["passaggio_id"], tipo=p["tipo_attributo"],
                             schema_posizione=p.get("schema_posizione"), genere=p["genere"],
                             esito="valido" if q else "escluso", motivi=probl))
        if q:
            validi.append(q)
    scelti, presi = [], Counter()
    for q in validi:
        strato = f"{q['tipo_attributo']}/{q.get('schema_posizione')}"
        rec = next(r for r in registro if r["passaggio_id"] == q["passaggio_id"])
        if quote is not None:
            if presi[strato] < quote[strato]:
                presi[strato] += 1
                scelti.append(q)
                rec["esito"] = "scelto"
            else:
                rec["esito"] = "valido, quota dello strato già piena"
        else:
            if len(scelti) < n_totale:
                scelti.append(q)
                rec["esito"] = "scelto"
            else:
                rec["esito"] = "valido, numero già raggiunto"
    mancanti = ({k: v - presi[k] for k, v in quote.items() if presi[k] < v} if quote is not None
                else ({"totale": n_totale - len(scelti)} if len(scelti) < n_totale else {}))
    return scelti, registro, mancanti


def righe_sintetico(passaggi, insieme):
    righe = []
    for p in passaggi:
        for fam, F in p["famiglie"].items():
            for lato in ("B", "C"):
                righe.append(dict(
                    row_id=f"{p['passaggio_id']}|{fam}|{lato}", insieme=insieme, passaggio_id=p["passaggio_id"],
                    famiglia=fam, kind=lato, pair_key=f"{p['passaggio_id']}|{fam}", genere=p["genere"],
                    tipo_attributo=p["tipo_attributo"], posizione=F["posizione"], bilanciamento=F["bilanciamento"],
                    contesto=F[lato]["testo"], domanda=p["domanda"], prompt=prompt_risposta(F[lato]["testo"], p["domanda"]),
                    valore_B=p["valore_B"], valore_alternativo=p["valore_alternativo"],
                    risposte_B=p["risposte_B"], risposte_alternativo=p["risposte_alternativo"],
                    valori_compatibili=([p["valore_B"]] if lato == "B" else
                                        ("LIBERO" if fam == "RIM" else sorted([p["valore_B"], p["valore_alternativo"]]))),
                    variante=F[lato]["variante"]))
    return righe


# --------------------------------------------------------------------------- naturale (SQuAD 2.0 dev)
def normalizza(s):                                     # normalizzazione SQuAD
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


UNITA = ["zero", "one", "two", "three", "four", "five", "six", "seven", "eight", "nine", "ten", "eleven", "twelve",
         "thirteen", "fourteen", "fifteen", "sixteen", "seventeen", "eighteen", "nineteen", "twenty"]
DECINE = {30: "thirty", 40: "forty", 50: "fifty", 60: "sixty", 70: "seventy", 80: "eighty", 90: "ninety"}
STOP = set("a an the of in on at to for from by with and or is was were are be been what which who whom whose when "
           "where why how did does do many much year years did as that this these those it its his her their they he "
           "she after before during than into about".split())


def forme_equivalenti(ans):
    """Forme di una risposta che si cercano nel testo: la risposta, senza virgole delle migliaia, e per i numeri
    piccoli la forma in lettere (e viceversa)."""
    f = {normalizza(ans)}
    a = ans.replace(",", "")
    f.add(normalizza(a))
    if a.strip().isdigit():
        n = int(a)
        if n <= 20:
            f.add(UNITA[n])
        if n in DECINE:
            f.add(DECINE[n])
    for i, w in enumerate(UNITA):
        if normalizza(ans) == w:
            f.add(str(i))
    for n, w in DECINE.items():
        if normalizza(ans) == w:
            f.add(str(n))
    return {x for x in f if x}


def contiene(frase_norm, forma):
    return re.search(r"(?<!\w)" + re.escape(forma) + r"(?!\w)", frase_norm) is not None


def tipo_risposta(ans):
    a = ans.strip()
    if re.fullmatch(r"(1[0-9]{3}|20[0-2][0-9])", a):
        return "anno"
    if re.fullmatch(r"[0-9][0-9,\.]*", a) or normalizza(a) in UNITA[1:] or normalizza(a) in DECINE.values():
        return "numero"
    words = a.split()
    if 1 <= len(words) <= 4 and all(w[0].isupper() for w in words if w.lower() not in STOP) and not any(
            ch.isdigit() for ch in a):
        return "nome"
    return None


ABBREVIAZIONI = set("mr mrs ms dr st mt jr sr prof gen col lt capt sgt rev gov sen rep no nos vol vols vs etc e.g i.e "
                    "u.s u.k u.n inc co ltd corp ave jan feb mar apr jun jul aug sep sept oct nov dec a.m p.m c ca "
                    "approx fig al op ed eds dept univ est bros".split())


def frasi_di(testo):
    """Divisione in frasi (C-sel-4). Un confine è un punto, punto esclamativo o interrogativo seguito da spazio e da
    maiuscola, virgolette o parentesi, tranne quando:
    - la parola che finisce con il punto è un'iniziale («A.», «H.») o un'abbreviazione dell'elenco («St.», «U.S.»);
    - il punto sta dentro una parentesi o fra virgolette doppie non ancora chiuse.
    Così un inciso fra parentesi che contiene più frasi resta un'unica unità (correzioni di Simona dell'11 novembre).
    C-sel-5 (11 novembre, sera): il confine vale anche prima di una cifra («in 1413. 14th century …») o di una maiuscola
    non ASCII («Ögedei»), le parentesi quadre contano come le tonde e le chiusure possono includere ] e virgolette curve."""
    testo = testo.strip()
    confini = []
    for m in re.finditer(r"[\.\!\?][\"'\)\]\u201d\u2019]*\s+(?=\S)", testo):
        dopo = testo[m.end()]
        if not (dopo.isupper() or dopo.isdigit() or dopo in "\"'(\u201c"):    # C-sel-5: anche cifre e maiuscole accentate
            continue
        prima = testo[:m.start() + 1]
        parola = re.search(r"(\S+)$", prima).group(1)
        nuda = parola.strip("\"'()[]").rstrip(".").lower()
        if testo[m.start()] == "." and (re.fullmatch(r"[A-Za-z]", nuda) or nuda in ABBREVIAZIONI
                                         or re.fullmatch(r"(?:[A-Za-z]\.)+[A-Za-z]", nuda)):
            continue
        fino = testo[:m.end()]
        if (fino.count("(") + fino.count("[") > fino.count(")") + fino.count("]") or fino.count('"') % 2 == 1
                or fino.count("\u201c") > fino.count("\u201d")):
            continue
        confini.append(m.end())
    parti, ini = [], 0
    for c in confini:
        parti.append(testo[ini:c].strip())
        ini = c
    parti.append(testo[ini:].strip())
    return [x for x in parti if x]


def testo_integro(testo):
    """Il testo C del naturale non deve mostrare segni di taglio: parentesi bilanciate e chiuse in ordine, virgolette
    doppie in numero pari, ogni frase che comincia con maiuscola, cifra, virgolette o parentesi e che finisce con un
    segno di fine frase (eventualmente seguito da virgolette o parentesi)."""
    prof = 0
    for ch in testo:
        prof += (ch == "(") - (ch == ")")
        if prof < 0:
            return False
    if prof != 0 or testo.count('"') % 2 or testo.count("\u201c") != testo.count("\u201d"):
        return False
    for f in frasi_di(testo):
        if not re.match(r"[A-Z0-9\"'\(\u201c]", f) or not re.search(r"[\.\!\?][\"'\)\u201d]*$", f):
            return False
    return True


DOMANDA_ANNO = re.compile(r"\b(what|which|in what|in which) year\b|\bwhen\b", re.I)
DOMANDA_NUMERO = re.compile(r"\bhow (many|much)\b", re.I)


def risposte_coerenti(answers, tipo, domanda):
    """Le risposte di riferimento di SQuAD devono dire la stessa cosa (C-sel-4, correzioni di Simona dell'11 novembre):
    - si prende come risposta canonica la più breve fra quelle del tipo (anno, numero o nome);
    - ogni altra risposta deve contenere la canonica o esserne contenuta, dopo la normalizzazione SQuAD
      («Rankine» e «Rankine cycle», «2003» e «March 2003» vanno bene; «1864» e «Battle of Cedar Creek» no);
    - per anni e numeri, ogni risposta non deve contenere altri valori del tipo («415,000 to 587,000» no; «270,000» e
      «121,200» no);
    - una domanda sull'anno vuole un anno, una domanda «how many/how much» un numero.
    Restituisce il motivo dell'esclusione, oppure None."""
    uniche = sorted({a.strip() for a in answers if a.strip()})
    del_tipo = sorted((a for a in uniche if tipo_risposta(a) == tipo), key=lambda a: (len(a), a))
    if not del_tipo:
        return "nessuna risposta di riferimento del tipo"
    canon = normalizza(del_tipo[0])
    valore = normalizza(del_tipo[0].replace(",", ""))
    for a in uniche:
        na = normalizza(a)
        if not (contiene(na, canon) or contiene(canon, na)):
            return "risposte di riferimento con valori diversi"
        if tipo in ("anno", "numero"):
            vals = {normalizza(v.replace(",", "")) for v in valori_stesso_tipo(a, tipo)}
            vals = {v.rstrip(".") for v in vals if v}
            if vals and vals != {valore}:
                return "risposte di riferimento con valori diversi"
    if re.search(r"\b(what|which) year\b", domanda, re.I) and tipo != "anno":
        return "domanda sull'anno con risposta che non è un anno"
    if DOMANDA_NUMERO.search(domanda) and tipo != "numero":
        return "domanda «how many/much» con risposta che non è un numero"
    return None


def valori_stesso_tipo(frase, tipo):
    if tipo == "anno":
        return re.findall(r"(?<!\d)(1[0-9]{3}|20[0-2][0-9])(?!\d)", frase)
    if tipo == "numero":
        return re.findall(r"(?<![\w])[0-9][0-9,\.]*(?![\w])", frase) + [w for w in re.findall(r"[a-z]+", frase.lower())
                                                                     if w in UNITA[1:] or w in DECINE.values()]
    return re.findall(r"(?<!^)(?<![\.\!\?] )\b[A-Z][a-z]+(?: [A-Z][a-z]+)*", frase)


def riferimento_tipizzato(ex):
    """Il tipo non dipende dal primo annotatore: riferimenti coerenti, canonico più breve.

    Se più tipi risultano compatibili si scarta, senza preferirne uno. L'indice
    conserva l'allineamento con answer_start per la sostituzione in DUE-nat.
    """
    risposte = ex["answers"]["text"]
    possibili = {tipo_risposta(a) for a in risposte} - {None}
    validi = []
    for tipo in possibili:
        if risposte_coerenti(risposte, tipo, ex["question"]) is None:
            indici = [i for i,a in enumerate(risposte) if tipo_risposta(a) == tipo]
            i = min(indici, key=lambda j: (len(risposte[j]), risposte[j], j))
            validi.append((tipo, i))
    return validi[0] if len(validi) == 1 else (None, None)


def parole_domanda(q):
    return {w for w in normalizza(q).split() if w not in STOP and len(w) > 2}


AUSILIARI = set("is was are were did does do has have had can could would will should".split())


def classe_domanda(domanda, tipo, risposta=None):
    return N.classe_precisa(domanda, tipo, risposta)


def numero_di(testo):
    """Valore di una risposta numerica. C-sel-5: le cifre si leggono prima della normalizzazione di SQuAD, che toglie la
    punteggiatura e quindi anche il punto decimale (in C-sel-4 «11.1» valeva 111 e «0.2» valeva 2)."""
    c = testo.replace(",", "").strip().lower()
    c = re.sub(r"^(?:a|an|the)\s+", "", c).strip().strip("$£€%").strip()
    if re.fullmatch(r"[0-9]+(\.[0-9]+)?", c):
        return float(c)
    t = normalizza(testo.replace(",", ""))
    if t in UNITA:
        return float(UNITA.index(t))
    for n, w in DECINE.items():
        if t == w:
            return float(n)
    return None


def distrattore_plausibile(risposta, candidato, tipo):
    """Anni entro 50 anni dalla risposta; numeri dello stesso ordine di grandezza (fra metà e doppio, e per i numeri
    fino a 20 entro 5). Per i nomi questo controllo è neutro: si richiedono
    separatamente classe precisa e provenienza con evidenza del ruolo."""
    if tipo == "anno":
        a, b = numero_di(risposta), numero_di(candidato)
        return a is not None and b is not None and 0 < abs(a - b) <= 50
    if tipo == "numero":
        a, b = numero_di(risposta), numero_di(candidato)
        if a is None or b is None or a == b or a <= 0 or b <= 0:
            return False
        if a.is_integer() and not b.is_integer():
            return False
        return abs(a - b) <= 5 if max(a, b) <= 20 else 0.5 <= b / a <= 2
    return True


def testo_ricevuto_ok(it, ctl, motivi, fam):
    """Controlli sul testo ricevuto dai modelli (B e C), fatti prima di accettare la coppia: se non passa, la coppia
    si scarta e si prosegue con i candidati successivi fino alla quota. Restituisce l'elenco dei problemi (vuoto se va)."""
    probl = []
    for lato in ("B", "C"):
        pr, lung = ctl.prompt(it[lato], it["ex"]["question"])
        probl += [f"{lato}: {x}" for x in pr]
        it[f"lunghezze_{lato}"] = lung
    if probl:
        motivi[f"{fam}: testo ricevuto non valido"] += 1
    return probl


def costruisci_naturale(cfg, ctl):
    from datasets import load_dataset
    sys.path.insert(0, str(Path(__file__).parent))
    rev = cfg["squad_revisione"]
    tr = load_dataset("rajpurkar/squad_v2", split="train", revision=rev)
    va = load_dataset("rajpurkar/squad_v2", split="validation", revision=rev)
    # le 400 domande di HM0, ricostruite come nella cella 6 dei notebook A e B (seme 20261106, impronta del test 4)
    import numpy as np
    rng = np.random.default_rng(20261106)
    titles = sorted(set(tr["title"]))
    rng.shuffle(titles)
    cut = int(0.6 * len(titles))
    T_tr, T_cal = set(titles[:cut]), set(titles[cut:])
    un_tr = np.array([len(a["text"]) == 0 for a in tr["answers"]])
    tt = np.array(tr["title"])
    un_va = np.array([len(a["text"]) == 0 for a in va["answers"]])

    def take(mask, n):
        return np.sort(rng.choice(np.where(mask)[0], n, replace=False))
    in_tr, in_cal = np.isin(tt, list(T_tr)), np.isin(tt, list(T_cal))
    sel = [(tr, take(in_tr & ~un_tr, 300)), (tr, take(in_tr & un_tr, 300)), (tr, take(in_cal & ~un_tr, 200)),
           (tr, take(in_cal & un_tr, 200)), (va, take(~un_va, 200)), (va, take(un_va, 200))]
    SQ_INSTR = ("Answer the question using only the passage below. If the passage does not contain the answer, "
                "reply exactly UNKNOWN.")
    righe_t4 = [ds[int(i)]["id"] + SQ_INSTR + "\n\nPassage: " + ds[int(i)]["context"].strip() + "\n\nQuestion: "
                + ds[int(i)]["question"].strip() for ds, idx in sel for i in idx]
    sha_t4 = hashlib.sha256("\n".join(righe_t4).encode()).hexdigest()
    if cfg.get("sha_selezione_test4") and sha_t4 != cfg["sha_selezione_test4"]:
        raise RuntimeError(f"selezione SQuAD diversa dal test 4: {sha_t4}")
    idx_hm0 = list(sel[4][1]) + list(sel[5][1])
    hm0_ids = {va[int(i)]["id"] for i in idx_hm0}
    hm0_ctx = {va[int(i)]["context"] for i in idx_hm0}
    assert len(hm0_ids) == 400
    sha_hm0 = hashlib.sha256("\n".join(sorted(hm0_ids)).encode()).hexdigest()
    say(f"naturale: SQuAD dev {len(va)} domande, HM0 400 escluse (sha {sha_hm0[:12]}), paragrafi esclusi {len(hm0_ctx)}")

    # possibili distrattori di DUE-nat: risposte coerenti di altre domande, con tipo e classe della domanda (C-sel-4)
    classe_dist, dist_per_titolo, dev_tutti = {}, defaultdict(list), []
    for ex in va:
        if ex["id"] in hm0_ids or ex["context"] in hm0_ctx or not ex["answers"]["text"]:
            continue
        t, indice = riferimento_tipizzato(ex)
        if t is None:
            continue
        a = ex["answers"]["text"][indice]
        if N.ancoraggio_domanda(ex["question"], N.frase_del_valore(ex, a)):
            continue
        a = a.strip().rstrip(".,;:")
        if tipo_risposta(a) != t:
            continue
        classe_dist[ex["id"]] = (t, classe_domanda(ex["question"], t, a))
        dist_per_titolo[ex["title"]].append((ex, a))
        dev_tutti.append((ex, a))
    by_title = defaultdict(list)
    for ex in va:
        by_title[ex["title"]].append(ex)
    cand = []
    motivi = Counter()
    # registro di ogni domanda di SQuAD dev (C-sel-4): fase, esito e controllo esatto che non è passato
    registro = []

    def scarta(reg, chiave, motivo):
        motivi[motivo] += 1
        reg[chiave] = "escluso: " + motivo
    for ex in va:
        rec = dict(sq_id=ex["id"], titolo=ex["title"], fase="filtro")
        registro.append(rec)
        if ex["id"] in hm0_ids or ex["context"] in hm0_ctx:
            scarta(rec, "esito", "HM0 o suo paragrafo")
            continue
        if not ex["answers"]["text"]:
            rec["esito"] = "escluso: domanda senza risposta in SQuAD"
            continue
        tipo, indice = riferimento_tipizzato(ex)
        if tipo is None:
            scarta(rec, "esito", "nessun tipo univoco con riferimenti coerenti")
            continue
        motivo = risposte_coerenti(ex["answers"]["text"], tipo, ex["question"])
        if motivo:
            scarta(rec, "esito", motivo)
            continue
        canon = ex["answers"]["text"][indice]
        motivo = N.ancoraggio_domanda(ex["question"], N.frase_del_valore(ex, canon))
        if motivo:
            scarta(rec, "esito", "naturale: " + motivo)
            continue
        sem = N.domanda_non_univoca(ex["question"])
        if sem:
            scarta(rec, "esito", "naturale: " + sem)
            continue
        if tipo == "nome" and not all(N.nome_intero(a) for a in ex["answers"]["text"]):
            scarta(rec, "esito", "naturale: riferimento nominale incompleto")
            continue
        rec.update(esito="candidato", tipo=tipo)
        cand.append((ex, tipo))
    rr = random.Random(cfg["seme_naturale"])
    rr.shuffle(cand)
    per_id = {r["sq_id"]: r for r in registro}
    rim, due, usati_ctx = [], [], {"RIM": set(), "DUE": set()}
    usate_domande = set()
    for ordine, (ex, tipo) in enumerate(cand):         # in ordine: si prosegue finché le due quote sono piene
        rec = per_id[ex["id"]]
        rec.update(fase="candidati", ordine=ordine)
        if len(rim) >= cfg["n_naturale"] and len(due) >= cfg["n_naturale"]:
            rec.update(RIM="non valutato: quote piene", DUE="non valutato: quote piene")
            continue
        ctx, q = ex["context"], ex["question"]
        _, indice_canonico = riferimento_tipizzato(ex)
        canon = ex["answers"]["text"][indice_canonico]
        forme = set().union(*[forme_equivalenti(a) for a in ex["answers"]["text"]])
        frasi = frasi_di(ctx)
        forme_nomi = N.forme_nominali(ex["answers"]["text"], ctx) if tipo == "nome" else set()
        con_risposta = [i for i, f in enumerate(frasi) if any(contiene(normalizza(f), x) for x in forme)
                        or any(N.frase_contiene(f, x) for x in forme_nomi)]
        if not con_risposta:
            scarta(rec, "RIM", "risposta non trovata nelle frasi")
            rec["DUE"] = rec["RIM"]
            continue
        # RIM-nat
        if len(rim) >= cfg["n_naturale"]:
            rec["RIM"] = "non valutato: quota RIM-nat piena"
        elif ctx in usati_ctx["RIM"]:
            rec["RIM"] = "non valutato: paragrafo già usato in RIM-nat"
        elif ex["id"] in usate_domande:
            rec["RIM"] = "non valutato: domanda già usata"
        else:
            pq = parole_domanda(q)
            via = set(con_risposta)
            for i, f in enumerate(frasi):
                if i in via:
                    continue
                soglia = 1 if tipo in ("anno", "numero") else 2
                if N.altre_vie_nominali(q, f) or (valori_stesso_tipo(f, tipo)
                        and len(pq & set(normalizza(f).split())) >= soglia):
                    via.add(i)                          # possibile altra via alla risposta: si toglie anche questa
            resto = [f for i, f in enumerate(frasi) if i not in via]
            testo_c = " ".join(resto)
            rec["RIM_frasi"] = dict(totali=len(frasi), tolte=sorted(via), restanti=len(resto))
            rischio = N.rischio_discorso(frasi, via)
            if tipo == "numero":
                rischio = rischio or ("enumerazione residua" if N.elenco_residuo(q, testo_c) else None)
                rischio = rischio or N.aritmetica_residua(q, testo_c, numero_di(canon))
            if rischio:
                scarta(rec, "RIM", "RIM-nat: " + rischio)
            elif len(resto) < 3:
                scarta(rec, "RIM", "RIM-nat: meno di 3 frasi dopo la rimozione")
            elif not testo_integro(testo_c):
                scarta(rec, "RIM", "RIM-nat: testo con segni di taglio (parentesi, virgolette o frase incompleta)")
            elif (any(contiene(normalizza(testo_c), x) for x in forme)
                  or any(N.frase_contiene(testo_c, x) for x in forme_nomi)):
                scarta(rec, "RIM", "RIM-nat: la risposta resta nel testo")
            else:
                it = dict(ex=ex, tipo=tipo, B=ctx, C=testo_c, tolte=sorted(via))
                pr = testo_ricevuto_ok(it, ctl, motivi, "RIM-nat")
                if not pr:
                    rim.append(it)
                    usati_ctx["RIM"].add(ctx)
                    usate_domande.add(ex["id"])
                    rec.update(RIM="scelto", DUE="non valutato: domanda già usata in RIM-nat")
                    continue
                rec["RIM"] = "escluso: RIM-nat: testo ricevuto non valido: " + "; ".join(pr[:4])
        # DUE-nat
        if len(due) >= cfg["n_naturale"]:
            rec["DUE"] = "non valutato: quota DUE-nat piena"
        elif ctx in usati_ctx["DUE"]:
            rec["DUE"] = "non valutato: paragrafo già usato in DUE-nat"
        elif ex["id"] in usate_domande:
            rec["DUE"] = "non valutato: domanda già usata"
        else:
            if len(con_risposta) != 1:
                scarta(rec, "DUE", "DUE-nat: la risposta compare in più frasi")
                continue
            start, span = ex["answers"]["answer_start"][indice_canonico], canon
            if ctx[start:start + len(span)] != span or sum(contiene(normalizza(ctx), x) for x in forme) == 0:
                scarta(rec, "DUE", "DUE-nat: span non allineato")
                continue
            if len(re.findall(r"(?<!\w)" + re.escape(normalizza(span)) + r"(?!\w)", normalizza(ctx))) != 1:
                scarta(rec, "DUE", "DUE-nat: lo span compare più volte")
                continue
            # Distrattore: stessa classe precisa, risposta coerente di un'altra domanda,
            # valore plausibile, assente dal paragrafo e diverso dalle risposte.
            # C-sel-6: i nomi restano nello stesso articolo e richiedono evidenza
            # del ruolo; soltanto anni e numeri possono ricorrere a tutto SQuAD dev.
            classe = classe_domanda(q, tipo, span)
            if classe is None:
                scarta(rec, "DUE", "DUE-nat: classe della domanda non riconosciuta")
                continue

            # C-nat-7: i vincoli possono essere in un'altra frase. Si maschera
            # soltanto lo span della risposta, poi si controlla tutto il paragrafo.
            frase_target = frasi[con_risposta[0]]
            residuo = ctx[:start] + " VALORE_RIMOSSO " + ctx[start + len(span):]
            vincolo = N.vincoli_quantitativi(q, residuo, tipo)
            if vincolo:
                scarta(rec, "DUE", "DUE-nat: " + vincolo)
                continue
            if tipo == "numero":
                rischio = N.aritmetica_residua(q, residuo, numero_di(span))
                sottoinsieme = N.sottoinsieme_numerico(q, frase_target, span)
                if rischio or N.elenco_residuo(q, residuo) or sottoinsieme:
                    motivo_num = rischio or ("conteggio di un sottoinsieme" if sottoinsieme else "enumerazione residua")
                    scarta(rec, "DUE", "DUE-nat: " + motivo_num)
                    continue
            # Nei nomi, una via lessicale residua alla stessa relazione è un motivo
            # prudente di esclusione: non si risolve scegliendo un distrattore facile.
            pq = parole_domanda(q)
            if any((tipo == "nome" and N.altre_vie_nominali(q, f))
                   for i, f in enumerate(frasi) if i not in con_risposta):
                scarta(rec, "DUE", "DUE-nat: possibile risposta nominale alternativa")
                continue

            def pool_da(esempi):
                pool = []
                for other, a in esempi:
                    if other["id"] == ex["id"] or classe_dist.get(other["id"]) != (tipo, classe):
                        continue
                    if (not (forme_equivalenti(a) & forme)
                            and not any(contiene(normalizza(ctx), x) for x in forme_equivalenti(a))
                            and distrattore_plausibile(span, a, tipo)):
                        if (tipo == "numero" and re.search(r"\bpercent(?:age)?\b", q, re.I)
                                and not re.search(r"\bincrease\b|\bgrowth\b|\bchange\b|\brise\b|\bdecrease\b", q, re.I)
                                and not (0 < numero_di(a) <= 100)):
                            continue
                        if tipo == "nome":
                            ok, evidenza = N.prova_distrattore(ex, other, a, classe)
                            if not ok:
                                continue
                        else:
                            evidenza = "stesso tipo e distanza numerica ammessa"
                        pool.append(dict(valore=a, sq_id=other["id"], titolo=other["title"], domanda=other["question"],
                                         frase=N.frase_del_valore(other, a), evidenza=evidenza))
                # Un valore si usa una volta sola; la fonte a parità è fissata per ID.
                unici = {}
                for item in sorted(pool, key=lambda x: (x["valore"], x["sq_id"])):
                    unici.setdefault(item["valore"], item)
                return list(unici.values())
            pool = pool_da(dist_per_titolo[ex["title"]])
            origine = "stesso articolo"
            if not pool and tipo != "nome" and classe in ("anno", "numero"):
                pool, origine = pool_da(dev_tutti), "SQuAD dev"
            if not pool:
                scarta(rec, "DUE", "DUE-nat: nessun distrattore plausibile")
                rec["DUE_classe"] = classe
                continue
            fonte_dis = random.Random(f"{cfg['seme_naturale']}|{ex['id']}").choice(pool)
            dis = fonte_dis["valore"]
            ordine_span = random.Random(f"{cfg['seme_naturale']}|{ex['id']}|ordine").random() < 0.5
            nuovo = f"{span} or {dis}" if ordine_span else f"{dis} or {span}"
            it = dict(ex=ex, tipo=tipo, B=ctx, C=ctx[:start] + nuovo + ctx[start + len(span):],
                      distrattore=dis, ordine_span_primo=ordine_span, classe_domanda=classe, origine_distrattore=origine,
                      fonte_distrattore=fonte_dis)
            rec.update(DUE_classe=classe, DUE_distrattore=dis, DUE_origine=origine,
                       DUE_fonte_sq_id=fonte_dis["sq_id"], DUE_evidenza=fonte_dis["evidenza"])
            pr = testo_ricevuto_ok(it, ctl, motivi, "DUE-nat")
            if pr:
                rec["DUE"] = "escluso: DUE-nat: testo ricevuto non valido: " + "; ".join(pr[:4])
                continue
            due.append(it)
            usati_ctx["DUE"].add(ctx)
            usate_domande.add(ex["id"])
            rec["DUE"] = "scelto"
    out = {"RIM-nat": rim, "DUE-nat": due}
    for fam, items in out.items():
        say(f"naturale {fam}: {len(items)} coppie su {cfg['n_naturale']}")
    righe = []
    for fam, items in out.items():
        for it in items:
            ex = it["ex"]
            pk = f"N|{fam}|{ex['id']}"
            for lato in ("B", "C"):
                r = dict(row_id=f"{pk}|{lato}", insieme="naturale", passaggio_id=f"N|{ex['id']}", famiglia=fam,
                         kind=lato, pair_key=pk, sq_id=ex["id"], titolo=ex["title"], tipo_attributo=it["tipo"],
                         contesto=it[lato], domanda=ex["question"], prompt=prompt_risposta(it[lato], ex["question"]),
                         risposte_B=ex["answers"]["text"], lunghezze=it[f"lunghezze_{lato}"])
                if fam == "RIM-nat":
                    r["frasi_tolte"] = it["tolte"]
                else:
                    r["distrattore"] = it["distrattore"]
                    r["classe_domanda"] = it["classe_domanda"]
                    r["origine_distrattore"] = it["origine_distrattore"]
                    r["fonte_distrattore"] = it["fonte_distrattore"]
                righe.append(r)
    info = dict(regole_semantiche=N.VERSIONE, squad_revisione=rev, sha_hm0_ids=sha_hm0, sha_selezione_test4=sha_t4, candidati=len(cand), motivi=dict(motivi),
                coppie={k: len(v) for k, v in out.items()}, quota=cfg["n_naturale"],
                completo={k: len(v) == cfg["n_naturale"] for k, v in out.items()},
                nota="si prosegue sui candidati successivi fino alla quota; se i candidati finiscono, l'insieme è dichiarato incompleto",
                tipi={k: dict(Counter(it["tipo"] for it in v)) for k, v in out.items()},
                registro=dict(domande=len(registro),
                              esiti_filtro=dict(Counter(r["esito"].split(":")[0] for r in registro)),
                              esiti_RIM=dict(Counter(r["RIM"].split(" non valido:")[0] for r in registro if "RIM" in r)),
                              esiti_DUE=dict(Counter(r["DUE"].split(" non valido:")[0] for r in registro if "DUE" in r))))
    return righe, info, registro


def campione_revisione(passaggi, righe_nat, seme):
    rng = random.Random(seme)
    out = ["# Test C: campione per la revisione manuale (protocollo v3, §3.4 e §3.5)", "",
           "Per ogni coppia: B determina la risposta e C no? In CONF le due fonti sono alla pari? In LOG la disgiunzione ha "
           "lo stesso ruolo in B e in C? Per RIM-nat: C permette ancora di rispondere? Per DUE-nat: il distrattore è "
           "plausibile, cioè C non si risolve dal solo tipo del valore? Le frasi che cambiano sono in grassetto.", ""]
    n = 0
    for fam in ["RIM", "DUE", "CONF", "LOG"]:
        for p in rng.sample(passaggi, 5):
            n += 1
            F = p["famiglie"][fam]
            b, c = F["B"]["frasi"], F["C"]["frasi"]
            out += [f"## {n}. {fam}, {p['passaggio_id']} ({p['genere']}, {p['tipo_attributo']}, {F['posizione']})", "",
                    f"**Domanda:** {p['domanda']}  ", f"**Risposta in B:** {p['valore_B']}", ""]
            for lab, fr, other in (("B", b, c), ("C", c, b)):
                out += ["**" + lab + ".** " + " ".join(f"**{x}**" if x != y else x for x, y in zip(fr, other)), ""]
    rim = [r for r in righe_nat if r["famiglia"] == "RIM-nat" and r["kind"] == "C"]
    for r in rng.sample(rim, min(30, len(rim))):
        n += 1
        b = next(x for x in righe_nat if x["row_id"] == r["row_id"][:-1] + "B")
        out += [f"## {n}. RIM-nat, {r['sq_id']} ({r['tipo_attributo']})", "", f"**Domanda:** {r['domanda']}  ",
                f"**Risposte di SQuAD:** {r['risposte_B']}", "", f"**B (originale).** {b['contesto']}", "",
                f"**C (frasi tolte: {r['frasi_tolte']}).** {r['contesto']}", ""]
    # DUE-nat (aggiunte in C-sel-4): il distrattore è plausibile, cioè C non si risolve dal solo tipo del valore?
    due = [r for r in righe_nat if r["famiglia"] == "DUE-nat" and r["kind"] == "C"]
    for r in rng.sample(due, min(20, len(due))):
        n += 1
        frase = next((f for f in frasi_di(r["contesto"]) if f" or " in f and r["distrattore"] in f), r["contesto"])
        out += [f"## {n}. DUE-nat, {r['sq_id']} ({r['tipo_attributo']}, classe {r.get('classe_domanda')}, "
                f"distrattore da {r.get('origine_distrattore')})", "", f"**Domanda:** {r['domanda']}  ",
                f"**Risposte di SQuAD:** {r['risposte_B']}  ", f"**Distrattore:** {r['distrattore']}", "",
                f"**Frase di C con «or».** {frase}", "",
                f"**Origine del distrattore:** {r.get('fonte_distrattore', {}).get('domanda', '')}", "",
                f"**B completo.** {next(x['contesto'] for x in righe_nat if x['row_id'] == r['row_id'][:-1] + 'B')}", "",
                f"**C completo.** {r['contesto']}", ""]
    return "\n".join(out)


def main(cfg_path):
    cfg = read_json(cfg_path)
    out = Path(cfg["uscite"])
    out.mkdir(parents=True, exist_ok=True)
    for k, exp in cfg["sha_riserve"].items():
        got = sha_testo(cfg["riserve"][k])
        if got != exp:
            raise RuntimeError(f"riserva {k}: sha256 {got} invece di {exp}")
    ctl = Controlli({k: tuple(v) for k, v in cfg["modelli"].items()}, tuple(cfg["giudice"]))
    rapporto = dict(versione=cfg["versione"], tolleranza_token=cfg["tolleranza"], max_token_sirin=MAX_TOKEN,
                    istruzione=ISTRUZIONE, modelli=cfg["modelli"], giudice=cfg["giudice"], sha_riserve=cfg["sha_riserve"],
                    sha_script=cfg["sha_script"])
    righe = []
    rapporto["tokenizer"] = ctl.descrizione()
    for nome, kw in (("definitivo", dict(quote=cfg["quote"])), ("prova", dict(n_totale=cfg["n_prova"]))):
        ris = read_json(cfg["riserve"][nome])["passaggi"]
        scelti, registro, mancanti = seleziona_sintetico(ris, ctl, cfg["tolleranza"], **kw)
        attesi = sum(cfg["quote"].values()) if nome == "definitivo" else cfg["n_prova"]
        scartati = [(r["passaggio_id"], r["motivi"]) for r in registro if r["esito"] == "escluso"]
        write_json(out / f"registro_candidati_{nome}.json", registro)
        say(f"sintetico {nome}: {len(scelti)} scelti su {attesi}, {len(scartati)} candidati esclusi, quote mancanti {mancanti}")
        varianti = Counter(f"{fam}:B{F['B']['variante']}C{F['C']['variante']}" for p in scelti for fam, F in p["famiglie"].items())
        rapporto[nome] = dict(scelti=len(scelti), attesi=attesi, completo=len(scelti) == attesi and not mancanti,
                              quote_mancanti=mancanti, candidati_valutati=len(registro),
                              esiti_candidati=dict(Counter(r["esito"] for r in registro)),
                              motivi_esclusione=dict(Counter(m.split(":")[0] for r in registro for m in r["motivi"])),
                              scartati=[dict(passaggio_id=a, problemi=b[:5]) for a, b in scartati],
                              strati=dict(Counter(f"{p['tipo_attributo']}/{p.get('schema_posizione')}" for p in scelti)),
                              tipi=dict(Counter(p["tipo_attributo"] for p in scelti)),
                              generi=dict(Counter(p["genere"] for p in scelti)),
                              posizioni={fam: dict(Counter(p["famiglie"][fam]["posizione"] for p in scelti)) for fam in
                                         ["RIM", "DUE", "CONF", "LOG"]},
                              varianti_scelte=dict(varianti),
                              differenza_token_max={k: max(abs(F["differenza_token_C_meno_B"][k]) for p in scelti
                                                           for F in p["famiglie"].values()) for k in cfg["modelli"]} if scelti else {},
                              token_sirin_max={k: max(F[l]["lunghezze"][f"sirin_{k}"] for p in scelti
                                                      for F in p["famiglie"].values() for l in "BC") for k in cfg["modelli"]} if scelti else {})
        write_json(out / f"testC_sintetico_{nome}.json", dict(versione=cfg["versione"], insieme=nome, passaggi=scelti))
        rapporto[nome]["sha256_file"] = sha_testo(out / f"testC_sintetico_{nome}.json")
        rapporto[nome]["sha256_id"] = hashlib.sha256("\n".join(p["passaggio_id"] for p in scelti).encode()).hexdigest()
        if nome == "definitivo":
            sint_def = scelti
        righe += righe_sintetico(scelti, f"sintetico_{nome}")
    righe_nat, info_nat, registro_nat = costruisci_naturale(cfg, ctl)
    write_json(out / "registro_candidati_naturale.json", registro_nat)
    rapporto["naturale"] = info_nat
    righe += righe_nat
    ids = [r["row_id"] for r in righe]
    assert len(ids) == len(set(ids))
    write_json(out / "righe_C.json", righe)
    rapporto["righe"] = dict(Counter(r["insieme"] for r in righe))
    rapporto["sha256_righe_C"] = sha_testo(out / "righe_C.json")
    (out / "campione_revisione.md").write_text(campione_revisione(sint_def, righe_nat, cfg["seme_revisione"]))
    write_json(out / "rapporto_selezione.json", rapporto)
    rapporto["sha256_campione_revisione"] = sha_testo(out / "campione_revisione.md")
    write_json(out / "rapporto_selezione.json", rapporto)
    say("selezione finita:", json.dumps({k: rapporto[k] for k in ("righe", "sha256_righe_C")}, ensure_ascii=False))


if __name__ == "__main__":
    main(sys.argv[1])


In [ ]:
%%writefile /content/testC_sel6/script/verifica_selezione_v4.py
# Test C: verifica indipendente della selezione (C-sel-7; divisore in frasi del naturale corretto l'11 novembre). Non importa né il generatore né lo script di selezione:
# rilegge il testo con regole proprie, scritte qui, e ricalcola la regola di selezione dai registri. Ogni controllo non
# superato è un errore: alla fine lo script esce con codice 1 se c'è almeno un errore.
#
# Uso: python -I verifica_selezione_v4.py <cartella selezione> [--squad dev-v2.0.json]
#   <cartella selezione> contiene dati/ (righe_C.json, testC_sintetico_*.json, registro_candidati_*.json,
#   rapporto_selezione.json) e le due riserve testC_sintetico_*_riserva.json.
#   --squad (facoltativo): il file ufficiale di SQuAD 2.0 dev; se c'è, contesti, domande e risposte del naturale si
#   confrontano con la fonte.
#
# Cosa non verifica: i conteggi dei token (servono i tokenizer; qui si controlla solo che i valori salvati rispettino
# i limiti) e l'esclusione delle 400 domande di HM0 (serve il train di SQuAD per ricostruirle).
import json
import re
import string
import sys
from collections import Counter, defaultdict

ISTRUZIONE = "Answer the question using the passage. Reply with a short answer or UNKNOWN."
QUOTE = {"anno/0": 17, "anno/1": 17, "anno/2": 16, "numero/0": 17, "numero/1": 16, "numero/2": 17, "luogo/0": 16,
         "luogo/1": 17, "luogo/2": 17, "persona/0": 17, "persona/1": 17, "persona/2": 16}
N_PROVA, N_NATURALE, TOLLERANZA, MAX_SIRIN = 10, 200, 3, 512
MARCATORI = ["maybe", "probably", "possibly", "perhaps", "likely", "unlikely", "uncertain", "unknown", "unclear",
             "seems", "appears to", "may", "might", "could", "believed", "said to", "reportedly", "allegedly",
             "estimated", "about", "around", "roughly", "approximately", "not known", "no record"]
CIFRE = {"two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9, "ten": 10,
         "eleven": 11, "twelve": 12}

err = Counter()
esempi = defaultdict(list)


def errore(chiave, dettaglio):
    err[chiave] += 1
    if len(esempi[chiave]) < 5:
        esempi[chiave].append(dettaglio)


def occorrenze(testo, w):
    return len(re.findall(r"(?<![A-Za-z0-9])" + re.escape(w) + r"(?![A-Za-z0-9])", testo))


def frasi_sintetiche(testo):
    # nei passaggi sintetici ogni frase finisce con un punto e la successiva comincia con una maiuscola
    out, cur = [], []
    for tok in testo.split(" "):
        cur.append(tok)
        if tok.endswith("."):
            out.append(" ".join(cur))
            cur = []
    if cur:
        out.append(" ".join(cur))
    return out


def anni(testo):
    return [int(x) for x in re.findall(r"(?<![0-9])(1[0-9]{3}|20[0-9]{2})(?![0-9])", testo)]


# --------------------------------------------------------------------------- sintetico: lettura dal testo
def maschera(frase, togli):
    for w in sorted(togli, key=len, reverse=True):
        frase = re.sub(r"(?<![A-Za-z0-9])" + re.escape(w) + r"(?![A-Za-z0-9])", "§", frase)
    return frase


def cifre(p, chiave):
    # per i numeri anche la forma in cifre («seven» e «7»); per gli anni la risposta coincide con il valore
    return [x for x in p[chiave] if x.isdigit()] if p["tipo_attributo"] == "numero" else []


def valori_in(frase, p):
    forme = {p["valore_B"]: [p["valore_B"]] + cifre(p, "risposte_B"),
             p["valore_alternativo"]: [p["valore_alternativo"]] + cifre(p, "risposte_alternativo")}
    return [v for v, ff in forme.items() if any(occorrenze(frase, f) for f in ff)]


def controlla_versione_famiglia(p, fam, B, C, tag):
    """Determinabilità letta dal testo, famiglia per famiglia, con regole indipendenti dai template."""
    v1, v2 = p["valore_B"], p["valore_alternativo"]
    fb, fc = frasi_sintetiche(B), frasi_sintetiche(C)
    if len(fb) != len(fc):
        errore("B e C con un numero diverso di frasi", tag)
        return
    diverse = [i for i in range(len(fb)) if fb[i] != fc[i]]
    con_b = [i for i, f in enumerate(fb) if valori_in(f, p)]
    con_c = [i for i, f in enumerate(fc) if valori_in(f, p)]
    if fam == "RIM":
        if [valori_in(fb[i], p) for i in con_b] != [[v1]] or con_c:
            errore("RIM: B non dà il solo valore o C contiene un valore", tag)
        if diverse != con_b:
            errore("RIM: B e C differiscono fuori dalla frase con il valore", (tag, diverse, con_b))
    elif fam == "DUE":
        if [valori_in(fb[i], p) for i in con_b] != [[v1]]:
            errore("DUE: B non afferma il solo valore", tag)
        if len(con_c) != 1 or sorted(valori_in(fc[con_c[0]], p)) != sorted([v1, v2]):
            errore("DUE: C non ha i due valori nella stessa frase", tag)
        else:
            f = fc[con_c[0]]
            if not any(re.search(re.escape(a) + r" or (?:[A-Za-z]+ ){0,3}" + re.escape(b) + r"(?![A-Za-z0-9])", f)
                       for a, b in ((v1, v2), (v2, v1))):
                errore("DUE: in C i due valori non sono uniti da «or»", (tag, f))
        altre = [i for i in diverse if i not in con_b]
        if len(altre) != 1 or fb[altre[0]].replace(" or ", " and ") != fc[altre[0]] or " or " not in fb[altre[0]]:
            errore("DUE: la frase con «or» irrilevante di B non diventa «and» in C (o B e C differiscono altrove)",
                   (tag, diverse))
    elif fam in ("CONF", "LOG"):
        inizio = "According to the " if fam == "CONF" else "Every "
        nomi = p["nomi"]["fonti"] if fam == "CONF" else p["nomi"]["gruppi"]
        for lato, ff in (("B", fb), ("C", fc)):
            blocco = [f for f in ff if f.startswith(inizio) and any(occorrenze(f, n) for n in nomi)]
            if len(blocco) != 2:
                errore(f"{fam}: {lato} non ha due frasi «{inizio.strip()}»", tag)
                continue
            nominati = [[n for n in nomi if occorrenze(f, n)] for f in blocco]
            vals = [valori_in(f, p) for f in blocco]
            if sorted(sum(nominati, [])) != sorted(nomi) or any(len(x) != 1 for x in nominati):
                errore(f"{fam}: le due frasi non nominano una {'fonte' if fam == 'CONF' else 'regola di gruppo'} "
                       "diversa ciascuna", (tag, lato, nominati))
            # pari rango: tolti il nome e il valore, le due frasi sono identiche
            m = [maschera(f, nomi + [v1, v2] + cifre(p, "risposte_B") + cifre(p, "risposte_alternativo"))
                 for f in blocco]
            if m[0] != m[1]:
                errore(f"{fam}: le due frasi non sono parallele (pari rango)", (tag, lato, m))
            attesi = [[v1], [v1]] if lato == "B" else None
            if lato == "B" and vals != attesi:
                errore(f"{fam}: in B le due frasi non danno lo stesso valore", (tag, vals))
            if lato == "C" and (sorted(sum(vals, [])) != sorted([v1, v2]) or any(len(x) != 1 for x in vals)):
                errore(f"{fam}: in C le due frasi non danno i due valori, uno ciascuna", (tag, lato, vals))
            fuori = [f for f in ff if valori_in(f, p) and f not in blocco]
            if fuori:
                errore(f"{fam}: valori fuori dal blocco", (tag, lato, fuori))
        if len(diverse) != 1 or not (fb[diverse[0]].startswith(inizio) and any(occorrenze(fb[diverse[0]], n) for n in nomi)):
            errore(f"{fam}: B e C non differiscono in una sola frase del blocco", (tag, diverse))
        if fam == "LOG":
            for lato, ff in (("B", fb), ("C", fc)):
                disg = [f for f in ff if all(occorrenze(f, n) for n in nomi) and " or " in f]
                if len(disg) != 1 or not occorrenze(disg[0], p["soggetto"]):
                    errore("LOG: manca la frase «soggetto … G1 … or … G2»", (tag, lato))
            if [f for f in fb if " or " in f] != [f for f in fc if " or " in f]:
                errore("LOG: la disgiunzione cambia fra B e C", tag)


def controlla_passaggio(p, insieme):
    pid = p["passaggio_id"]
    v1, v2, tipo = p["valore_B"], p["valore_alternativo"], p["tipo_attributo"]
    if occorrenze(p["domanda"], v1) or occorrenze(p["domanda"], v2):
        errore("domanda con un valore", pid)
    if not occorrenze(p["domanda"], p["entita"]):
        errore("domanda senza l'entità", pid)
    if v1 == v2:
        errore("valori uguali", pid)
    atteso_occ = {("RIM", "B"): (1, 0), ("RIM", "C"): (0, 0), ("DUE", "B"): (1, 0), ("DUE", "C"): (1, 1),
                  ("CONF", "B"): (2, 0), ("CONF", "C"): (1, 1), ("LOG", "B"): (2, 0), ("LOG", "C"): (1, 1)}
    attesi_or = {"RIM": 0, "CONF": 0, "DUE": 1, "LOG": 1}
    for fam, F in p["famiglie"].items():
        tag = f"{insieme}/{pid}/{fam}"
        B, C = F["B"]["testo"], F["C"]["testo"]
        for lato, t in (("B", B), ("C", C)):
            o1 = occorrenze(t, v1) + sum(occorrenze(t, x) for x in cifre(p, "risposte_B"))
            o2 = occorrenze(t, v2) + sum(occorrenze(t, x) for x in cifre(p, "risposte_alternativo"))
            if (o1, o2) != atteso_occ[(fam, lato)]:
                errore("occorrenze dei valori", (tag, lato, (o1, o2)))
            if occorrenze(t, "or") != attesi_or[fam]:
                errore("numero di «or»", (tag, lato, occorrenze(t, "or")))
            for w in MARCATORI:
                if re.search(r"(?<![A-Za-z])" + re.escape(w) + r"(?![A-Za-z])", t, re.I):
                    errore("marcatore di riserva nel testo", (tag, lato, w))
            if not 120 <= len(t.split()) <= 180:
                errore("parole fuori da 120–180", (tag, lato, len(t.split())))
            if " ".join(F[lato]["frasi"]) != t:
                errore("testo diverso dall'unione delle frasi salvate", (tag, lato))
            for k, n in F[lato]["lunghezze"].items():
                if k.startswith("sirin") and n > MAX_SIRIN:
                    errore("prompt SIRIN oltre 512 token", (tag, lato, k, n))
            # cronologia: nei passaggi di tipo anno ogni altro anno viene dopo entrambi i valori
            if tipo == "anno":
                prima = [y for y in anni(t) if str(y) not in (v1, v2) and y <= max(int(v1), int(v2))]
                if prima:
                    errore("cronologia: anni di contorno non successivi all'anno bersaglio", (tag, lato, prima, v1, v2))
        if sorted(y for y in anni(B) if str(y) not in (v1, v2)) != sorted(y for y in anni(C) if str(y) not in (v1, v2)):
            errore("anni di contorno diversi fra B e C", tag)
        d = F["differenza_token_C_meno_B"]
        if not d or max(abs(x) for x in d.values()) > TOLLERANZA:
            errore("differenza di token oltre la tolleranza", (tag, d))
        controlla_versione_famiglia(p, fam, B, C, tag)


# --------------------------------------------------------------------------- selezione: primi validi
def controlla_regola(riserva, scelti, registro, nome, quote=None, n=None):
    ordine_ris = [p["passaggio_id"] for p in riserva]
    if [r["passaggio_id"] for r in registro] != ordine_ris:
        errore("registro non nell'ordine della riserva", nome)
    for r in registro:
        if r["esito"] == "escluso" and not r["motivi"]:
            errore("candidato escluso senza motivo", (nome, r["passaggio_id"]))
        if r["esito"] != "escluso" and r["motivi"]:
            errore("candidato valido con motivi", (nome, r["passaggio_id"]))
    validi = [r for r in registro if r["esito"] != "escluso"]
    attesi, presi = [], Counter()
    for r in validi:
        st = f"{r['tipo']}/{r['schema_posizione']}"
        if quote is not None and presi[st] < quote[st]:
            presi[st] += 1
            attesi.append(r["passaggio_id"])
        elif quote is None and len(attesi) < n:
            attesi.append(r["passaggio_id"])
    ids = [p["passaggio_id"] for p in scelti]
    if ids != attesi:
        errore("la selezione non è «i primi validi» (ricalcolo dal registro)", (nome, len(ids), len(attesi)))
    if {r["passaggio_id"] for r in registro if r["esito"] == "scelto"} != set(ids):
        errore("esiti «scelto» del registro diversi dai passaggi scelti", nome)
    if quote is not None and dict(presi) != quote:
        errore("quote per strato non piene", (nome, dict(presi)))
    if quote is None and len(ids) != n:
        errore("numero della prova diverso da 10", (nome, len(ids)))
    # il testo scelto è quello della variante indicata nella riserva, senza modifiche
    per_id = {p["passaggio_id"]: p for p in riserva}
    for p in scelti:
        q = per_id.get(p["passaggio_id"])
        if q is None:
            errore("passaggio scelto assente dalla riserva", p["passaggio_id"])
            continue
        for fam, F in p["famiglie"].items():
            for lato in ("B", "C"):
                if q["famiglie"][fam][lato][F[lato]["variante"]]["testo"] != F[lato]["testo"]:
                    errore("testo scelto diverso dalla variante della riserva", (p["passaggio_id"], fam, lato))
        for k in ("valore_B", "valore_alternativo", "domanda", "entita", "tipo_attributo", "genere"):
            if q[k] != p[k]:
                errore("metadati diversi dalla riserva", (p["passaggio_id"], k))


def parole_nomi(ps):
    w = set()
    for p in ps:
        w |= {p["entita"].lower(), p["nomi"]["regione"].lower()} | {x.lower() for x in p["nomi"]["gruppi"] + p["nomi"]["fonti"]}
        if p["tipo_attributo"] in ("luogo", "persona"):
            w |= {x.lower() for v in (p["valore_B"], p["valore_alternativo"]) for x in v.split()}
    return w


# --------------------------------------------------------------------------- naturale
def norm(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


ABBR = {"mr", "mrs", "ms", "dr", "st", "jr", "sr", "prof", "gen", "col", "lt", "gov", "sen", "rep", "rev", "capt",
        "sgt", "mt", "ft", "no", "vs", "etc", "inc", "ltd", "co", "corp", "jan", "feb", "mar", "apr", "jun", "jul",
        "aug", "sep", "sept", "oct", "nov", "dec", "e.g", "i.e", "u.s", "u.k", "a.d", "b.c", "c", "ca", "approx", "fig",
        "vol", "op", "pp", "ed", "eds", "est", "dept", "univ", "assn", "bros", "cf", "al"}


def frasi_naturali(testo):
    """Frasi di un paragrafo naturale, scritte qui da capo: una frase si chiude su . ! ? seguito da eventuali virgolette
    o parentesi di chiusura, poi da uno o più spazi e da maiuscola, cifra, virgolette o parentesi; il confine vale solo
    se, dopo le chiusure, non resta aperta nessuna parentesi o virgoletta, e mai dopo un'iniziale (una lettera sola) o
    un'abbreviazione. La parola prima del punto è l'intera sequenza senza spazi («1950s» non è un'iniziale).
    Versione corretta l'11 novembre (C-sel-5): la versione per C-sel-4 non chiudeva le frasi che finiscono con «."» o
    «.)», leggeva «1950s.» come iniziale e non accettava due spazi dopo il punto."""
    out, inizio, prof, virg, curve = [], 0, 0, False, 0
    i, n = 0, len(testo)
    while i < n:
        ch = testo[i]
        if ch in "([":
            prof += 1
        elif ch in ")]":
            prof = max(0, prof - 1)
        elif ch == '"':
            virg = not virg
        elif ch == "\u201c":
            curve += 1
        elif ch == "\u201d":
            curve = max(0, curve - 1)
        if ch in ".!?":
            j, p2, v2, c2 = i + 1, prof, virg, curve
            while j < n and testo[j] in "\"')]\u201d\u2019":     # chiusure subito dopo il segno
                if testo[j] in ")]":
                    p2 = max(0, p2 - 1)
                elif testo[j] == '"':
                    v2 = not v2
                elif testo[j] == "\u201d":
                    c2 = max(0, c2 - 1)
                j += 1
            k = j
            while k < n and testo[k] in " \t\n":
                k += 1
            if (k > j and k < n and p2 == 0 and not v2 and c2 == 0
                    and (testo[k].isupper() or testo[k].isdigit() or testo[k] in "\"'(\u201c")):
                parola = re.findall(r"(\S+)$", testo[inizio:i])
                parola = parola[0].lstrip("\"'([\u201c").lower() if parola else ""
                if ch == "." and (re.fullmatch(r"[a-z]", parola) or parola in ABBR
                                  or re.fullmatch(r"(?:[a-z]\.)+[a-z]", parola)):
                    i += 1
                    continue
                out.append(testo[inizio:j].strip())
                inizio = k
                prof, virg, curve = p2, v2, c2
                i = k
                continue
        i += 1
    if testo[inizio:].strip():
        out.append(testo[inizio:].strip())
    return out


def numeri(testo):
    t = testo.lower().replace(",", "")
    out = {float(x) for x in re.findall(r"(?<![0-9.])[0-9]+(?:\.[0-9]+)?(?![0-9])", t)}
    parole = {"one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
              "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13, "fourteen": 14, "fifteen": 15, "sixteen": 16,
              "seventeen": 17, "eighteen": 18, "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50,
              "sixty": 60, "seventy": 70, "eighty": 80, "ninety": 90, "hundred": 100}
    out |= {float(parole[w]) for w in re.findall(r"[a-z]+", t) if w in parole}
    return out


def risposte_coerenti(risposte, tipo):
    """Regola di validazione dei riferimenti (C-sel-4), riscritta qui: per anni e numeri tutte le risposte portano lo
    stesso unico valore; per i nomi la più corta è contenuta in tutte le altre (o le contiene)."""
    rr = [r for r in risposte if norm(r)]
    if not rr:
        return "nessuna risposta"
    if tipo == "anno":
        vv = [set(anni(r)) for r in rr]
        if any(len(v) != 1 for v in vv) or len(set.union(*vv)) != 1:
            return f"anni diversi o multipli nelle risposte: {rr}"
        return None
    if tipo == "numero":
        vv = [numeri(r) for r in rr]
        if any(len(v) != 1 for v in vv) or len(set.union(*vv)) != 1:
            return f"numeri diversi o multipli nelle risposte: {rr}"
        return None
    base = norm(min(rr, key=len))
    for r in rr:
        n = norm(r)
        if not (re.search(r"(?<!\w)" + re.escape(base) + r"(?!\w)", n) or re.search(r"(?<!\w)" + re.escape(n) + r"(?!\w)", base)):
            return f"risposte che non si contengono: {rr}"
    return None


def testo_integro(testo):
    if testo.count("(") != testo.count(")") or testo.count("[") != testo.count("]") or testo.count('"') % 2:
        return "parentesi o virgolette non bilanciate"
    for f in frasi_naturali(testo):
        if not (f[0].isupper() or f[0].isdigit() or f[0] in "\"'("):
            return f"frase che non comincia con la maiuscola: {f[:40]}"
        if not re.search(r"[.!?][\"')\]]*$", f):
            return f"frase senza punto finale: {f[-40:]}"
    return None


# Controlli semantici C-verifica-4, riscritti qui: non importano i filtri della selezione.
def parole_semantiche_v(testo):
    s = re.sub(r"['’]s\b", "", testo.casefold())
    return re.findall(r"[a-z0-9]+", s)


def rim_nome_residuo(B, C, risposte):
    parole_c = parole_semantiche_v(C)
    alias = {"china": ["chinese"], "france": ["french"], "germany": ["german", "germans"],
             "scotland": ["scottish", "scots"], "poland": ["polish"], "england": ["english"],
             "netherlands": ["dutch"], "japan": ["japanese"], "india": ["indian", "indians"],
             "norman": ["normans"]}
    sequenze = []
    for risposta in risposte:
        pp = parole_semantiche_v(risposta)
        if not pp:
            continue
        sequenze.append(pp)
        if len(pp) == 1:
            sequenze += [[w] for w in alias.get(pp[0], [])]
            if len(pp[0]) >= 4 and not pp[0].endswith("s"):
                sequenze.append([pp[0] + "s"])
                if pp[0].endswith("a"):
                    sequenze.append([pp[0] + "n"])
        else:
            generici = {"the", "school", "university", "college", "river", "city", "town", "court", "justice", "king", "queen",
                        "prince", "princess", "saint", "doctor", "sir", "lady", "lord", "admiral", "general", "captain",
                        "hurricane", "storm", "war", "battle", "north", "south", "east", "west", "new", "old", "great",
                        "grand", "royal", "national", "united", "company", "corporation"}
            sequenze.extend([w] for w in pp if len(w) >= 4 and w not in generici)
        for app in re.finditer(re.escape(risposta) + r"\s*\(([^)]*)\)", B, flags=re.I):
            a = app.group(1).strip().strip('"\u201c\u201d')
            if re.fullmatch(r"[A-Za-z -]{1,45}", a) and len(a.split()) <= 3:
                sequenze.append(parole_semantiche_v(a))
    for pp in sequenze:
        if pp and any(parole_c[i:i + len(pp)] == pp for i in range(len(parole_c) - len(pp) + 1)):
            return " ".join(pp)
    return None


def valore_base_v(testo):
    vals = numeri(testo)
    return next(iter(vals)) if len(vals) == 1 else None


def via_calcolo_v(domanda, residuo, target):
    if target is None or not re.search(r"\b(?:how many|how much|population|total|overall|number)\b", domanda, re.I):
        return False
    import itertools
    import math
    tokens = list(re.finditer(r"\d[\d,]*(?:\.\d+)?|\b[A-Za-z]+\b", residuo))
    vv = []
    lettere = {w: i for i, w in enumerate("zero one two three four five six seven eight nine ten eleven twelve "
                                         "thirteen fourteen fifteen sixteen seventeen eighteen nineteen twenty".split())}
    lettere.update(dict(thirty=30, forty=40, fifty=50, sixty=60, seventy=70, eighty=80, ninety=90))
    for match in tokens:
        tok = match.group()
        if tok[0].isdigit():
            before = residuo[match.start() - 1] if match.start() else ""
            after = residuo[match.end()] if match.end() < len(residuo) else ""
            if (before and (before.isalnum() or before in "_.")) or (after and (after.isalnum() or after == "_")):
                continue
            vv.append(float(tok.replace(",", "")))
        elif tok.lower() in lettere:
            vv.append(float(lettere[tok.lower()]))
    if len(vv) > 18:
        return True
    uguale = lambda x: math.isclose(x, target, rel_tol=1e-9, abs_tol=1e-9)
    for k in range(2, min(len(vv), 6) + 1):
        if any(uguale(sum(v)) for v in itertools.combinations(vv, k)):
            return True
    return any(uguale(abs(a - b)) or (min(a, b) > 1 and uguale(a * b))
               for a, b in itertools.combinations(vv, 2))


def controlli_nuovi_naturale(B, C, sq, fam, squad):
    q, b, c = B["domanda"], B["contesto"], C["contesto"]
    qn = q.lower()
    if re.search(r"\bborn\b|\bbirth\b", q, re.I):
        frasi_riferimento = [f for f in frasi_naturali(b) if any(a in f for a in B["risposte_B"])]
        if not any(re.search(r"\bborn\b|\bbirth\b", f, re.I) for f in frasi_riferimento):
            errore("naturale: nascita non esplicita nella frase della risposta", sq)
    if ("one of" in qn or re.search(r"\bname (a|an)\b|\bgive (?:an? )?example\b|\bsome (researchers|scientists|historians)\b", qn)):
        errore("naturale: domanda con risposta aperta o alcune posizioni", (sq, q))
    if any(re.search(r"['’]s\s*$", a) for a in B["risposte_B"]):
        errore("naturale: riferimento possessivo incompleto", sq)
    if fam == "RIM-nat":
        if B["tipo_attributo"] == "nome":
            a = rim_nome_residuo(b, c, B["risposte_B"])
            if a:
                errore("RIM-nat: nome o variante morfologica ancora presente", (sq, a))
        fb, fc = frasi_naturali(b), frasi_naturali(c)
        scelti = set(fc)
        for i, f in enumerate(fb):
            if f in scelti and i and fb[i - 1] not in scelti:
                if (f.split()[0] in {"It", "Its", "They", "Their", "He", "She", "His", "Her", "These", "Those", "This", "Such"}
                        or (f.startswith("The ") and len(f.split()) > 1 and f.split()[1][0].islower())):
                    errore("RIM-nat: anafora o sintagma definito dopo una frase rimossa", (sq, f[:90]))
                if re.search(r"^(?:In (?:the )?(?:following|subsequent|next)|Similarly\b|Later\b|Afterwards\b|(?:A|One) (?:mere )?[^.!?]{0,35}\blater\b)", f):
                    errore("RIM-nat: riferimento temporale dopo una frase rimossa", (sq, f[:90]))
        if B["tipo_attributo"] == "numero":
            if re.search(r"\bhow many\b|\bnumber of\b", q, re.I) and re.search(r"(?:^|[.!?]\s+)(?:These|Those) (?:are|include)\b[^.!?]+(?:,|\band\b)", c):
                errore("RIM-nat: enumerazione residua", sq)
            if re.search(r"\bhow many\b|\bnumber of\b", q, re.I) and re.search(r":\s*[^.!?]*(?:;|,\s*[A-Za-z]|\band\b)", c):
                errore("RIM-nat: possibile enumerazione dopo i due punti", sq)
            if via_calcolo_v(q, c, valore_base_v(B["risposte_B"][0])):
                errore("RIM-nat: calcolo residuo", sq)
    else:
        dis = C.get("distrattore", "")
        # Si ricostruisce lo span effettivamente cambiato, non il primo
        # riferimento dell'elenco (che può includere 'rows' oppure '%').
        span_modificato = None
        for ref in sorted(set(B["risposte_B"]), key=lambda x: (len(x), x)):
            for occ in re.finditer(re.escape(ref), b):
                left, right = b[:occ.start()], b[occ.end():]
                if c in (left + ref + " or " + dis + right, left + dis + " or " + ref + right):
                    span_modificato = ref
                    break
            if span_modificato is not None:
                break
        a0 = span_modificato or min(B["risposte_B"], key=len)
        residuo_contesto = b.replace(a0, " VALORE_RIMOSSO ", 1)
        if B["tipo_attributo"] == "numero" and re.search(r"\b(?:average|averaged|mean)\b", residuo_contesto, re.I):
            if re.search(r"\b(?:maximum|minimum|at most|at least|as high as|as low as|up to|no more than)\b", residuo_contesto, re.I):
                errore("DUE-nat: media con estremi da verificare", sq)
        if B["tipo_attributo"] == "anno" and re.search(r"\b(?:after|before|since|until|from|between)\s+(?:the (?:year|years) )?\d{4}\b", residuo_contesto, re.I):
            errore("DUE-nat: vincolo cronologico residuo", sq)
        if re.search(r"['’]s\s*$", dis):
            errore("DUE-nat: distrattore possessivo incompleto", (sq, dis))
        if B["tipo_attributo"] == "numero":
            a = a0
            frase = next((f for f in frasi_naturali(b) if a in f), "")
            residuo = b.replace(a, " VALORE_RIMOSSO ", 1)
            if via_calcolo_v(q, residuo, valore_base_v(a)):
                errore("DUE-nat: totale o valore ricavabile dalle altre quantità", sq)
            if re.search(r"\bhow many of\b", q, re.I) or re.search(re.escape(a) + r"\s+(?:out )?of\s+(?:the\s+)?", frase, re.I):
                errore("DUE-nat: conteggio di un sottoinsieme", sq)
            if re.search(r"\bhow many\b|\bnumber of\b", q, re.I) and (re.search(r":\s*[^.!?]*(?:;|,\s*[A-Za-z]|\band\b)", residuo)
                    or re.search(r"(?:^|[.!?]\s+)(?:These|Those) (?:are|include)\b[^.!?]+(?:,|\band\b)", residuo)):
                errore("DUE-nat: enumerazione residua", sq)
        fonte = C.get("fonte_distrattore")
        if not fonte or not fonte.get("sq_id") or not fonte.get("domanda"):
            errore("DUE-nat: provenienza del distrattore mancante", sq)
        elif squad is not None:
            ex = squad.get(fonte["sq_id"])
            if not ex or ex["question"] != fonte["domanda"] or fonte.get("valore") != dis:
                errore("DUE-nat: fonte del distrattore non corrispondente a SQuAD", sq)
            elif not any(norm(dis) == norm(a) or re.search(r"(?<!\w)" + re.escape(norm(dis)) + r"(?!\w)", norm(a)) for a in ex["answers"]):
                errore("DUE-nat: il distrattore non è una risposta della fonte", sq)
            if ex is not None and (fonte.get("frase", "") not in ex["context"]
                                   or (ex.get("title") and fonte.get("titolo") != ex["title"])):
                errore("DUE-nat: frase o articolo della fonte non corrispondenti", sq)
        if B["tipo_attributo"] == "nome":
            if C.get("origine_distrattore") != "stesso articolo" or (fonte or {}).get("titolo") != B["titolo"]:
                errore("DUE-nat: nome fuori dall'articolo", sq)
            if C.get("classe_domanda") in {None, "cosa:other", "cosa:entity", "cosa:thing"}:
                errore("DUE-nat: classe nominale generica", sq)
            if not fonte or not fonte.get("evidenza"):
                errore("DUE-nat: ruolo nominale non documentato", sq)
            else:
                if C.get("classe_domanda") == "persona":
                    if re.search(r"\b(?:muslims|christians|catholics|protestants|mongols|chinese|administrators|troops|soldiers|people|immigrants|settlers|officials|scientists|researchers)\b", dis, re.I):
                        errore("DUE-nat: nome collettivo usato come persona", sq)
                    source_question = fonte["domanda"]
                    autore = r"\b(?:wrote|write|written|author|writer|novelist|poet|journalist|bestseller)\b"
                    if re.search(autore, q, re.I) and not re.search(autore, source_question, re.I):
                        errore("DUE-nat: ruolo di autore non documentato nella domanda fonte", sq)
                    if re.search(r"\bfounder\b|\bfounded\b", q, re.I) and re.search(r"\b(?:corporation|company|firm|business)\b", q, re.I):
                        if not (re.search(r"\bfounder\b|\bfounded\b", source_question, re.I)
                                and re.search(r"\b(?:corporation|company|firm|business)\b", source_question, re.I)):
                            errore("DUE-nat: fondatore di impresa non documentato nella domanda fonte", sq)
                    militare = r"\b(?:army|military|attack|campaign|battle|general|commander)\b"
                    if re.search(militare, q, re.I) and not re.search(militare, source_question, re.I):
                        errore("DUE-nat: ruolo militare non documentato nella domanda fonte", sq)
                for lemma in fonte["evidenza"].split(", "):
                    inizio = re.escape(lemma[:-1]) + r"(?:y|ies)" if lemma.endswith("y") else re.escape(lemma)
                    if not all(re.search(r"\b" + inizio, testo, re.I)
                               for testo in (q, fonte["domanda"])):
                        errore("DUE-nat: evidenza del ruolo assente dalle domande", (sq, lemma))
                regioni = {"england": ("england", "english"), "scotland": ("scotland", "scottish"),
                           "holland": ("holland", "netherlands", "dutch"), "netherlands": ("holland", "netherlands", "dutch"),
                           "france": ("france", "french"), "germany": ("germany", "german"), "china": ("china", "chinese"),
                           "japan": ("japan", "japanese"), "india": ("india", "indian"), "nepal": ("nepal", "nepali", "nepalese")}
                for regione, aliases in regioni.items():
                    if re.search(r"\b" + regione + r"\b", q, re.I) and not any(
                            re.search(r"\b" + a + r"\b", fonte["domanda"] + " " + fonte["frase"], re.I) for a in aliases):
                        errore("DUE-nat: vincolo geografico non corroborato dalla fonte", (sq, regione))
    # Rischio ferroviario: domanda generica sul fornitore e altri operatori ancora nominati.
    if re.search(r"\btrain|\brail", q, re.I) and re.search(r"compan(?:y|ies)|service", q, re.I):
        residue = frasi_naturali(c) if fam == "RIM-nat" else [f for f in frasi_naturali(b) if not any(a in f for a in B["risposte_B"])]
        if any(re.search(r"\b(?:train|trains|railway|railways|railroad|railroads|branchline|branchlines)\b", f, re.I) for f in residue):
            errore("naturale: altra via nominale ferroviaria non esclusa", sq)



def controlla_naturale(righe, registro, squad):
    nat = [r for r in righe if r["insieme"] == "naturale"]
    coppie = defaultdict(dict)
    for r in nat:
        coppie[(r["famiglia"], r["sq_id"])][r["kind"]] = r
    per_fam = Counter(f for f, _ in coppie)
    for fam in ("RIM-nat", "DUE-nat"):
        if per_fam[fam] != N_NATURALE:
            errore("naturale: coppie diverse dalla quota", (fam, per_fam[fam]))
    usate = defaultdict(set)
    for (fam, sq), d in coppie.items():
        if set(d) != {"B", "C"}:
            errore("naturale: coppia incompleta", (fam, sq))
            continue
        B, C = d["B"], d["C"]
        if sq in usate["domande"]:
            errore("naturale: domanda usata due volte (anche fra famiglie)", sq)
        usate["domande"].add(sq)
        if B["contesto"] in usate[fam]:
            errore("naturale: paragrafo usato due volte nella stessa famiglia", (fam, sq))
        usate[fam].add(B["contesto"])
        if B["domanda"] != C["domanda"] or B["risposte_B"] != C["risposte_B"]:
            errore("naturale: domanda o risposte diverse fra B e C", sq)
        if squad is not None:
            ex = squad.get(sq)
            if ex is None or ex["context"] != B["contesto"] or ex["question"] != B["domanda"] \
                    or list(ex["answers"]) != list(B["risposte_B"]):
                errore("naturale: B diverso dalla fonte SQuAD", sq)
        m = risposte_coerenti(B["risposte_B"], B["tipo_attributo"])
        if m:
            errore("naturale: riferimenti non coerenti", (fam, sq, m))
        controlli_nuovi_naturale(B, C, sq, fam, squad)
        forme = {norm(a) for a in B["risposte_B"] if norm(a)}
        if fam == "RIM-nat":
            fb, fc = frasi_naturali(B["contesto"]), frasi_naturali(C["contesto"])
            it = iter(fb)
            if not all(any(x == y for y in it) for x in fc):
                errore("RIM-nat: le frasi di C non sono frasi intere di B, nello stesso ordine", sq)
            if len(fc) < 3:
                errore("RIM-nat: meno di 3 frasi vere in C", (sq, len(fc)))
            m = testo_integro(C["contesto"])
            if m:
                errore("RIM-nat: segni di taglio in C", (sq, m))
            nc = norm(C["contesto"])
            if any(re.search(r"(?<!\w)" + re.escape(a) + r"(?!\w)", nc) for a in forme):
                errore("RIM-nat: la risposta resta in C", sq)
            if B["tipo_attributo"] in ("anno", "numero"):
                v = anni(B["risposte_B"][0]) if B["tipo_attributo"] == "anno" else numeri(B["risposte_B"][0])
                vc = anni(C["contesto"]) if B["tipo_attributo"] == "anno" else numeri(C["contesto"])
                if set(v) & set(vc):
                    errore("RIM-nat: il valore della risposta resta in C (in altra forma)", sq)
        else:
            b, c, dis = B["contesto"], C["contesto"], C.get("distrattore", "")
            # C deve essere B con una risposta di SQuAD sostituita da «risposta or distrattore» o «distrattore or risposta»
            ok = False
            for span in set(B["risposte_B"]):
                for mm in re.finditer(re.escape(span), b):
                    s, e = mm.start(), mm.end()
                    if c in (b[:s] + f"{span} or {dis}" + b[e:], b[:s] + f"{dis} or {span}" + b[e:]):
                        ok = True
            if not dis or not ok:
                errore("DUE-nat: C non è B con la sola aggiunta «or distrattore» accanto alla risposta", (sq, dis))
            if occorrenze(c, "or") != occorrenze(b, "or") + 1:
                errore("DUE-nat: «or» non aumenta di uno", sq)
            if norm(dis) and re.search(r"(?<!\w)" + re.escape(norm(dis)) + r"(?!\w)", norm(b)):
                errore("DUE-nat: il distrattore è già nel paragrafo", sq)
            if norm(dis) in forme:
                errore("DUE-nat: il distrattore coincide con una risposta", sq)
            t = B["tipo_attributo"]
            if t == "anno":
                a, x = anni(B["risposte_B"][0]), anni(dis)
                if len(a) != 1 or len(x) != 1 or not 0 < abs(a[0] - x[0]) <= 50:
                    errore("DUE-nat: anno distrattore non plausibile (oltre 50 anni)", (sq, B["risposte_B"][0], dis))
            elif t == "numero":
                a, x = numeri(B["risposte_B"][0]), numeri(dis)
                if len(a) != 1 or len(x) != 1:
                    errore("DUE-nat: numero non leggibile", (sq, B["risposte_B"][0], dis))
                else:
                    (a,), (x,) = a, x
                    ok = a != x and a > 0 and x > 0 and (abs(a - x) <= 5 if max(a, x) <= 20 else 0.5 <= x / a <= 2)
                    if a.is_integer() and not x.is_integer():
                        ok = False
                    q = B["domanda"]
                    if (re.search(r"\bpercent(?:age)?\b", q, re.I)
                            and not re.search(r"\bincrease\b|\bgrowth\b|\bchange\b|\brise\b|\bdecrease\b", q, re.I)
                            and x > 100):
                        ok = False
                    if not ok:
                        errore("DUE-nat: numero distrattore non plausibile", (sq, B["risposte_B"][0], dis))
    # registro dei candidati del naturale: ogni coppia scelta è «scelto», ogni esclusione ha il suo controllo, e prima
    # dell'ultima coppia scelta nessun candidato è saltato per quota piena (primi validi nell'ordine fissato)
    if registro is None:
        errore("registro dei candidati del naturale mancante", "registro_candidati_naturale.json")
        return
    ids = [r["sq_id"] for r in registro]
    if len(ids) != len(set(ids)):
        errore("registro naturale con domande ripetute", len(ids) - len(set(ids)))
    for chiave, fam in (("RIM", "RIM-nat"), ("DUE", "DUE-nat")):
        scelti_reg = {r["sq_id"] for r in registro if r.get(chiave) == "scelto"}
        scelti_righe = {sq for f, sq in coppie if f == fam}
        if scelti_reg != scelti_righe:
            errore("registro naturale: scelti diversi dalle righe", (fam, len(scelti_reg), len(scelti_righe)))
        ordini = sorted(r["ordine"] for r in registro if r.get(chiave) == "scelto")
        if ordini:
            for r in registro:
                if r.get("ordine", 1e9) < ordini[-1] and str(r.get(chiave, "")).startswith("non valutato: quota"):
                    errore("registro naturale: candidato saltato per quota prima dell'ultimo scelto", (fam, r["sq_id"]))
        for r in registro:
            s = r.get(chiave)
            if s is None:
                continue
            if not (s == "scelto" or (s.startswith("escluso: ") and len(s) > 10) or s.startswith("non valutato: ")):
                errore("registro naturale: esito senza motivo", (fam, r["sq_id"], s))
    for r in registro:
        if r.get("fase") == "filtro" and not str(r.get("esito", "")).startswith("escluso: "):
            errore("registro naturale: filtro senza motivo", r["sq_id"])


def main(cart, squad_path=None):
    L = lambda f: json.load(open(f))
    dati = f"{cart}/dati"
    righe = L(f"{dati}/righe_C.json")
    rap = L(f"{dati}/rapporto_selezione.json")
    if rap.get("versione") != "C-sel-7" or rap.get("naturale", {}).get("regole_semantiche") != "C-nat-7":
        errore("versione della selezione o dei filtri semantici non attesa", rap.get("versione"))
    sel = {n: L(f"{dati}/testC_sintetico_{n}.json")["passaggi"] for n in ("definitivo", "prova")}
    ris = {n: L(f"{cart}/testC_sintetico_{n}_riserva.json")["passaggi"] for n in ("definitivo", "prova")}
    reg = {n: L(f"{dati}/registro_candidati_{n}.json") for n in ("definitivo", "prova")}
    try:
        reg_nat = L(f"{dati}/registro_candidati_naturale.json")
    except FileNotFoundError:
        reg_nat = None
    squad = None
    if squad_path:
        sq = L(squad_path)
        if "data" in sq:                      # file ufficiale dev-v2.0.json
            squad = {q["id"]: dict(title=art["title"], context=par["context"], question=q["question"], answers=[a["text"] for a in q["answers"]])
                     for art in sq["data"] for par in art["paragraphs"] for q in par["qas"]}
        else:                                 # {id: {context, question, answers: [testi]}}, scritto dal notebook
            squad = sq
    # righe: uniche, coppie complete, prompt nel formato, coincidenti con i passaggi scelti
    ids = [r["row_id"] for r in righe]
    if len(ids) != len(set(ids)):
        errore("row_id ripetuti", len(ids) - len(set(ids)))
    for r in righe:
        if r["prompt"] != f"{ISTRUZIONE}\n\nPassage: {r['contesto']}\n\nQuestion: {r['domanda']}":
            errore("prompt fuori formato", r["row_id"])
    for nome, ps in sel.items():
        attese = {(p["passaggio_id"], fam, lato): F[lato]["testo"] for p in ps for fam, F in p["famiglie"].items()
                  for lato in ("B", "C")}
        trovate = {(r["passaggio_id"], r["famiglia"], r["kind"]): r["contesto"] for r in righe
                   if r["insieme"] == f"sintetico_{nome}"}
        if attese != trovate:
            errore("righe sintetiche diverse dai passaggi scelti", (nome, len(attese), len(trovate)))
        for p in ps:
            controlla_passaggio(p, nome)
    controlla_regola(ris["definitivo"], sel["definitivo"], reg["definitivo"], "definitivo", quote=QUOTE)
    controlla_regola(ris["prova"], sel["prova"], reg["prova"], "prova", n=N_PROVA)
    # intersezioni: ID e nomi fra prova e definitivo (sulle intere riserve), domande del naturale
    a, b = ris["definitivo"], ris["prova"]
    if {p["passaggio_id"] for p in a} & {p["passaggio_id"] for p in b}:
        errore("prova e definitivo con ID in comune", "")
    comuni = parole_nomi(a) & parole_nomi(b)
    if comuni:
        errore("prova e definitivo con nomi in comune", sorted(comuni)[:10])
    controlla_naturale(righe, reg_nat, squad)
    print(f"righe: {dict(Counter(r['insieme'] for r in righe))}")
    print(f"controlli sintetici: {sum(len(p['famiglie']) for ps in sel.values() for p in ps)} coppie; naturale: "
          f"{len({(r['famiglia'], r['sq_id']) for r in righe if r['insieme'] == 'naturale'})} coppie"
          f"{' (confrontate con SQuAD)' if squad else ' (senza il file di SQuAD: fonte non confrontata)'}")
    if err:
        print(f"ERRORI: {sum(err.values())}")
        for k, n in err.most_common():
            print(f"  {n:5d}  {k}")
            for e in esempi[k]:
                print(f"         es. {str(e)[:200]}")
        sys.exit(1)
    print("VERIFICA SUPERATA: nessun errore")


if __name__ == "__main__":
    args = sys.argv[1:]
    sq = None
    if "--squad" in args:
        i = args.index("--squad")
        sq = args[i + 1]
        args = args[:i] + args[i + 2:]
    main(args[0], sq)


In [ ]:
%%writefile /content/testC_sel6/script/tm_comune.py
# Funzioni comuni agli script del Test M. Girano in tutti e tre gli ambienti (Python 3.11 e 3.12):
# usano solo la libreria standard, numpy e torch.
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

LABELS = ("YES", "NO", "UNKNOWN")


def encode(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, allow_nan=False).encode("utf-8")


def fingerprint(obj):
    return hashlib.sha256(encode(obj)).hexdigest()


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_bytes(encode(obj))
    os.replace(tmp, path)


def read_json(path):
    return json.loads(Path(path).read_text())


def say(*args):
    print(time.strftime("%H:%M:%S"), *args, flush=True)


def parse_label(text):
    match = re.match(r"""^\s*[*#`"']*\s*(YES|NO|UNKNOWN)\b""", text, re.IGNORECASE)
    return match.group(1).upper() if match else "ALTRO"


def finite(x):
    return x is not None and x == x and x not in (float("inf"), float("-inf"))


def clean(x):
    # valori non finiti -> None, così il JSON resta valido e la coppia si esclude nell'analisi
    x = float(x)
    return x if finite(x) else None


def batches(rows, size):
    for start in range(0, len(rows), size):
        yield rows[start:start + size]


def checkpoint(folder, stage, rows, compute, size=64, key="row_id"):
    # Salva ogni blocco su Drive: una run interrotta riprende dai blocchi già scritti.
    # l'impronta copre il contenuto completo delle righe (prompt compresi), non solo gli identificativi
    sig = fingerprint(rows)[:16]
    folder = Path(folder) / "checkpoint" / f"{stage}-{sig}"
    folder.mkdir(parents=True, exist_ok=True)
    out = []
    for i, chunk in enumerate(batches(rows, size)):
        path = folder / f"{i:04d}.json"
        expected = [r[key] for r in chunk]
        if path.exists():
            part = read_json(path)
        else:
            part = list(compute(chunk, i))
            if [r[key] for r in part] != expected:
                raise RuntimeError(f"Risultati incompleti in {stage}, blocco {i}")
            write_json(path, part)
        if [r[key] for r in part] != expected:
            raise RuntimeError(f"Checkpoint incoerente in {stage}, blocco {i}")
        out.extend(part)
        if i == 0 or (i + 1) % 5 == 0 or len(out) == len(rows):
            say(f"  {stage} {len(out)}/{len(rows)}")
    return out


# ---------- codifica dei prompt e p(UNKNOWN), identiche al Test U2 ----------

def chat_text(tokenizer, model_key, prompt):
    options = {"enable_thinking": False} if model_key == "qwen" else {}
    return tokenizer.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                         add_generation_prompt=True, **options)


def prompt_ids(tokenizer, model_key, prompt):
    return tokenizer(chat_text(tokenizer, model_key, prompt), add_special_tokens=False)["input_ids"]


def label_ids(tokenizer):
    audit = {}
    for label in LABELS:
        variants = [prefix + word for word in (label, label.capitalize(), label.lower()) for prefix in ("", " ")]
        audit[label] = {v: tokenizer.encode(v, add_special_tokens=False) for v in variants}
    ids = {label: sorted({sequence[0] for sequence in audit[label].values()}) for label in LABELS}
    if len(set().union(*map(set, ids.values()))) != sum(map(len, ids.values())):
        raise RuntimeError(f"Il tokenizer condivide i primi token fra etichette: {ids}")
    return ids


def stop_ids(tokenizer, model):
    ids = set()
    for value in (tokenizer.eos_token_id, getattr(model.generation_config, "eos_token_id", None)):
        if isinstance(value, int):
            ids.add(value)
        elif isinstance(value, (list, tuple)):
            ids.update(v for v in value if isinstance(v, int))
    return ids


def predict_reference(model, tokenizer, model_key, rows, ids, keep_answer_ids=False):
    # Come la cella 8 del Test U2: batch 1, greedy, 8 token, probabilità delle etichette sul primo token.
    import torch
    stops = stop_ids(tokenizer, model)
    for row in rows:
        enc = tokenizer(chat_text(tokenizer, model_key, row["prompt"]), return_tensors="pt",
                        add_special_tokens=False).to(model.device)
        with torch.inference_mode():
            result = model.generate(**enc, do_sample=False, max_new_tokens=8, use_cache=True,
                                    return_dict_in_generate=True, output_logits=True,
                                    pad_token_id=tokenizer.pad_token_id if tokenizer.pad_token_id is not None
                                    else tokenizer.eos_token_id)
        logits = result.logits[0].float()
        if not bool(torch.isfinite(logits).all()):
            raise RuntimeError("Logit non finiti al primo passo")
        p = logits.softmax(dim=-1)[0]
        probability = [float(p[ids[label]].sum()) for label in LABELS]
        new = result.sequences[0, enc["input_ids"].shape[1]:].tolist()
        text = tokenizer.decode(new, skip_special_tokens=True)
        out = dict(row_id=row["row_id"], generated=text, parsed=parse_label(text),
                   prompt_tokens=int(enc["input_ids"].shape[1]), mass_total=sum(probability),
                   **{f"p_{k}": v for k, v in zip(LABELS, probability)})
        if keep_answer_ids:
            answer = []
            for t in new:                       # la risposta senza il token di fine turno
                if t in stops:
                    break
                answer.append(t)
            out["prompt_ids"] = enc["input_ids"][0].tolist()
            out["answer_ids"] = answer
        yield out


def load_causal_lm(model_key, model_id, revision, token):
    # Come il Test U2: bf16, attention eager, tutto sulla GPU 0, padding a sinistra.
    import torch
    import transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer
    extra = {"fix_mistral_regex": True} if model_key == "mistral" else {}
    tokenizer = AutoTokenizer.from_pretrained(model_id, revision=revision, token=token, trust_remote_code=False, **extra)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    dtype_arg = "dtype" if (major, minor) >= (4, 56) else "torch_dtype"
    model = AutoModelForCausalLM.from_pretrained(model_id, revision=revision, token=token, device_map={"": 0},
                                                 attn_implementation="eager", trust_remote_code=False,
                                                 low_cpu_mem_usage=True, **{dtype_arg: torch.bfloat16}).eval()
    return model, tokenizer


def alignment(results, reference):
    # reference: [(pair_key, kind, valore)] da P−S del Test U2
    by_key = {(r["pair_key"], r["kind"]): r for r in results}
    errors = [abs(by_key[(k, kind)]["p_UNKNOWN"] - v) for k, kind, v in reference if (k, kind) in by_key]
    return dict(n=len(errors), errore_assoluto_max=max(errors) if errors else None)


def manifest(**extra):
    import platform
    info = dict(python=platform.python_version(), eseguibile=sys.executable)
    from importlib import metadata
    for name in ("torch", "transformers", "numpy", "peft", "datasets", "lm-polygraph", "accelerate", "tokenizers"):
        try:
            info[name] = metadata.version(name)
        except Exception:
            info[name] = None
    try:
        import torch
        info["gpu"] = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
    except Exception:
        pass
    info.update(extra)
    return info


In [ ]:
%%writefile /content/testC_sel6/script/tm_m2_comune.py
# Funzioni comuni agli script di M2 (env_sirin): testo d'ingresso, resa con il chat template di SIRIN, lunghezze,
# confine dell'assistente e sottoinsiemi annidati dei dati di addestramento.
import re
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))

MAX_TOKEN = 512


def inspection_prompt(context, question):
    # come M3 e come _inspection_prompt dell'interfaccia di SIRIN (nota M2 v2, §2.5)
    return f'Context:\n{context.strip()}\n\nQuestion:\n{question.strip()}'


def sample_of(row):
    # campione di SIRIN: turno utente con il testo d'ingresso, turno dell'assistente vuoto, nessun prompt di sistema
    return [{"role": "user", "content": inspection_prompt(row["contesto"], row["domanda"])},
            {"role": "assistant", "content": ""}]


class Renderer:
    """Resa del campione come la fa HfModelAdapter di SIRIN (stesso tokenizer, stesso _preprocess_input).

    Il tokenizer si carica con la stessa chiamata di _load_model_and_tokenizer (AutoTokenizer, revisione fissata),
    senza caricare i pesi del modello."""

    def __init__(self, model_id, revision, tokenizer=None, adapter=None):
        from sirin.inference.adapters import HfModelAdapter
        from sirin.models.inference import HFConfig
        from transformers import AutoTokenizer
        if adapter is None:
            adapter = HfModelAdapter(HFConfig(model_path=model_id, revision=revision, device="cpu"))
            tok = tokenizer or AutoTokenizer.from_pretrained(model_id, revision=revision)
            if adapter.config.max_length is not None:
                tok.max_length = min(adapter.config.max_length, tok.model_max_length)
            tok.padding_side = adapter.config.padding_side or "right"
            adapter.tokenizer = tok
        self.adapter = adapter
        self.tok = adapter.tokenizer
        # Il localizzatore di SIRIN (HfTokenLocator.locate) decodifica i token del prompt e li ritokenizza per avere gli
        # offset. Con clean_up_tokenization_spaces=True (default di alcuni tokenizer) la decodifica toglie gli spazi
        # prima di «, . ? ! 's n't…», la ritokenizzazione dà meno token e il localizzatore si ferma (prova del 11
        # novembre: «200 token ids but 198 offsets»). La pulizia agisce solo sulla decodifica: la si spegne sull'istanza
        # del tokenizer, così la decodifica è il testo esatto. La tokenizzazione, e quindi gli stati, non cambiano.
        self.pulizia_spazi_originale = bool(getattr(self.tok, "clean_up_tokenization_spaces", False))
        self.tok.clean_up_tokenization_spaces = False

    def render(self, sample):
        return self.adapter._preprocess_input([sample])[0]

    def ids(self, text):
        # come generate_hiddens: add_special_tokens=False sul testo già reso
        return self.tok(text, add_special_tokens=False)["input_ids"]

    def full_length(self, row):
        return len(self.ids(self.render(sample_of(row))))

    def boundary(self, row):
        """Controlli del confine dell'assistente per un campione.

        n_full: token dell'intero prompt (quelli che entrano nel passaggio in avanti, senza padding né troncamento);
        n_user: token della resa del solo turno utente; ans_start: primo token della parte dell'assistente secondo il
        localizzatore di SIRIN (HfTokenLocator, locate_answer_start), cioè il confine usato da side=LEFT;
        prefix_ok: la resa completa comincia con quella del solo turno utente."""
        from sirin.inference.token_locators import HfTokenLocator
        from sirin.models.inference import TokenLocatorConfig
        sample = sample_of(row)
        full = self.render(sample)
        user = self.adapter._preprocess_input([sample[:1]])[0]
        full_ids = self.ids(full)
        answer = full[len(user):]                    # come locate_tokens di HfModelAdapter
        roundtrip = self.ids(self.tok.decode(full_ids, skip_special_tokens=False)) == full_ids
        try:
            loc = HfTokenLocator(TokenLocatorConfig(locate_answer_start=True)).locate(
                tokens=full_ids, tokenizer=self.tok, answer_text=answer)
            ans_start, err = int(loc["answer_start"]), None
        except ValueError as e:                     # il localizzatore di SIRIN non trova il confine: errore registrato
            ans_start, err = -1, str(e)[:300]
        return dict(n_full=len(full_ids), n_user=len(self.ids(user)), ans_start=ans_start,
                    prefix_ok=full.startswith(user), roundtrip_ok=roundtrip, errore_localizzatore=err, suffix=answer)


def word_ends(text):
    # fine (carattere escluso) di ogni parola, dove una parola è una sequenza di caratteri non bianchi
    return [m.end() for m in re.finditer(r"\S+", text)]


def nested(rows, n_total):
    """Sottoinsieme annidato: le prime n_total/2 righe di ciascuna classe nell'ordine fissato dalla selezione,
    poi l'ordine di addestramento fissato (campo 'ordine'). Le dimensioni piccole sono contenute nelle grandi."""
    assert n_total % 2 == 0, n_total
    half = n_total // 2
    out = []
    for label in (0, 1):
        part = [r for r in rows if r["label"] == label][:half]
        if len(part) != half:
            raise RuntimeError(f"servono {half} righe della classe {label}, ce ne sono {len(part)}")
        out += part
    return sorted(out, key=lambda r: r["ordine"])


In [ ]:
%%writefile /content/testC_sel6/script/test_regressioni_naturale.py
"""Regressioni con esempi e controesempi; solo libreria standard, nessuna inferenza."""

import sys

import unittest

from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

import controlli_semantici_naturale as N
import sel_testC as S
import verifica_selezione_v4 as V

class Naturale(unittest.TestCase):
    def test_decodifica_letterale_conserva_spazi(self):
        # Simula la pulizia attivata di default in alcuni tokenizer HF.
        class TokenizerFinto:
            def decode(self, ids, skip_special_tokens=False, clean_up_tokenization_spaces=True):
                originale = 'The answer is 1929 . Who founded it ?'
                if clean_up_tokenization_spaces:
                    return originale.replace(' .', '.').replace(' ?', '?')
                return originale
        self.assertEqual(S.decodifica_letterale(TokenizerFinto(), [1, 2]),
                         'The answer is 1929 . Who founded it ?')

    def test_tipo_da_tutti_i_riferimenti(self):
        ex = dict(question="Who designed the garden?", answers=dict(text=["the architect Jane Smith", "Jane Smith"], answer_start=[0,14]))
        self.assertEqual(S.riferimento_tipizzato(ex), ("nome", 1))
        ex = dict(question="What year?", answers=dict(text=["1815", "Battle of Waterloo"], answer_start=[0,5]))
        self.assertEqual(S.riferimento_tipizzato(ex), (None, None))

    def test_nascita_e_carica(self):
        self.assertIsNotNone(N.ancoraggio_domanda("When was the chancellor born?", "The chancellor (1862–90) opposed colonies."))
        self.assertIsNone(N.ancoraggio_domanda("When was the chancellor born?", "The chancellor was born in 1815."))

    def test_media_e_cronologia(self):
        self.assertIsNotNone(N.vincoli_quantitativi("What was the average duty?", "As high as 25 million, averaged VALORE_RIMOSSO.", "numero"))
        self.assertIsNone(N.vincoli_quantitativi("How many students?", "The college had VALORE_RIMOSSO students.", "numero"))
        self.assertIsNotNone(N.vincoli_quantitativi("When was the battle?", "Difficulties arose after 1279. The battle was in VALORE_RIMOSSO.", "anno"))
        self.assertIsNone(N.vincoli_quantitativi("When was the battle?", "The battle was in VALORE_RIMOSSO.", "anno"))

    def test_uno_e_un_numero(self):
        self.assertEqual(S.tipo_risposta("one"), "numero")
        self.assertEqual(S.numero_di("one"), 1.0)
        self.assertIn("one", S.valori_stesso_tipo("One student arrived.", "numero"))
        self.assertTrue(S.distrattore_plausibile("one", "two", "numero"))

    def test_risposta_abbreviata_e_aggettivo_geografico(self):
        forme = N.forme_nominali(["Louis Agassiz"], "Louis Agassiz gave lectures.")
        self.assertTrue(any(N.frase_contiene("Agassiz's view of nature.", f) for f in forme))
        self.assertIsNotNone(V.rim_nome_residuo("Louis Agassiz gave lectures.", "Agassiz's view of nature.", ["Louis Agassiz"]))
        forme = N.forme_nominali(["Victoria"], "Victoria is a state.")
        self.assertTrue(any(N.frase_contiene("Victorian fishing crews.", f) for f in forme))
        self.assertIsNotNone(V.rim_nome_residuo("Victoria is a state.", "Victorian fishing crews.", ["Victoria"]))

    def test_anafore_temporali_e_definite(self):
        for dopo in ("It is home to many people.", "The eye crossed the coast.", "In the following years the town expanded.", "A mere half decade later there were houses."):
            self.assertIsNotNone(N.rischio_discorso(["A key event happened in 1914.", dopo], {0}))
        self.assertIsNone(N.rischio_discorso(["A key event happened.", "A school opened in 1920."], {0}))
        self.assertIsNone(N.rischio_discorso(["A key event happened.", "The University of Example opened in 1920."], {0}))

    def test_elenco_e_sottoinsieme(self):
        self.assertTrue(N.elenco_residuo("How many metropolitan areas?", "The areas are: Alpha; Beta; Gamma."))
        self.assertFalse(N.elenco_residuo("How many students?", "The school admits students every year."))
        self.assertTrue(N.sottoinsieme_numerico("How many lines?", "Six of the seven lines run downtown.", "Six"))
        self.assertTrue(N.sottoinsieme_numerico("How many of the six packages?", "Two packages were awarded.", "Two"))
        self.assertFalse(N.sottoinsieme_numerico("How many museums?", "There are 60 museums.", "60"))

    def test_calcolo_tra_frasi(self):
        testo = "Setanta received VALORE_RIMOSSO of the six packages. Sky took the remaining four."
        self.assertIsNotNone(N.aritmetica_residua("How many packages did Setanta receive?", testo, 2))
        self.assertTrue(V.via_calcolo_v("How many packages did Setanta receive?", testo, 2))

    def test_ruolo_nominale_specifico(self):
        ex = dict(title="University_of_Chicago", question="What alumni member wrote the bestseller Before I Fall?", context="")
        other = dict(title="University_of_Chicago", question="What alumni is the Governor of the Bank of Japan?", context="The governor is Masaaki Shirakawa.")
        self.assertFalse(N.prova_distrattore(ex, other, "Masaaki Shirakawa", "persona")[0])
        ex = dict(title="Yuan_dynasty", question="Who led an attack on China?", context="")
        other = dict(title="Yuan_dynasty", question="Who did the Mongols bring to China?", context="Central Asian Muslims served as administrators.")
        self.assertFalse(N.prova_distrattore(ex, other, "Central Asian Muslims", "persona")[0])
        ex = dict(title="Business", question="Who was the founder of the Oracle Corporation?", context="")
        other = dict(title="Business", question="Who was the founder of community organizing?", context="Saul Alinsky was the founder.")
        self.assertFalse(N.prova_distrattore(ex, other, "Saul Alinsky", "persona")[0])
        ex = dict(title="Economics", question="Who wrote The Hidden Prosperity of the Poor?", context="")
        other = dict(title="Economics", question="Who wrote Capital in the Twenty-First Century?", context="Thomas Piketty wrote Capital.")
        self.assertTrue(N.prova_distrattore(ex, other, "Thomas Piketty", "persona")[0])

    def test_decimali_del_selettore(self):
        for risposta, candidato in (("11.1", "180"), ("0.2", "six"), ("94", "8.8"), ("91.3", "700")):
            self.assertFalse(S.distrattore_plausibile(risposta, candidato, "numero"))
        self.assertEqual(S.numero_di("11.1"), 11.1)
        self.assertEqual(S.numero_di("0.2"), 0.2)
        self.assertTrue(S.distrattore_plausibile("94", "100", "numero"))
        self.assertTrue(S.distrattore_plausibile("0.2", "0.3", "numero"))
        self.assertFalse(S.distrattore_plausibile("2500", "4404.5", "numero"))

    def test_confini_frasi_indipendenti(self):
        esempi = (
            '"It opened in 1950."  The site is old.',
            'The hall opened (in 1950.)  The site is old.',
            'The hall opened in the 1950s.  The site is old.',
        )
        for testo in esempi:
            with self.subTest(testo=testo):
                self.assertEqual(len(S.frasi_di(testo)), 2)
                self.assertEqual(len(V.frasi_naturali(testo)), 2)
        testo = 'Arthur H. Compton arrived. The site is old.'
        self.assertEqual(len(S.frasi_di(testo)), 2)
        self.assertEqual(len(V.frasi_naturali(testo)), 2)

    def test_nominali_plural_trattino_possessivo(self):
        for testo in ("Normans came into Scotland.", "A Norman-derived system."):
            self.assertTrue(any(N.frase_contiene(testo, f) for f in N.forme_nominali(["Norman"], testo)))
        self.assertTrue(any(N.frase_contiene("Watt's engine.", f) for f in N.forme_nominali(["Watt"], "Watt invented it.")))
        self.assertFalse(N.frase_contiene("Northumberland", "north"))

    def test_alias_esplicito_e_demonimo(self):
        forme = N.forme_nominali(["Porifera"], "Porifera (sponges) is a group.")
        self.assertIn("sponges", forme)
        self.assertTrue(any(N.frase_contiene("Chinese records", f) for f in N.forme_nominali(["China"], "China")))

    def test_domande_non_univoche(self):
        self.assertIsNotNone(N.domanda_non_univoca("Who was one of the famous residents?"))
        self.assertIsNotNone(N.domanda_non_univoca("What do some researchers believe?"))
        self.assertIsNone(N.domanda_non_univoca("Who designed the library garden?"))

    def test_nomi_completi(self):
        self.assertFalse(N.nome_intero("Napoleon's"))
        self.assertTrue(N.nome_intero("Maria Skłodowska-Curie"))

    def test_classi_nominali(self):
        self.assertEqual(N.classe_precisa("The Rhine and what other river drained the Alps?", "nome", "Danube"), "cosa:river")
        self.assertEqual(N.classe_precisa("Which other city is nearby?", "nome", "Zaltbommel"), "cosa:city")
        self.assertEqual(N.classe_precisa("What entity deals with staff?", "nome", "Civil Service Tribunal"), "cosa:tribunal")
        self.assertEqual(N.classe_precisa("What entity deals with staff?", "nome", "Conciliation Committee"), "cosa:committee")

    def test_somma_residua(self):
        self.assertIsNotNone(N.aritmetica_residua("How many students in total?", "5,792 students, 3,468 students and 5,984 students.", 15244))
        self.assertIsNone(N.aritmetica_residua("How many students?", "There are 4 teachers and 7 rooms.", 30))

    def test_enumerazione(self):
        self.assertTrue(N.elenco_residuo("How many extended metropolitan areas?", "These are Greater Los Angeles and San Diego–Tijuana."))
        self.assertFalse(N.elenco_residuo("When was the town founded?", "These are the parks and gardens."))

    def test_vincolo_geografico(self):
        ex = dict(title="Huguenot", question="Where did Huguenots settle in England?", context="")
        no = dict(title="Huguenot", question="Where did Huguenots settle in Holland?", context="Huguenots settled in Rotterdam in the Netherlands.")
        si = dict(title="Huguenot", question="Where did Huguenots settle in England?", context="Huguenots settled in Norwich in England.")
        self.assertFalse(N.prova_distrattore(ex, no, "Rotterdam", "luogo")[0])
        self.assertTrue(N.prova_distrattore(ex, si, "Norwich", "luogo")[0])

    def test_anafore(self):
        f = ["Refugees came to the city.", "They built workshops.", "The church opened in 1600."]
        self.assertIsNotNone(N.rischio_discorso(f, {0}))
        self.assertIsNone(N.rischio_discorso(f, {2}))

    def test_altri_operatori(self):
        self.assertTrue(N.altre_vie_nominali("Which company provides train service?", "A different railroad operates branchlines."))
        self.assertFalse(N.altre_vie_nominali("When was the town founded?", "A railroad operates here."))

if __name__ == "__main__":
    unittest.main(verbosity=2)


In [ ]:
# 3. Copia degli script e regressioni, prima di installare gli ambienti
NOMI_SCRIPT = ("gen_testC.py", "sel_testC.py", "controlli_semantici_naturale.py",
               "verifica_selezione_v4.py", "test_regressioni_naturale.py", "tm_comune.py", "tm_m2_comune.py")
(RADICE / "script").mkdir(exist_ok=True)
for nome in NOMI_SCRIPT:
    shutil.copy(SCRIPT / nome, RADICE / "script" / nome)
SHA_SCRIPT = {nome: sha_file(SCRIPT / nome) for nome in NOMI_SCRIPT}
scrivi_json(RADICE / "impronte_script.json", SHA_SCRIPT)
test = subprocess.run([sys.executable, "-I", str(SCRIPT / "test_regressioni_naturale.py")],
                      capture_output=True, text=True)
(RADICE / "regressioni.txt").write_text(test.stdout + test.stderr)
REGRESSIONI_OK = test.returncode == 0
print(test.stdout + test.stderr)
scrivi_json(RADICE / "regressioni.json", dict(superate=REGRESSIONI_OK, codice_uscita=test.returncode,
            sha256_script=SHA_SCRIPT["test_regressioni_naturale.py"]))
if not REGRESSIONI_OK:
    crea_zip()
    raise RuntimeError("Regressioni non superate: non eseguire le celle successive; manda lo ZIP.")


In [ ]:
# 3b. Versioni di env_sirin (le stesse dei notebook A e B, cella 7a)
PIN_SIRIN = '''\
absl-py==2.5.1
accelerate==1.15.0
agent-detector==2.0.0
aiohappyeyeballs==2.7.1
aiohttp==3.14.5
aiosignal==1.4.0
annotated-doc==0.0.5
annotated-types==0.8.0
anthropic==0.71.0
antlr4-python3-runtime==4.9.3
anyio==4.16.0
apache-tvm-ffi==0.1.15.post2
astor==0.8.1
asttokens==3.0.2
attrs==26.1.0
backoff==2.2.1
bert-score==0.3.13
bitsandbytes==0.50.2
blake3==1.0.11
blis==1.3.4
cachetools==8.0.0
catalogue==2.0.10
catboost==1.2.10
cbor2==6.2.1
certifi==2026.10.23
charset-normalizer==3.5.2
click==8.5.1
cloudpathlib==0.26.0
cloudpickle==3.1.2
cmake==3.24.2
colorama==0.4.6
comm==0.2.3
compressed-tensors==0.12.2
confection==1.3.4
contourpy==1.3.3
cuda-bindings==13.5.0
cuda-core==1.2.2
cuda-pathfinder==1.8.6
cuda-python==13.5.0
cupy-cuda12x==13.6.0
cut-cross-entropy==25.1.1
cycler==0.12.1
cymem==2.0.16
datasets==3.6.0
debugpy==1.8.22
defusedxml==0.7.1
depyf==0.20.0
detect-installer==0.2.1
diffusers==0.39.0
dill==0.3.8
diskcache==5.6.3
distro==1.9.0
dnspython==2.9.2
docopt==0.6.2
docstring-parser==0.18.0
einops==0.8.2
email-validator==2.3.0
eval-type-backport==0.4.0
evaluate==0.4.6
executing==2.3.0
faiss-cpu==1.15.1
fastapi==0.141.1
fastapi-cli==0.0.32
fastapi-cloud-cli==0.27.1
fastar==0.12.0
fastrlock==0.8.4
filelock==4.2.10
flashinfer-python==0.5.2
fonttools==4.68.0
frozenlist==1.8.0
fsspec==2025.3.0
gguf==0.19.0
gputil==1.4.0
graphviz==0.21
h11==0.16.0
hf-transfer==0.1.9
hf-xet==1.8.0
httpcore==1.0.9
httptools==0.9.0
httpx==0.28.1
huggingface-hub==0.36.2
hydra-core==1.3.8
idna==3.20
importlib-metadata==10.2.0
iniconfig==2.3.1
interegular==0.3.3
ipykernel==6.31.0
ipython==9.18.1
ipython-pygments-lexers==1.1.1
jedi==0.20.1
jinja2==3.1.6
jiter==0.19.0
jmespath==1.1.0
joblib==1.6.0
jsonschema==4.26.0
jsonschema-specifications==2025.9.1
jupyter-client==8.10.0
jupyter-core==5.9.1
kiwisolver==1.5.1
lark==1.2.2
lazy-importer==0.1.0
llguidance==1.3.0
llvmlite==0.44.0
lm-format-enforcer==0.11.3
loguru==0.7.3
lxml==6.1.3
markdown-it-py==4.2.0
markupsafe==3.0.4
matplotlib==3.11.2
matplotlib-inline==0.2.2
mdurl==0.1.2
mistral-common==1.13.1
model-hosting-container-standards==0.1.16
mpmath==1.3.0
msgpack==1.2.4
msgspec==0.22.0
multidict==7.2.1
multiprocess==0.70.16
murmurhash==1.0.16
narwhals==2.28.0
nest-asyncio==1.6.0
networkx==3.7
ninja==1.13.2
nltk==3.10.4
num2words==0.5.14
numba==0.61.2
numpy==1.26.4
nvidia-cublas-cu12==12.8.4.1
nvidia-cuda-cupti-cu12==12.8.90
nvidia-cuda-nvrtc-cu12==12.8.93
nvidia-cuda-runtime-cu12==12.8.90
nvidia-cudnn-cu12==9.10.2.21
nvidia-cudnn-frontend==1.28.0
nvidia-cufft-cu12==11.3.3.83
nvidia-cufile-cu12==1.13.1.3
nvidia-curand-cu12==10.3.9.90
nvidia-cusolver-cu12==11.7.3.90
nvidia-cusparse-cu12==12.5.8.93
nvidia-cusparselt-cu12==0.7.1
nvidia-cutlass-dsl==4.5.3
nvidia-cutlass-dsl-libs-base==4.5.3
nvidia-ml-py==13.615.71
nvidia-nccl-cu12==2.27.5
nvidia-nvjitlink-cu12==12.8.93
nvidia-nvshmem-cu12==3.3.20
nvidia-nvtx-cu12==12.8.90
omegaconf==2.3.1
openai==2.54.0
openai-harmony==0.0.8
opencv-python-headless==4.11.0.86
outlines-core==0.2.11
packaging==26.3
pandas==2.3.3
parso==0.8.7
partial-json-parser==0.2.1.1.post7
peft==0.17.1
pexpect==4.9.0
pillow==13.0.0
platformdirs==4.14.5
plotly==7.1.0
pluggy==1.7.0
portalocker==4.4.2
posthog==7.86.5
preshed==3.0.15
prometheus-client==0.26.0
prometheus-fastapi-instrumentator==8.1.0
prompt-toolkit==3.0.53
propcache==0.5.4
protobuf==3.20.3
psutil==7.2.2
ptyprocess==0.7.0
pure-eval==0.2.4
py-cpuinfo==9.0.0
pyarrow==25.0.1
pybase64==1.5.1
pycountry==26.2.16
pydantic==2.14.3
pydantic-core==2.50.4
pydantic-extra-types==2.11.1
pydantic-settings==2.16.2
pygments==2.21.0
pyparsing==3.3.3
pytest==9.2.0
python-dateutil==4.0.0
python-dotenv==1.2.4
python-json-logger==4.2.0
python-multipart==0.0.33
pytz==2026.6
pyyaml==6.0.3
pyzmq==27.3.0
ray==2.60.0
referencing==0.37.0
regex==2026.11.9
requests==2.34.2
rich==15.0.0
rich-toolkit==0.20.6
rignore==0.8.1
rouge-score==0.1.2
rpds-py==2026.9.1
sacrebleu==2.6.0
safetensors==0.8.0
scikit-dimension==0.3.7
scikit-learn==1.6.1
scipy==1.17.1
selfcheckgpt==0.1.7
sentence-transformers==5.7.0
sentencepiece==0.2.3
sentry-sdk==2.74.0
setproctitle==1.3.8
setuptools==80.10.2
shellingham==1.5.4
six==1.17.0
smart-open==9.0.3
sniffio==1.3.1
spacy==3.8.18
spacy-legacy==3.0.12
spacy-loggers==1.0.5
srsly==2.5.6
stack-data==0.6.3
starlette==1.10.0
supervisor==4.3.0
sympy==1.14.0
tabpfn==2.2.1
tabpfn-common-utils==0.2.23
tabulate==0.10.0
thinc==8.3.14
threadpoolctl==3.8.0
tiktoken==0.14.0
tokenizers==0.22.2
torch==2.9.0
torchao==0.14.1
torchaudio==2.9.0
torchvision==0.24.0
tornado==6.6.1
tqdm==4.70.1
traitlets==5.16.1
transformers==4.56.2
triton==3.5.0
trl==0.15.2
typeguard==4.6.0
typer==0.29.0
typing-extensions==4.16.0
typing-inspection==0.4.5
tyro==1.0.17
tzdata==2026.6
unsloth==2025.3.18
unsloth-zoo==2025.10.9
urllib3==2.8.0
uvicorn==0.54.0
uvloop==0.23.0
vllm==0.11.2
wasabi==1.1.3
watchfiles==1.3.0
wcwidth==0.9.4
weasel==1.0.1
websockets==17.4
wheel==0.48.0
wrapt==2.5.2
xformers==0.0.33.post1
xgrammar==0.1.25
xxhash==4.1.0
yarl==1.25.1
zipp==4.1.2
'''
assert 'transformers==4.56.2' in PIN_SIRIN and 'torchao==0.14.1' in PIN_SIRIN


In [ ]:
# 4. Rigenerazione della riserva (Python 3.12 in un ambiente uv a parte, con le tre librerie fissate) e controllo sha256
UV_VERSION = "0.11.32"
log = LOG / "generatore.log"
sh(f"{sys.executable} -m pip install -q uv=={UV_VERSION}", log)
GENENV = ENVS / "gen"
if not (GENENV / "bin/python").exists():
    sh(f"uv venv -q --python 3.12 {GENENV}", log)
    sh(f"uv pip install -q --python {GENENV / 'bin/python'} " + " ".join(PIN_GENERATORE), log)
GEN_OUT = LAVORO / "riserva"
sh(f"{GENENV / 'bin/python'} -I {SCRIPT / 'gen_testC.py'} {GEN_OUT}", log)
for nome, atteso in SHA_RISERVE.items():
    got = sha_file(GEN_OUT / f"testC_sintetico_{nome}_riserva.json")
    assert got == atteso, f"riserva {nome}: sha256 {got} invece di {atteso}. Il generatore non riproduce la riserva: fermarsi."
    print(f"riserva {nome}: sha256 uguale a quello atteso ({atteso[:12]})")
for f in ("testC_sintetico_definitivo_riserva.json", "testC_sintetico_prova_riserva.json", "controlli_generatore.json"):
    shutil.copy(GEN_OUT / f, RADICE / f)


In [ ]:
# 5. Ambiente env_sirin, come nei notebook A e B (stesse versioni; SIRIN al commit fissato)
log = LOG / "env_sirin.log"
SIRIN_DIR = LAVORO / "SIRIN"
PY_SIRIN = ENVS / "sirin/bin/python"
if not (SIRIN_DIR / ".git").exists():
    sh(f"git clone -q https://github.com/sb-ai-lab/SIRIN.git {SIRIN_DIR}", log)
sh(f"git -C {SIRIN_DIR} checkout -q {SIRIN_COMMIT}", log)
head = subprocess.run(["git", "-C", str(SIRIN_DIR), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
assert head == SIRIN_COMMIT, head
if not PY_SIRIN.exists():
    (LAVORO / "pin_sirin.txt").write_text(PIN_SIRIN)
    sh(f"uv venv -q --python 3.12 {ENVS / 'sirin'}", log)
    sh(f"uv pip install -q --python {PY_SIRIN} -r {LAVORO / 'pin_sirin.txt'}", log)
freeze = subprocess.run(["uv", "pip", "freeze", "--python", str(PY_SIRIN)], capture_output=True, text=True).stdout
(RADICE / "env_sirin.txt").write_text(freeze)
assert "transformers==4.56.2" in freeze, "env_sirin senza transformers 4.56.2"
print("env_sirin pronto; impronta dei pacchetti", hashlib.sha256(freeze.encode()).hexdigest()[:12])


In [ ]:
# 6. Selezione con tokenizer reali, nessuna inferenza
cfg = dict(versione=VERSIONE, uscite=str(RADICE / "dati"), tolleranza=TOLLERANZA, quote=QUOTE,
           n_prova=N_PROVA, riserve={k: str(GEN_OUT / f"testC_sintetico_{k}_riserva.json") for k in SHA_RISERVE},
           sha_riserve=SHA_RISERVE, modelli=MODELLI, giudice=GIUDICE, squad_revisione=SQUAD_REVISIONE,
           sha_selezione_test4=SHA_SELEZIONE_TEST4, seme_naturale=SEME_NATURALE, n_naturale=N_NATURALE,
           seme_revisione=SEME_REVISIONE, sha_script=SHA_SCRIPT)
scrivi_json(LAVORO / "cfg_sel.json", cfg)
scrivi_json(RADICE / "cfg_sel.json", cfg)
(RADICE / "pin_sirin.txt").write_text(PIN_SIRIN)
scrivi_json(RADICE / "versioni_fissate.json", dict(
    selezione=VERSIONE, generatore="C-gen-4", naturale="C-nat-7", verificatore="C-verifica-4",
    sirin_commit=SIRIN_COMMIT, uv=UV_VERSION, generatore_pacchetti=PIN_GENERATORE,
    sha_riserve=SHA_RISERVE, sha_contenuto_sintetico=SHA_CONTENUTO_SINTETICO,
    sha_id_sintetico=SHA_ID_SINTETICO))
env = dict(os.environ, PYTHONPATH=str(SIRIN_DIR), HF_HOME=str(LAVORO / "hf"), TOKENIZERS_PARALLELISM="false")
token = hf_token()
if token:
    env["HF_TOKEN"] = token
log = LOG / "selezione.log"
t0 = time.time()
SELEZIONE_EXIT_CODE = None
try:
    with open(log, "a") as fh:
        p = subprocess.Popen([str(PY_SIRIN), str(SCRIPT / "sel_testC.py"), str(LAVORO / "cfg_sel.json")], env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            fh.write(line)
            fh.flush()
            if not line.startswith(("Warning", "  warnings")):
                print(line, end="")
        SELEZIONE_EXIT_CODE = p.wait()
except BaseException:
    if "p" in globals() and p.poll() is None:
        p.terminate()
    crea_zip()
    raise
if SELEZIONE_EXIT_CODE != 0:
    print("Selezione interrotta con errore: l'ultima cella prepara lo ZIP dei dati disponibili.")
    print(log.read_text()[-4000:])
else:
    print(f"selezione finita in {(time.time() - t0) / 60:.1f} minuti")


In [ ]:
# 7. Verifica, manifest e ZIP (anche in caso di selezione incompleta)
errori = []
rap = None
quote_complete = False
sintetico_invariato = False
verifica_ok = False
confronti_sintetico = {}
v_code = None
try:
    if SELEZIONE_EXIT_CODE != 0:
        errori.append(f"selezione: codice di uscita {SELEZIONE_EXIT_CODE}")
    path_rap = RADICE / "dati/rapporto_selezione.json"
    if not path_rap.exists():
        raise RuntimeError("Rapporto della selezione assente; lo ZIP conterrà gli script e i log disponibili.")
    rap = json.loads(path_rap.read_text())
    if sha_file(RADICE / "dati/righe_C.json") != rap["sha256_righe_C"]:
        raise RuntimeError("Il file delle righe non corrisponde all'impronta del rapporto")
    if sha_file(RADICE / "dati/campione_revisione.md") != rap["sha256_campione_revisione"]:
        raise RuntimeError("Il campione non corrisponde all'impronta del rapporto")
    for nome in ("definitivo", "prova"):
        r = rap[nome]
        if sha_file(RADICE / f"dati/testC_sintetico_{nome}.json") != r["sha256_file"]:
            raise RuntimeError(f"Il file sintetico {nome} non corrisponde all'impronta del rapporto")
        print(f"{nome}: {r['scelti']} passaggi su {r['attesi']} ({'COMPLETO' if r['completo'] else 'INCOMPLETO'}), "
              f"tipi {r['tipi']}, generi {r['generi']}")
        print(f"  varianti scelte {r['varianti_scelte']}")
        print(f"  differenza massima di token C−B {r['differenza_token_max']}; token massimi SIRIN {r['token_sirin_max']}")
        print(f"  candidati valutati {r['candidati_valutati']}; motivi di esclusione {r['motivi_esclusione']}")
        if nome == "definitivo":
            print(f"  strati {r['strati']}; quote mancanti {r['quote_mancanti'] or 'nessuna'}")
        print(f"  sha256 del file {r['sha256_file'][:16]}, degli ID {r['sha256_id'][:16]}")
    n = rap["naturale"]
    print(f"naturale: coppie {n['coppie']} su {n['quota']}; completo {n['completo']}; tipi {n['tipi']}")
    print("regole del naturale:", n["regole_semantiche"])
    print("sha della selezione del test 4:", n["sha_selezione_test4"][:12])
    print("righe:", rap["righe"], "sha256", rap["sha256_righe_C"])
    reg = n.get("registro", {})
    print(f"registro naturale: {reg.get('domande')} domande; RIM-nat {reg.get('esiti_RIM')}; DUE-nat {reg.get('esiti_DUE')}")
    quote_complete = (all(rap[k]["completo"] for k in ("definitivo", "prova"))
                       and all(n["completo"].get(k, False) for k in ("RIM-nat", "DUE-nat")))
    if not quote_complete:
        errori.append("quote non tutte complete")
    for nome in ("definitivo", "prova"):
        path = RADICE / f"dati/testC_sintetico_{nome}.json"
        ps = json.loads(path.read_text())["passaggi"]
        h_body = sha_oggetto(ps)
        h_ids = hashlib.sha256("\n".join(p["passaggio_id"] for p in ps).encode()).hexdigest()
        ok_body = h_body == SHA_CONTENUTO_SINTETICO[nome]
        ok_ids = h_ids == SHA_ID_SINTETICO[nome]
        confronti_sintetico[nome] = dict(contenuto_identico=ok_body, id_identici=ok_ids,
            sha256_contenuto=h_body, sha256_id=h_ids, sha256_file=sha_file(path))
        print(f"sintetico {nome}: contenuto {'IDENTICO' if ok_body else 'DIVERSO'}, ID {'IDENTICI' if ok_ids else 'DIVERSI'} rispetto a C-sel-5")
    sintetico_invariato = all(x["contenuto_identico"] and x["id_identici"] for x in confronti_sintetico.values())
    if not sintetico_invariato:
        errori.append("contenuto o ID del sintetico diversi dall'atteso")
    for nome, atteso in SHA_RISERVE.items():
        if sha_file(RADICE / f"testC_sintetico_{nome}_riserva.json") != atteso:
            raise RuntimeError(f"Riserva {nome} con hash diverso dall'atteso")
    if rap.get("versione") != VERSIONE or rap.get("sha_script") != SHA_SCRIPT:
        raise RuntimeError("Versione o impronte degli script nel rapporto diverse dall'esecuzione corrente")
    # Dump della revisione fissata di SQuAD: solo fonte e metadati, mai token o risposte dei modelli.
    DUMP = """import json, sys
from datasets import load_dataset
va = load_dataset("rajpurkar/squad_v2", split="validation", revision=sys.argv[1])
json.dump({ex["id"]: dict(title=ex["title"], context=ex["context"], question=ex["question"],
                        answers=list(ex["answers"]["text"])) for ex in va}, open(sys.argv[2], "w"))
"""
    (SCRIPT / "dump_squad_dev.py").write_text(DUMP)
    shutil.copy(SCRIPT / "dump_squad_dev.py", RADICE / "script/dump_squad_dev.py")
    dump = LAVORO / "squad_dev.json"
    sh(f"{PY_SIRIN} {SCRIPT / 'dump_squad_dev.py'} {SQUAD_REVISIONE} {dump}", LOG / "verifica.log", env=env)
    scrivi_json(RADICE / "fonte_verifica.json", dict(dataset="rajpurkar/squad_v2", revisione=SQUAD_REVISIONE,
                split="validation", sha256_dump=sha_file(dump),
                sha256_script_dump=sha_file(SCRIPT / "dump_squad_dev.py")))
    v = subprocess.run([sys.executable, "-I", str(SCRIPT / "verifica_selezione_v4.py"), str(RADICE),
                        "--squad", str(dump)], capture_output=True, text=True)
    v_code = v.returncode
    (RADICE / "verifica_indipendente.txt").write_text(v.stdout + v.stderr)
    print("\n--- verifica indipendente ---\n" + (v.stdout + v.stderr)[-3500:])
    verifica_ok = v.returncode == 0
    if not verifica_ok:
        errori.append(f"verifica indipendente: codice di uscita {v.returncode}")
except Exception as exc:
    errori.append(str(exc))
    print("Verifica non completata:", exc)
finally:
    pronto = (not errori and SELEZIONE_EXIT_CODE == 0 and REGRESSIONI_OK
              and quote_complete and sintetico_invariato and verifica_ok)
    # Congelamento degli artefatti realmente prodotti; non approva la revisione manuale.
    file_dati = {str(f.relative_to(RADICE)): sha_file(f) for f in sorted((RADICE / "dati").glob("*")) if f.is_file()}
    manifest = dict(versione=VERSIONE, generatore="C-gen-4", regole_naturale="C-nat-7",
        verificatore="C-verifica-4", fase="completata" if pronto else "incompleta_o_con_errori",
        selezione_exit_code=SELEZIONE_EXIT_CODE, regressioni_superate=REGRESSIONI_OK,
        quote_complete=quote_complete, sintetico_invariato=sintetico_invariato,
        confronto_sintetico=confronti_sintetico,
        verifica_indipendente=dict(superata=verifica_ok, codice_uscita=v_code),
        pronto_per_revisione_manuale=pronto, revisione_manuale_approvata=False,
        modelli=MODELLI, giudice=GIUDICE, squad_revisione=SQUAD_REVISIONE,
        sha_selezione_test4=SHA_SELEZIONE_TEST4, sha_script=SHA_SCRIPT, sha256_file_dati=file_dati,
        errori=errori, nota="Usare nel completo solo dopo revisione e aggiornamento degli hash del notebook di inferenza.")
    scrivi_json(RADICE / "manifest_selezione.json", manifest)
    if pronto:
        print("\nSELEZIONE COMPLETA E VERIFICHE AUTOMATICHE SUPERATE. Campione da rivedere manualmente.")
    else:
        print("\nATTENZIONE: selezione incompleta o verifiche non superate. Manda lo ZIP con i dettagli.")
        for e in errori:
            print(" -", e)
    crea_zip()


# Audit C-sel-6: annotazioni del campione naturale

| N | Famiglia | ID SQuAD | Esito della revisione | Motivo |
| --- | --- | --- | --- | --- |
| 21 | RIM-nat | 570d4606b3d812140066d619 | errore | Victoria resta riconoscibile in Victorian; It rimane senza l’antecedente eliminato. |
| 22 | RIM-nat | 57286b003acd2414000df9c1 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 23 | RIM-nat | 57264845f1498d1400e8db0b | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 24 | RIM-nat | 5726241189a1e219009ac2e1 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 25 | RIM-nat | 57112686b654c5140001fbd4 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 26 | RIM-nat | 572820512ca10214002d9e74 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 27 | RIM-nat | 5733266d4776f41900660716 | dubbio | C inizia con In the following years senza l’evento che ne fissa l’anno. |
| 28 | RIM-nat | 57297547af94a219006aa45d | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 29 | RIM-nat | 571126dfa58dae1900cd6cb6 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 30 | RIM-nat | 572867d72ca10214002da2fc | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 31 | RIM-nat | 5727213c708984140094da38 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 32 | RIM-nat | 5729f8516aef05140015516d | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 33 | RIM-nat | 572759dbdd62a815002e9b96 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 34 | RIM-nat | 572fd8efb2c2fd14005684f9 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 35 | RIM-nat | 5728202c4b864d19001644ee | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 36 | RIM-nat | 5727c0402ca10214002d9564 | errore | Louis Agassiz è rimosso per esteso, ma Agassiz compare ripetutamente in C. |
| 37 | RIM-nat | 572fffb404bcaa1900d76ff1 | dubbio | La domanda Where è ampia: il riferimento residuo al mare può fornire una risposta geografica alternativa, pur non nominando il Reno. |
| 38 | RIM-nat | 5726b58f5951b619008f7b59 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 39 | RIM-nat | 570d2af0fed7b91900d45c96 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 40 | RIM-nat | 56de10b44396321400ee2593 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 41 | RIM-nat | 57286f373acd2414000df9df | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 42 | RIM-nat | 5728804b4b864d1900164a47 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 43 | RIM-nat | 572f57c704bcaa1900d7686e | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 44 | RIM-nat | 57340a094776f41900661780 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 45 | RIM-nat | 5725ec8289a1e219009ac0b0 | dubbio | A mere half decade later perde l’evento del 1914 a cui si riferiva. |
| 46 | RIM-nat | 57274ca8dd62a815002e9aa8 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 47 | RIM-nat | 57339902d058e614000b5e73 | dubbio | Eliminata la frase sul giardino, It e The building possono far riferire il progetto del giardino agli architetti dell’edificio. |
| 48 | RIM-nat | 572881704b864d1900164a50 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 49 | RIM-nat | 572824f13acd2414000df58f | dubbio | The eye e Similarly, four years prior to this perdono gli antecedenti; restano altri uragani ed effetti comparativi. |
| 50 | RIM-nat | 5733e771d058e614000b6545 | dubbio | C comincia con The treaty dopo aver eliminato la frase che identifica il trattato. |
| 51 | DUE-nat | 571093aba58dae1900cd6a5d | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 52 | DUE-nat | 57286bf84b864d19001649d2 | dubbio | Masaaki Shirakawa è documentato come governatore e alumni, senza evidenza del ruolo di autore del bestseller. |
| 53 | DUE-nat | 5705e3f252bb89140068966b | errore | Le sette aree sono ancora elencate dopo i due punti. L’alternativa two è anche inferiore alle three aree maggiori citate dopo. |
| 54 | DUE-nat | 572691545951b619008f76e4 | dubbio | Per Greenland la fonte attesta un trattato diverso del 1985; la comune parola treaty non documenta il ruolo nel contesto della domanda. |
| 55 | DUE-nat | 57286951ff5b5019007da20f | dubbio | Per Saul Alinsky la fonte attesta un fondatore di community organizing, non un fondatore di impresa. |
| 56 | DUE-nat | 57106d2fb654c5140001f8f1 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 57 | DUE-nat | 57300a9a04bcaa1900d77067 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 58 | DUE-nat | 572a0c541d046914007796f6 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 59 | DUE-nat | 57113be3a58dae1900cd6d28 | dubbio | Watt’s e Watt restano nel paragrafo: indizio asimmetrico a favore di James Watt, anche se non prova da solo la data del brevetto. |
| 60 | DUE-nat | 5706111a52bb89140068984e | errore | Ten of the seven lines è impossibile. Il totale sette permette di scartare il distrattore dieci. |
| 61 | DUE-nat | 572881d34b864d1900164a5c | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 62 | DUE-nat | 572867212ca10214002da2f2 | errore | Central Asian Muslims è un gruppo, contrapposto a una persona; He died in 1259 continua a richiedere un referente individuale. |
| 63 | DUE-nat | 57094a79efce8f15003a7dc5 | errore | Sei pacchetti totali e quattro remaining per Sky determinano due per Setanta, anche se in un’altra frase si aggiunge or 5. |
| 64 | DUE-nat | 5709686c200fba1400367f7a | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 65 | DUE-nat | 5727cebc2ca10214002d96f2 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 66 | DUE-nat | 57264a74708984140094c18e | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 67 | DUE-nat | 5730285a04bcaa1900d7726f | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 68 | DUE-nat | 572fd6aa04bcaa1900d76da2 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
| 69 | DUE-nat | 5710e9f8a58dae1900cd6b34 | dubbio | Colonia fondata nel 1564 e sopravvissuta only a short time: il 1604 ha un forte indizio cronologico sfavorevole rispetto al 1565. |
| 70 | DUE-nat | 57339eb9d058e614000b5ef7 | nessun problema rilevato | La rimozione o la disgiunzione non presenta un problema evidente nella lettura del campione. |
